# SMBHB Population Analysis & CGW SNR Diagnostics

This notebook analyzes SMBHB population outputs from the HPC pipeline and computes Continuous GW (CGW) SNR diagnostics.

## Quick Start

To run a complete CGW analysis:

1. **Activate the venv** (in terminal):
   ```bash
   . ~/jupyter/bin/activate
   cd /fred/oz005/users/bhoulden/SMBHB_population_injections
   export PYTHONPATH=$(pwd)
   jupyter lab
   ```

2. **Run cells in this order**:
   - Cell 1-2: **Setup** (imports, compatibility, plotting)
   - Cell 3-5: **Discovery & Loaders** (find result files, define unpickling)
   - Cell 6: **Load Binary Data** (populates `binary_df`)
   - Cell 7-8: **Simulation Summary** (aggregates to `sim_df`)
   - Cell 16-26: **CGW Analysis** (main plots: nearest distance, loudest binary, sky maps)

3. **Outputs**: Plots saved to `figures/` directory

## Formats Supported

- Old format: `data/YYYY-MM-DD/*/consistent_pop_synth*.pkl.gz`
- New Slurm format: `runs/YYYY-MM-DD_<scenario>/sim<NNN>/summary.pkl.gz` + `populations/subpop_*.pkl.gz`

Both are detected and loaded automatically.

In [ ]:
# Attach per-simulation SGWB S/N to per-binary DataFrames
# This cell scans `runs/**/metadata/sgwb_snr_summary.json` (uses `utils.get_sgwb_snr_map`)
# and merges the scalar `sgwb_snr` into each DataFrame that contains `sim_key`.
import warnings
try:
    from utils import get_sgwb_snr_map
except Exception as e:
    raise ImportError('Could not import get_sgwb_snr_map from utils.py') from e

# Build SGWB map (change scenario_label to 'baseline' if you prefer)
sgwb_map = get_sgwb_snr_map('runs', scenario_label='baseline_forecast', as_dataframe=True)
print(f'Found {len(sgwb_map)} SGWB records (example):')
print(sgwb_map.head())

def _attach_sgwb(df, sgwb_df=sgwb_map):
    if df is None:
        return df
    if 'sim_key' not in df.columns:
        warnings.warn('DataFrame has no sim_key column; skipping merge')
        return df
    # In-place assignment so existing references to the DataFrame see the new column
    try:
        mapping = sgwb_df.set_index('sim_key')['sgwb_snr']
        df['sgwb_snr'] = df['sim_key'].map(mapping)
    except Exception:
        # Fallback to a non-inplace merge if mapping fails
        return df.merge(sgwb_df[['sim_key', 'sgwb_snr']], on='sim_key', how='left')
    return df

# Attempt to attach to common DataFrames if they exist in the notebook globals.
for name in ('binary_df', 'cgw_sim_df', 'cgw_threshold_sim_df', 'synpta_sim_df'):
    if name in globals():
        df = globals()[name]
        globals()[name] = _attach_sgwb(df)
        n_nonna = globals()[name]['sgwb_snr'].notna().sum() if 'sgwb_snr' in globals()[name].columns else 0
        print(f'Updated {name}: {n_nonna} rows with SGWB SNR')
    else:
        print(f'{name} not present — run data-loading cells first if you want this merged now')

In [17]:
from __future__ import annotations

import json
import gzip
import pickle
import re
import sys
from typing import Optional
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NumPy compatibility: handle both NumPy 1.x (has numpy.core) and NumPy 2.x (has numpy._core)
NUMPY_VERSION_TUPLE = tuple(map(int, np.__version__.split('.')[:2]))
IS_NUMPY_2X = NUMPY_VERSION_TUPLE >= (2, 0)

if IS_NUMPY_2X:
    # We're on NumPy 2.x, create shim for unpickling NumPy 1.x pickles
    try:
        import numpy._core
        if not hasattr(np, 'core'):
            np.core = numpy._core
            sys.modules['numpy.core'] = numpy._core
    except (ImportError, AttributeError):
        pass

plt.style.use('default')
pd.set_option('display.max_columns', 50)

import types as _types

def _patch_numpy_modules():
    """Shim numpy.core → numpy._core for pickles created on NumPy 1.x."""
    if IS_NUMPY_2X:
        if 'numpy._core' not in sys.modules:
            _mod = _types.ModuleType('numpy._core')
            sys.modules['numpy._core'] = _mod
        if hasattr(np, '_core'):
            sys.modules['numpy._core'] = np._core
            if hasattr(np._core, 'multiarray'):
                sys.modules['numpy._core.multiarray'] = np._core.multiarray
    else:
        # NumPy 1.x: shim numpy._core → numpy.core
        if 'numpy._core' not in sys.modules:
            sys.modules['numpy._core'] = np.core
        if 'numpy._core.multiarray' not in sys.modules:
            sys.modules['numpy._core.multiarray'] = np.core.multiarray

_patch_numpy_modules()

SCENARIOS = ('optimistic', 'realistic', 'pessimistic')

In [18]:
from apj_style import apply_apj_style, APJ_COL_WIDTH, APJ_FIGSIZE
apply_apj_style()

In [8]:
def infer_scenario(path: Path) -> str:
    lower_parts = [p.lower() for p in path.parts]
    for s in SCENARIOS:
        if any(s in p for p in lower_parts):
            return s
        if s in path.name.lower():
            return s
    return 'unknown'


def infer_run_id(path: Path) -> str:
    date_cfg_re = re.compile(r'^\d{4}-\d{2}-\d{2}_.+$')
    date_re = re.compile(r'^\d{4}-\d{2}-\d{2}$')
    sim_re = re.compile(r'^sim\d+$')

    for p in path.parts:
        if date_cfg_re.match(p):
            return p
    for p in path.parts:
        if date_re.match(p):
            return p

    for parent in path.parents:
        name = parent.name
        if not name:
            continue
        if name in {'data', 'runs', 'results', 'output', 'outputs'}:
            continue
        if sim_re.match(name):
            continue
        return name
    return 'legacy'


def discover_result_files(data_roots = Path('data')):
    # Summary-only mode: only load summary.pkl.gz files from runs/.
    if isinstance(data_roots, (str, Path)):
        roots = [Path(data_roots)]
    else:
        roots = [Path(root) for root in data_roots]

    patterns = ('summary.pkl.gz',)

    ordered_files: list[Path] = []
    seen: set[Path] = set()
    for root in roots:
        if not root.exists():
            continue
        root_matches = set()
        for pattern in patterns:
            root_matches.update(root.rglob(pattern))
        for path in sorted(p for p in root_matches if p.is_file()):
            if path not in seen:
                seen.add(path)
                ordered_files.append(path)

    return ordered_files


# ANALYSIS_ROOTS = [Path('data'), Path('runs')]
ANALYSIS_ROOTS = [Path('runs')]
result_files = discover_result_files(ANALYSIS_ROOTS)
print(f'Discovered {len(result_files)} summary files')
for p in result_files[::-1][:20]:  # Show most recent 20 files
    print('-', p)
if len(result_files) > 20:
    print('...')

Discovered 1500 summary files
- runs/2026-07-21_pessimistic/sim790/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim789/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim788/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim787/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim786/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim785/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim784/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim783/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim782/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim781/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim780/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim779/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim778/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim777/summary.pkl.gz
- runs/2026-07-21_pessimistic/sim776/summary.pkl.gz
- runs/2026-07-19_realistic/sim626/summary.pkl.gz
- runs/2026-07-19_realistic/sim625/summary.pkl.gz
- runs/2026-07-19_realistic/sim624/summary.pkl.gz
- runs/2026-07-19_realistic/sim623/summa

In [19]:
def _extract_array_from_population_string(pop_text: str, key: str) -> Optional[np.ndarray]:
    # Extract key=array([ ... ]) from stringified PopulationArrays(...) output.
    pattern = rf"{re.escape(key)}=array\(\[(.*?)\]"
    match = re.search(pattern, pop_text, flags=re.DOTALL)
    if not match:
        return None

    raw = match.group(1).replace('\n', ' ')

    try:
        arr = np.fromstring(raw, sep=',')
    except ValueError:
        arr = np.array([], dtype=float)

    if arr.size == 0 and raw.strip():
        try:
            arr = np.fromstring(raw.replace(',', ' '), sep=' ')
        except ValueError:
            arr = np.array([], dtype=float)

    if arr.size == 0 and raw.strip():
        tokens = re.findall(
            r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?|[-+]?inf|nan',
            raw,
            flags=re.IGNORECASE,
        )
        if tokens:
            arr = np.asarray([float(t) for t in tokens], dtype=float)

    return arr if arr.size > 0 else None


def _population_to_arrays(population_obj) -> Optional[dict[str, np.ndarray]]:
    # Case 0: Slurm summary payload written by stage2_inject.py.
    if isinstance(population_obj, dict):
        arrays = population_obj.get('arrays')
        if isinstance(arrays, dict):
            out = {}
            for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'Mtot', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr_baseline_forecast'):
                arr = arrays.get(k)
                if arr is not None:
                    out[k] = np.asarray(arr, dtype=float)
            for k in ('global_idx', 'sim_id'):
                arr = arrays.get(k)
                if arr is not None:
                    out[k] = np.asarray(arr)
            return out if out else None

    # Case 1: population stored as dict with list/array values (compact representative format)
    if isinstance(population_obj, dict):
        has_pop_fields = any(k in population_obj for k in ('f', 'Mc', 'Mtot', 'D_comov', 'h0', 'z'))
        if has_pop_fields:
            out = {}
            for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'Mtot', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr_baseline_forecast'):
                if k in population_obj:
                    arr = population_obj[k]
                    if arr is not None:
                        out[k] = np.asarray(arr, dtype=float)
            return out if out else None

    # Case 2: population stored as list of dict rows (legacy format)
    if isinstance(population_obj, list):
        if not population_obj:
            return None
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr_baseline_forecast'):
            vals = [row.get(k, np.nan) for row in population_obj]
            out[k] = np.asarray(vals, dtype=float)
        return out

    # Case 3: true PopulationArrays object (via pickle)
    if hasattr(population_obj, 'f') and hasattr(population_obj, 'Mc'):
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr_baseline_forecast'):
            if hasattr(population_obj, k):
                arr = getattr(population_obj, k)
                if arr is not None:
                    out[k] = np.asarray(arr, dtype=float)
        return out if out else None

    # Case 4: stringified PopulationArrays(...) (legacy JSON fallback)
    if isinstance(population_obj, str):
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr_baseline_forecast'):
            arr = _extract_array_from_population_string(population_obj, k)
            if arr is not None:
                out[k] = arr.astype(float)
        return out if out else None

    return None


def _entry_to_binary_rows(entry: dict, source_file: Path, scenario: str, run_id: str, fallback_sim_indexx: int) -> list[dict]:
    """Build a binary dataframe from array-backed summary data without a Python row loop."""
    if not arrays or 'f' not in arrays:
        return pd.DataFrame()

    n = len(arrays['f'])
    d = arrays.get('D_comov')
    if d is None:
        d = arrays.get('D')

    return pd.DataFrame({
        'scenario': np.repeat(scenario, n),
        'run_id': np.repeat(run_id, n),
        'source_file': np.repeat(str(source_file), n),
        'sim_index': np.repeat(np.int32(sim_index), n),
        'sim_key': np.repeat(f'{scenario}:{run_id}:{sim_index}', n),
        'binary_index': np.arange(n, dtype=np.int32),
        'f': np.asarray(arrays.get('f', np.full(n, np.nan)), dtype=float),
        'Mc': np.asarray(arrays.get('Mc', np.full(n, np.nan)), dtype=float),
        'D': np.asarray(d if d is not None else np.full(n, np.nan), dtype=float),
        'h0': np.asarray(arrays.get('h0', np.full(n, np.nan)), dtype=float),
        'z': np.asarray(arrays.get('z', np.full(n, np.nan)), dtype=float),
        'cgw_snr': np.asarray(arrays.get('cgw_snr', np.full(n, np.nan)), dtype=float),
        'cgw_snr_baseline_forecast': np.asarray(arrays.get('cgw_snr_baseline_forecast', np.full(n, np.nan)), dtype=float),
        'ra': np.asarray(arrays.get('ra', np.full(n, np.nan)), dtype=float),
        'dec': np.asarray(arrays.get('dec', np.full(n, np.nan)), dtype=float),
        'psi': np.asarray(arrays.get('psi', np.full(n, np.nan)), dtype=float),
        'iota': np.asarray(arrays.get('iota', np.full(n, np.nan)), dtype=float),
        'phi0': np.asarray(arrays.get('phi0', np.full(n, np.nan)), dtype=float),
        'Mtot': np.asarray(arrays.get('Mtot', np.full(n, np.nan)), dtype=float),
        'global_idx': np.asarray(arrays.get('global_idx', np.arange(n, dtype=np.int64)), dtype=np.int64),
    })

In [20]:
def _summary_arrays_to_dataframe(arrays: dict, scenario: str, run_id: str, source_file: Path, sim_index: int = -1) -> pd.DataFrame:
    """Build a binary dataframe from array-backed summary data without a Python row loop."""
    if not arrays or 'f' not in arrays:
        return pd.DataFrame()

    n = len(arrays['f'])
    d = arrays.get('D_comov')
    if d is None:
        d = arrays.get('D')

    run_scope = _infer_run_scope_from_path(source_file)
    sim_name = source_file.parent.name
    sim_key = f'{run_scope}/{sim_name}' if run_scope else sim_name

    frame = pd.DataFrame({
        'scenario': np.repeat(scenario, n),
        'run_id': np.repeat(run_id, n),
        'run_scope': np.repeat(run_scope, n),
        'sim_name': np.repeat(sim_name, n),
        'source_file': np.repeat(str(source_file), n),
        'sim_index': np.repeat(np.int32(sim_index), n),
        'sim_key': np.repeat(sim_key, n),
        'binary_index': np.arange(n, dtype=np.int32),
        'f': np.asarray(arrays.get('f', np.full(n, np.nan)), dtype=np.float32),
        'Mc': np.asarray(arrays.get('Mc', np.full(n, np.nan)), dtype=np.float32),
        'D': np.asarray(d if d is not None else np.full(n, np.nan), dtype=np.float32),
        'h0': np.asarray(arrays.get('h0', np.full(n, np.nan)), dtype=np.float32),
        'z': np.asarray(arrays.get('z', np.full(n, np.nan)), dtype=np.float32),
        'cgw_snr': np.asarray(arrays.get('cgw_snr', np.full(n, np.nan)), dtype=np.float32),
        'cgw_snr_baseline_forecast': np.asarray(arrays.get('cgw_snr_baseline_forecast', np.full(n, np.nan)), dtype=np.float32),
        'ra': np.asarray(arrays.get('ra', np.full(n, np.nan)), dtype=np.float32),
        'dec': np.asarray(arrays.get('dec', np.full(n, np.nan)), dtype=np.float32),
        'psi': np.asarray(arrays.get('psi', np.full(n, np.nan)), dtype=np.float32),
        'iota': np.asarray(arrays.get('iota', np.full(n, np.nan)), dtype=np.float32),
        'phi0': np.asarray(arrays.get('phi0', np.full(n, np.nan)), dtype=np.float32),
        'Mtot': np.asarray(arrays.get('Mtot', np.full(n, np.nan)), dtype=np.float32),
        'global_idx': np.asarray(arrays.get('global_idx', np.arange(n, dtype=np.int32)), dtype=np.int32),
    })

    for column in ('scenario', 'run_id', 'run_scope', 'sim_name', 'source_file', 'sim_key'):
        frame[column] = frame[column].astype('category')

    return frame


In [21]:
from dataclasses import dataclass, field

@dataclass
class PopulationArrays:
    f       : np.ndarray
    Mc      : np.ndarray
    Mtot    : np.ndarray
    D_comov : np.ndarray
    z       : np.ndarray
    h0      : np.ndarray
    ra      : np.ndarray
    dec     : np.ndarray
    psi     : np.ndarray
    iota    : np.ndarray
    phi0    : np.ndarray
    cgw_snr : np.ndarray
    cgw_snr_baseline_forecast : np.ndarray
    amp_A   : Dict[str, np.ndarray] = field(default_factory=dict)
    amp_B   : Dict[str, np.ndarray] = field(default_factory=dict)

    def __len__(self): return len(self.f)

    def __getitem__(self, idx):
        new = PopulationArrays(
            f=self.f[idx], Mc=self.Mc[idx], Mtot=self.Mtot[idx],
            D_comov=self.D_comov[idx], z=self.z[idx], h0=self.h0[idx],
            ra=self.ra[idx], dec=self.dec[idx], psi=self.psi[idx],
            iota=self.iota[idx], phi0=self.phi0[idx], cgw_snr=self.cgw_snr[idx], cgw_snr_baseline_forecast=self.cgw_snr_baseline_forecast[idx],
        )
        for k, v in self.amp_A.items(): new.amp_A[k] = v[idx]
        for k, v in self.amp_B.items(): new.amp_B[k] = v[idx]
        return new
    
from pathlib import Path

def _summary_rows_from_sim_directory(
    summary_file: Path,
    payload: dict,
    scenario: str,
    run_id: str,
    verbose: bool = False,
) -> pd.DataFrame:
    arrays = payload.get('arrays', {}) if isinstance(payload, dict) else {}
    if not isinstance(arrays, dict):
        return pd.DataFrame()

    global_idx = arrays.get('global_idx')
    if global_idx is None:
        if verbose:
            print(f'  No global_idx in summary: {summary_file}')
        return pd.DataFrame()

    global_idx = np.asarray(global_idx, dtype=np.int64)
    if global_idx.size == 0:
        return pd.DataFrame()

    FIELDS = ['f','Mc','Mtot','D_comov','z','h0','ra','dec','psi','iota','phi0', 'cgw_snr', 'cgw_snr_baseline_forecast']

    # ── Fast path: physical arrays already in summary ──────────────────────
    if 'f' in arrays:
        if verbose:
            print(f'  Fast path: reading arrays directly from summary')
        frame = _summary_arrays_to_dataframe(arrays, scenario, run_id, summary_file, sim_index=-1)
        if not frame.empty:
            frame['global_idx'] = np.asarray(global_idx, dtype=np.int64)
        return frame

    # ── Fallback: read physical arrays from shard files ────────────────────
    pop_dir = summary_file.parent / 'populations'
    if not pop_dir.exists():
        if verbose:
            print(f'  No arrays in summary and no populations/ dir: {summary_file}')
        return pd.DataFrame()

    if verbose:
        print(f'  Sparse summary — reading from shards in {pop_dir}')

    # Build a map: global_idx → shard file + local position
    shard_files = sorted(pop_dir.glob('subpop_*.pkl.gz'))
    needed = set(global_idx.tolist())
    collected: dict[int, dict] = {}
    global_counter = 0

    for shard_path in shard_files:
        if not needed:
            break
        try:
            with gzip.open(shard_path, 'rb') as fh:
                pop = _CompatibilityUnpickler(fh).load()
        except Exception as e:
            if verbose:
                print(f'    Failed to load {shard_path.name}: {e}')
            global_counter += 0
            continue

        n = len(pop.f)
        for local_i in range(n):
            gidx = global_counter + local_i
            if gidx in needed:
                collected[gidx] = {
                    f: float(getattr(pop, f)[local_i])
                    for f in FIELDS
                    if hasattr(pop, f) and getattr(pop, f) is not None
                }
                needed.discard(gidx)
        global_counter += n
        del pop

    if verbose:
        print(f'  Recovered {len(collected)}/{global_idx.size} binaries from shards')

    # Extract sim_index from directory name
    sim_match = re.search(r'sim(\d+)', summary_file.parent.name)
    sim_index = int(sim_match.group(1)) if sim_match else -1

    rows = []
    for i, gidx in enumerate(global_idx.tolist()):
        row_data = collected.get(int(gidx), {})
        rows.append({
            'scenario': scenario, 'run_id': run_id,
            'source_file': str(summary_file),
            'sim_index': np.int32(sim_index),
            'binary_index': np.int32(i),
            'global_idx': np.int64(gidx),
            **{f: float(row_data.get(f, np.nan)) for f in FIELDS},
            'D': float(row_data.get('D_comov', np.nan)),
        })
    return pd.DataFrame(rows)

def _load_payload(path: Path, verbose: bool = False):
    """Load a .pkl.gz or .pkl file, trying CompatibilityUnpickler first."""
    import gzip, pickle
    try:
        with gzip.open(path, 'rb') as f:
            try:
                return _CompatibilityUnpickler(f).load()
            except Exception:
                pass
        with gzip.open(path, 'rb') as f:
            return pickle.load(f)
    except Exception as e:
        if verbose:
            print(f"  Failed to load {path}: {e}")
        return None
    
class _CompatibilityUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Redirect PopulationArrays from ANY module path to our local definition
        if name == 'PopulationArrays':
            return PopulationArrays
        # numpy core shim
        if module.startswith('numpy._core'):
            module = module.replace('numpy._core', 'numpy.core')
        elif module.startswith('numpy.core') and IS_NUMPY_2X:
            try:
                return super().find_class(module, name)
            except Exception:
                module = module.replace('numpy.core', 'numpy._core')
        # stub anything missing (numba etc.)
        try:
            return super().find_class(module, name)
        except Exception:
            _stub_missing_modules(module)
            try:
                return super().find_class(module, name)
            except Exception:
                return type(name, (), {'__module__': module})

In [22]:
# Override loader stubs for the Jupyter venv.
# The shard pickles import numba decorators such as `from numba import njit`.
# If we create `numba.njit` as a submodule, Python binds it as a module object
# and unpickling fails with `'module' object is not callable`.
# Keep the base `numba` module callable via decorator attributes, but do not
# create `numba.njit`/`numba.prange` submodules.

for _bad_mod in ('numba.njit', 'numba.prange', 'numba.vectorize', 'numba.guvectorize'):
    sys.modules.pop(_bad_mod, None)


def _stub_missing_modules(*module_names):
    for name in module_names:
        if name in sys.modules:
            continue
        try:
            __import__(name)
            continue
        except Exception:
            parts = name.split('.')
            if parts[0] == 'numba':
                stub = sys.modules.get('numba')
                if stub is None:
                    stub = types.ModuleType('numba')
                    stub.__path__ = []
                    sys.modules['numba'] = stub
                stub.jit = lambda *a, **k: (lambda f: f)
                stub.njit = lambda *a, **k: (lambda f: f)
                stub.vectorize = lambda *a, **k: (lambda f: f)
                stub.guvectorize = lambda *a, **k: (lambda f: f)
                stub.prange = range
                continue

            for i in range(len(parts)):
                parent = '.'.join(parts[: i + 1])
                if parent not in sys.modules:
                    stub = types.ModuleType(parent)
                    stub.__path__ = []
                    sys.modules[parent] = stub


# Re-run the numpy shim just in case this cell is executed standalone.
_patch_numpy_modules()

In [23]:
# DEBUG: Inspect what's in the discovered files
result_files_all = discover_result_files([Path('runs')])
print(f"Total files discovered: {len(result_files_all)}")

# Group by parent structure
from collections import defaultdict
by_sim = defaultdict(list)
for f in result_files_all:
    sim_dir = f.parent.name if 'sim' in f.parent.name else 'other'
    by_sim[sim_dir].append(f)

print(f"\nFile organization:")
for sim, files in sorted(by_sim.items())[:5]:
    print(f"  {sim}: {len(files)} files")
    for f in files[:2]:
        print(f"    - {f.name}")

# Try loading first few files and inspect their structure
print(f"\nInspecting file contents:")
for i, fp in enumerate(result_files_all[:3]):
    print(f"\n{i}. {fp.relative_to('.')}")
    try:
        payload = _load_payload(fp, verbose=False)
        print(f"   Type: {type(payload)}")
        if isinstance(payload, dict):
            print(f"   Keys: {list(payload.keys())[:10]}")
            if 'arrays' in payload:
                arrays = payload['arrays']
                print(f"   arrays type: {type(arrays)}")
                if isinstance(arrays, dict):
                    print(f"   arrays keys: {list(arrays.keys())}")
                    if 'global_idx' in arrays:
                        print(f"   global_idx: {type(arrays['global_idx'])}, len={len(arrays['global_idx']) if hasattr(arrays['global_idx'], '__len__') else '?'}")
        elif hasattr(payload, '__dict__'):
            print(f"   Attrs: {list(vars(payload).keys())[:5]}")
    except Exception as e:
        print(f"   Error: {type(e).__name__}: {e}")


Total files discovered: 1500

File organization:
  sim000: 1 files
    - summary.pkl.gz
  sim001: 1 files
    - summary.pkl.gz
  sim002: 1 files
    - summary.pkl.gz
  sim003: 1 files
    - summary.pkl.gz
  sim004: 1 files
    - summary.pkl.gz

Inspecting file contents:

0. runs/2026-07-16_optimistic/sim000/summary.pkl.gz
   Type: <class 'dict'>
   Keys: ['arrays', 'meta']
   arrays type: <class 'dict'>
   arrays keys: ['f', 'Mc', 'Mtot', 'D_comov', 'z', 'h0', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr', 'cgw_proxy', 'cgw_snr_baseline_forecast', 'cgw_snr_4x_cadence', 'cgw_snr_2x_precision', 'cgw_snr_4x_cad_2x_prec', 'cgw_snr_4x_cadence_conserved', 'cgw_snr_2x_precision_conserved', 'cgw_snr_4x_cad_2x_prec_conserved', 'cgw_snr_max_cadence_top10', 'cgw_snr_max_cadence_top20', 'cgw_snr_max_cadence_top30', 'cgw_snr_max_cadence_top40', 'global_idx']
   global_idx: <class 'numpy.ndarray'>, len=3936

1. runs/2026-07-16_optimistic/sim001/summary.pkl.gz
   Type: <class 'dict'>
   Keys: ['array

In [24]:
# ============================================
# STEP 1: LOAD BINARY DATAFRAME FROM RUNS
# ============================================

import gc


def _infer_run_scope_from_path(path: Path) -> str:
    try:
        return path.parent.parent.relative_to(Path('runs')).as_posix()
    except Exception:
        return path.parent.parent.as_posix()


def _infer_sim_index_from_path(path: Path) -> int:
    match = re.search(r'sim(\d+)', path.parent.name)
    return int(match.group(1)) if match else -1


def load_population_binary_table_robust(files, verbose=False):
    """
    Load binaries from summary files, handling multiple formats:
    1. Sparse summary: dict with 'arrays' key
    2. Compact format: dict with 'populations' list
    3. Direct array format: PopulationArrays-like object
    """
    frames = []

    for fp in files:
        if verbose:
            print(f"Loading: {fp.name}")

        payload = None
        try:
            scenario = infer_scenario(fp)
            run_id = infer_run_id(fp)
            run_scope = _infer_run_scope_from_path(fp)
            sim_name = fp.parent.name
            sim_index = _infer_sim_index_from_path(fp)
            sim_key = f'{run_scope}/{sim_name}' if run_scope else sim_name
            payload = _load_payload(fp, verbose=False)

            if payload is None or (isinstance(payload, dict) and not payload):
                if verbose:
                    print("  → Empty payload")
                continue

            # FORMAT 1: Sparse summary (dict with 'arrays' key)
            if isinstance(payload, dict) and 'arrays' in payload:
                if verbose:
                    print("  → Sparse format detected")
                df_rows = _summary_rows_from_sim_directory(fp, payload, scenario, run_id, verbose=verbose)
                if not df_rows.empty:
                    df_rows['run_scope'] = run_scope
                    df_rows['sim_name'] = sim_name
                    df_rows['sim_index'] = np.int32(sim_index)
                    df_rows['sim_key'] = sim_key
                    if 'source_file' in df_rows.columns:
                        df_rows['source_file'] = str(fp)
                    frames.append(df_rows)
                    if verbose:
                        print(f"    → {len(df_rows)} rows from sparse")
                continue

            # FORMAT 2: Old compact format (dict with 'populations' list)
            if isinstance(payload, dict) and 'populations' in payload:
                pops = payload.get('populations', [])
                if isinstance(pops, list) and pops:
                    if verbose:
                        print(f"  → Compact format: {len(pops)} populations")
                    rows = []

                    for pop_idx, pop_entry in enumerate(pops):
                        if not isinstance(pop_entry, dict):
                            continue
                        pop_obj = pop_entry.get('population', pop_entry)
                        arrays = _population_to_arrays(pop_obj)
                        if not arrays or 'f' not in arrays:
                            continue

                        n = len(arrays['f'])
                        for bi in range(n):
                            rows.append({
                                'scenario': scenario,
                                'run_id': run_id,
                                'run_scope': run_scope,
                                'sim_name': sim_name,
                                'source_file': str(fp),
                                'sim_index': np.int32(sim_index),
                                'binary_index': np.int32(bi),
                                'global_idx': np.int32(bi),
                                'sim_key': sim_key,
                                'f': np.float32(arrays['f'][bi]),
                                'Mc': np.float32(arrays.get('Mc', [np.nan] * n)[bi]),
                                'D': np.float32(arrays.get('D_comov', arrays.get('D', [np.nan] * n))[bi]),
                                'h0': np.float32(arrays.get('h0', [np.nan] * n)[bi]),
                                'z': np.float32(arrays.get('z', [np.nan] * n)[bi]),
                                'cgw_snr': np.float32(arrays.get('cgw_snr', [np.nan] * n)[bi]),
                                'cgw_snr_baseline_forecast': np.float32(arrays.get('cgw_snr_baseline_forecast', [np.nan] * n)[bi]),
                                'ra': np.float32(arrays.get('ra', [np.nan] * n)[bi]),
                                'dec': np.float32(arrays.get('dec', [np.nan] * n)[bi]),
                                'psi': np.float32(arrays.get('psi', [np.nan] * n)[bi]),
                                'iota': np.float32(arrays.get('iota', [np.nan] * n)[bi]),
                                'phi0': np.float32(arrays.get('phi0', [np.nan] * n)[bi]),
                                'Mtot': np.float32(arrays.get('Mtot', [np.nan] * n)[bi]),
                            })

                    if rows:
                        frames.append(pd.DataFrame(rows))
                        if verbose:
                            print(f"    → {len(rows)} rows from compact")
                    continue

            # FORMAT 3: Direct array object (has f, Mc, D attributes)
            if hasattr(payload, 'f'):
                if verbose:
                    print(f"  → Direct array format: {type(payload).__name__}")
                rows = []
                n = len(payload.f)
                for bi in range(n):
                    rows.append({
                        'scenario': scenario,
                        'run_id': run_id,
                        'run_scope': run_scope,
                        'sim_name': sim_name,
                        'source_file': str(fp),
                        'sim_index': np.int32(sim_index),
                        'binary_index': np.int32(bi),
                        'global_idx': np.int32(bi),
                        'sim_key': sim_key,
                        'f': np.float32(payload.f[bi]),
                        'Mc': np.float32(payload.Mc[bi] if hasattr(payload, 'Mc') else np.nan),
                        'D': np.float32(payload.D_comov[bi] if hasattr(payload, 'D_comov') else np.nan),
                        'h0': np.float32(payload.h0[bi] if hasattr(payload, 'h0') else np.nan),
                        'z': np.float32(payload.z[bi] if hasattr(payload, 'z') else np.nan),
                        'cgw_snr': np.float32(payload.cgw_snr[bi] if hasattr(payload, 'cgw_snr') else np.nan),
                        'cgw_snr_baseline_forecast': np.float32(payload.cgw_snr_baseline_forecast[bi] if hasattr(payload, 'cgw_snr_baseline_forecast') else np.nan),
                        'ra': np.float32(payload.ra[bi] if hasattr(payload, 'ra') else np.nan),
                        'dec': np.float32(payload.dec[bi] if hasattr(payload, 'dec') else np.nan),
                        'psi': np.float32(payload.psi[bi] if hasattr(payload, 'psi') else np.nan),
                        'iota': np.float32(payload.iota[bi] if hasattr(payload, 'iota') else np.nan),
                        'phi0': np.float32(payload.phi0[bi] if hasattr(payload, 'phi0') else np.nan),
                        'Mtot': np.float32(payload.Mtot[bi] if hasattr(payload, 'Mtot') else np.nan),
                    })
                if rows:
                    frames.append(pd.DataFrame(rows))
                    if verbose:
                        print(f"    → {n} rows from direct arrays")
                continue

            if verbose:
                print(f"  → Unrecognized format: {type(payload)}")

        except Exception as e:
            if verbose:
                print(f"  Error: {type(e).__name__}: {e}")
        finally:
            del payload
            if len(frames) % 8 == 0:
                gc.collect()

    if not frames:
        if verbose:
            print("→ No rows loaded from any file")
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    if verbose:
        print(f"\n✓ Total: {len(df)} rows across {len(files)} files")
    return df


def _population_to_arrays(pop_obj):
    """Convert various population formats to a dict of arrays."""
    if pop_obj is None:
        return None

    if isinstance(pop_obj, dict) and 'f' in pop_obj:
        return pop_obj

    if hasattr(pop_obj, 'f'):
        return {
            'f': pop_obj.f,
            'Mc': getattr(pop_obj, 'Mc', None),
            'D_comov': getattr(pop_obj, 'D_comov', getattr(pop_obj, 'D', None)),
            'h0': getattr(pop_obj, 'h0', None),
            'z': getattr(pop_obj, 'z', None),
            'cgw_snr': getattr(pop_obj, 'cgw_snr', None),
            'cgw_snr_baseline_forecast': getattr(pop_obj, 'cgw_snr_baseline_forecast', None),
            'ra': getattr(pop_obj, 'ra', None),
            'dec': getattr(pop_obj, 'dec', None),
            'psi': getattr(pop_obj, 'psi', None),
            'iota': getattr(pop_obj, 'iota', None),
            'phi0': getattr(pop_obj, 'phi0', None),
            'Mtot': getattr(pop_obj, 'Mtot', None),
        }

    if isinstance(pop_obj, str) and 'PopulationArrays' in pop_obj:
        try:
            import ast
            pass
        except Exception:
            pass

    return None


print("Step 1: Discovering result files...")
result_files = list(discover_result_files([Path('runs')]))
print(f"  → Found {len(result_files)} result files")

if result_files:
    print("\nStep 2: Loading population binary data with robust parser...")
    binary_df = load_population_binary_table_robust(result_files, verbose=True)
    print(f"\n  → Final: {len(binary_df)} binary rows loaded")
    if len(binary_df) > 0:
        print(f"  → Scenarios: {sorted(binary_df['scenario'].unique().tolist())}")
        if 'cgw_snr_baseline_forecast' in binary_df.columns:
            print(f"  → CGW SNR > 0: {(binary_df['cgw_snr_baseline_forecast'] > 0).sum()} binaries")
        sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
        if sky_cols:
            print(f"  → Sky-location columns loaded: {sky_cols}")
        display(binary_df.head())
else:
    print("ERROR: No result files found in runs/")
    binary_df = pd.DataFrame()

print(f"\nDataFrame shape: {binary_df.shape}")
print(f"Columns: {list(binary_df.columns) if not binary_df.empty else 'N/A'}")


Step 1: Discovering result files...
  → Found 1500 result files

Step 2: Loading population binary data with robust parser...
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 3936 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 1988 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 3949 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 2376 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 1980 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 2378 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary

,scenario,run_id,run_scope,sim_name,source_file,sim_index,sim_key,binary_index,f,Mc,D,h0,z,cgw_snr,cgw_snr_baseline_forecast,ra,dec,psi,iota,phi0,Mtot,global_idx
0,optimistic,2026-07-16_optimistic,2026-07-16_optimistic,sim000,runs/2026-07-16_optimistic/sim000/summary.pkl.gz,0,2026-07-16_optimistic/sim000,0,3.655662e-09,1.125677e+08,225.244171,1.697444e-17,0.050929,0.001427,0.00000,3.613281,0.791016,0.541992,2.521484,1.567383,2.608937e+08,0
1,optimistic,2026-07-16_optimistic,2026-07-16_optimistic,sim000,runs/2026-07-16_optimistic/sim000/summary.pkl.gz,0,2026-07-16_optimistic/sim000,1,2.098704e-09,6.783758e+08,5411.289551,1.959905e-17,1.999694,0.000730,0.00000,3.341797,0.541992,2.431641,2.919922,5.062500,1.688294e+09,1
2,optimistic,2026-07-16_optimistic,2026-07-16_optimistic,sim000,runs/2026-07-16_optimistic/sim000/summary.pkl.gz,0,2026-07-16_optimistic/sim000,2,4.862610e-09,9.094547e+07,202.190292,1.597527e-17,0.045661,0.002661,0.00000,0.772461,0.281982,1.186523,0.121582,2.044922,2.132784e+08,2
3,optimistic,2026-07-16_optimistic,2026-07-16_optimistic,sim000,runs/2026-07-16_optimistic/sim000/summary.pkl.gz,0,2026-07-16_optimistic/sim000,3,3.187234e-09,4.363650e+09,3364.490479,6.991386e-16,0.966009,0.041763,0.12647,2.632812,0.138062,2.228516,0.815918,4.074219,1.002708e+10,3
4,optimistic,2026-07-16_optimistic,2026-07-16_optimistic,sim000,runs/2026-07-16_optimistic/sim000/summary.pkl.gz,0,2026-07-16_optimistic/sim000,4,3.382941e-09,3.446250e+09,3680.392822,4.677813e-16,1.092399,0.028929,0.00000,5.621094,-0.132202,0.111206,1.372070,2.402344,7.926856e+09,4



DataFrame shape: (7544967, 22)
Columns: ['scenario', 'run_id', 'run_scope', 'sim_name', 'source_file', 'sim_index', 'sim_key', 'binary_index', 'f', 'Mc', 'D', 'h0', 'z', 'cgw_snr', 'cgw_snr_baseline_forecast', 'ra', 'dec', 'psi', 'iota', 'phi0', 'Mtot', 'global_idx']


In [30]:
"""
CGW simulation table builder, with an ISCO frequency cut.

Physical rationale
-------------------
A binary whose current GW frequency f already exceeds f_ISCO(M_total) is
past the adiabatic-inspiral regime (it has already merged / plunged in this
simple approximation), so it cannot be the source producing an ongoing,
loudest continuous-wave (CW) signal. When ranking binaries within a
simulation by cgw_snr, we therefore first drop any binary with
f >= f_ISCO(M_total), then take the highest-cgw_snr binary among the
survivors as "loudest".
"""

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# Physical constants (SI)
# ----------------------------------------------------------------------
G = 6.674299999999999e-11        # m^3 kg^-1 s^-2
C = 2.99792458e8                 # m / s
MSUN_KG = 1.98892e30              # kg

# f_ISCO = c^3 / (6^{3/2} * pi * G * M)   [Schwarzschild, non-spinning test limit]
_ISCO_COEFF = C**3 / (6.0**1.5 * np.pi * G)   # units: kg * Hz


def f_isco_hz(M_total_solar):
    """
    GW-frequency (l=2,m=2 harmonic, i.e. twice orbital frequency) at the
    innermost stable circular orbit, for total mass M_total_solar [Msun].

    Vectorized: accepts scalars, numpy arrays, or pandas Series.
    Returns NaN for non-positive / non-finite mass.
    """
    M_total_solar = np.asarray(M_total_solar, dtype=float)
    M_kg = M_total_solar * MSUN_KG
    with np.errstate(divide='ignore', invalid='ignore'):
        f = _ISCO_COEFF / M_kg
    f = np.where(M_total_solar > 0, f, np.nan)
    return f


class MissingMassRatioError(ValueError):
    """Raised when total mass cannot be derived without a real mass ratio."""
    pass


def total_mass_solar(df: pd.DataFrame) -> pd.Series:
    """
    Total binary mass in solar masses.

    Priority:
      1. Use 'M' column directly, for any rows where it's present and finite.
      2. For remaining rows, derive from chirp mass 'Mc' + mass ratio 'q'
         (q = m2/m1 <= 1):
             Mc = M * q^(3/5) / (1+q)^(6/5)
         =>  M  = Mc * (1+q)^(6/5) / q^(3/5)

    No equal-mass fallback. If a row needs the Mc/q derivation and 'q' is
    missing, non-numeric, or NaN for that row, this raises
    MissingMassRatioError rather than guessing q=1 -- an assumed mass ratio
    would silently bias M (and therefore f_ISCO) for any unequal-mass
    binary, which is exactly what this filter is supposed to get right.
    """
    M = pd.Series(np.nan, index=df.index, dtype=float)

    if 'Mtot' in df.columns:
        M = pd.to_numeric(df['Mtot'], errors='coerce')

    need_fill = M.isna()
    if not need_fill.any():
        return M

    if 'Mc' not in df.columns:
        raise MissingMassRatioError(
            f"{int(need_fill.sum())} row(s) have no usable 'M' and no 'Mc' "
            f"column exists to derive total mass from. Cannot compute f_ISCO "
            f"for these rows."
        )

    Mc = pd.to_numeric(df['Mc'], errors='coerce')

    if 'q' not in df.columns:
        raise MissingMassRatioError(
            f"{int(need_fill.sum())} row(s) need total mass derived from Mc, "
            f"but no 'q' (mass ratio) column exists. Refusing to assume "
            f"q=1 (equal mass) -- that would silently bias M and f_ISCO. "
            f"Provide 'M' directly or add a real 'q' column."
        )

    q = pd.to_numeric(df['q'], errors='coerce')
    bad_q = need_fill & q.isna()
    if bad_q.any():
        bad_idx = df.index[bad_q].tolist()
        raise MissingMassRatioError(
            f"{len(bad_idx)} row(s) have no usable 'M' and a missing/non-numeric "
            f"'q', so total mass cannot be derived without assuming a mass "
            f"ratio. Refusing to default to q=1. Offending row indices "
            f"(first 20 shown): {bad_idx[:20]}"
        )

    q_valid = q.clip(lower=1e-6, upper=1.0)
    M_from_Mc = Mc * (1.0 + q_valid) ** 1.2 / q_valid ** 0.6
    M.loc[need_fill] = M_from_Mc.loc[need_fill]

    return M


# ============================================
# STEP 1B: CGW HELPER FUNCTIONS
# ============================================
def build_cgw_simulation_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Build CGW simulation table from the loaded binary dataframe.

    Loudest binary is chosen as the max-cgw_snr binary among those that
    satisfy f < f_ISCO(M_total). Binaries with f >= f_ISCO are excluded
    from loudest-selection entirely (they cannot be "the loudest" CW
    source), regardless of their cgw_snr rank.

    Total mass (needed for f_ISCO) is derived via total_mass_solar(), which
    raises MissingMassRatioError if any row needs Mc->M derivation but has
    no usable 'q' -- there is no equal-mass fallback here by design.
    """
    if binary_df.empty or 'cgw_snr_baseline_forecast' not in binary_df.columns:
        return pd.DataFrame()
    df_cgw = binary_df[binary_df['cgw_snr_baseline_forecast'] > 0].copy()
    if df_cgw.empty:
        return pd.DataFrame()

    # Precompute total mass and f_ISCO once, vectorized, for the whole frame.
    # This will raise MissingMassRatioError if mass can't be determined
    # without assuming a mass ratio -- intentionally, no silent q=1 fallback.
    df_cgw['_M_total'] = total_mass_solar(df_cgw)
    df_cgw['_f_isco'] = f_isco_hz(df_cgw['_M_total'].values)
    df_cgw['_isco_valid'] = df_cgw['f'] < df_cgw['_f_isco']
    # If f_ISCO itself is NaN (shouldn't happen given the strict mass
    # derivation above, but just in case M_total is NaN/non-positive),
    # don't silently exclude the binary -- treat as valid but note it.
    df_cgw.loc[df_cgw['_f_isco'].isna(), '_isco_valid'] = True

    print(f"[ISCO filter] {(~df_cgw['_isco_valid']).sum()}/{len(df_cgw)} binaries excluded (f >= f_ISCO) "
          f"across {df_cgw.loc[~df_cgw['_isco_valid'], 'sim_key'].nunique()}/{df_cgw['sim_key'].nunique()} sims; "
          f"{(df_cgw.groupby('sim_key')['_isco_valid'].any() == False).sum()} sim(s) left with zero valid binaries")

    records = []
    for (scenario, sim_key), group in df_cgw.groupby(['scenario', 'sim_key'], dropna=False):
        nearest_d = group['D'].min() if 'D' in group.columns else np.nan

        valid = group[group['_isco_valid']]
        n_excluded = int((~group['_isco_valid']).sum())

        if valid.empty:
            # Every binary in this sim is past its own ISCO -- no valid
            # "loudest" CW source exists for this simulation.
            loudest = None
        else:
            loudest_idx = valid['cgw_snr_baseline_forecast'].idxmax()
            loudest = group.loc[loudest_idx]

        fallback = group.iloc[0]
        rec = {
            'scenario': scenario,
            'run_id': (loudest if loudest is not None else fallback).get('run_id', ''),
            'run_scope': (loudest if loudest is not None else fallback).get('run_scope', ''),
            'sim_name': (loudest if loudest is not None else fallback).get('sim_name', ''),
            'source_file': str((loudest if loudest is not None else fallback).get('source_file', '')),
            'sim_index': int((loudest if loudest is not None else fallback).get('sim_index', -1)),
            'sim_key': sim_key,
            'nearest_D': float(nearest_d),
            'n_binaries_total': int(len(group)),
            'n_excluded_by_isco': n_excluded,
            'has_valid_loudest': loudest is not None,
        }

        if loudest is not None:
            rec.update({
                'loudest_cgw_snr': float(loudest['cgw_snr_baseline_forecast']),
                'loudest_h0': float(loudest.get('h0', np.nan)),
                'loudest_Mc': float(loudest.get('Mc', np.nan)),
                'loudest_M_total': float(loudest.get('Mtot', np.nan)),
                'loudest_D': float(loudest.get('D', np.nan)),
                'loudest_z': float(loudest.get('z', np.nan)),
                'loudest_f': float(loudest.get('f', np.nan)),
                'loudest_f_isco': float(loudest.get('_f_isco', np.nan)),
                'loudest_ra': float(loudest.get('ra', np.nan)) if pd.notna(loudest.get('ra')) else np.nan,
                'loudest_dec': float(loudest.get('dec', np.nan)) if pd.notna(loudest.get('dec')) else np.nan,
                'loudest_psi': float(loudest.get('psi', np.nan)) if pd.notna(loudest.get('psi')) else np.nan,
                'loudest_iota': float(loudest.get('iota', np.nan)) if pd.notna(loudest.get('iota')) else np.nan,
                'loudest_phi0': float(loudest.get('phi0', np.nan)) if pd.notna(loudest.get('phi0')) else np.nan,
            })
        else:
            for col in ['loudest_cgw_snr', 'loudest_h0', 'loudest_Mc', 'loudest_M_total',
                        'loudest_D', 'loudest_z', 'loudest_f', 'loudest_f_isco',
                        'loudest_ra', 'loudest_dec', 'loudest_psi', 'loudest_iota',
                        'loudest_phi0']:
                rec[col] = np.nan

        records.append(rec)

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


def build_cgw_sim_inventory_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Track whether each simulation has any nonzero CGW SNR binaries."""
    if binary_df.empty or 'cgw_snr_baseline_forecast' not in binary_df.columns:
        return pd.DataFrame()
    records = []
    for (scenario, sim_key), group in binary_df.groupby(['scenario', 'sim_key'], dropna=False):
        records.append({
            'scenario': scenario,
            'run_id': group['run_id'].iloc[0] if 'run_id' in group.columns and not group.empty else '',
            'run_scope': group['run_scope'].iloc[0] if 'run_scope' in group.columns and not group.empty else '',
            'sim_name': group['sim_name'].iloc[0] if 'sim_name' in group.columns and not group.empty else '',
            'sim_index': int(group['sim_index'].iloc[0]) if 'sim_index' in group.columns and not group.empty else -1,
            'sim_key': sim_key,
            'has_cgw': bool((group['cgw_snr_baseline_forecast'] > 0).any()),
        })
    return pd.DataFrame.from_records(records) if records else pd.DataFrame()

In [31]:
f_isco_hz(10**10.0)

array(4.39604695e-07)

In [ ]:
# ============================================
# STEP 2: BUILD CGW ANALYSIS TABLE
# ============================================

print("Building CGW analysis table...")

# Always reload from the summary files so Step 2 cannot use stale binary_df state.
binary_df = load_population_binary_table_robust(result_files, verbose=True)
print(f"\n  → Reloaded: {len(binary_df)} binary rows loaded from summary.pkl.gz files")
if not binary_df.empty:
    print(f"  → Columns: {list(binary_df.columns)}")
    if 'cgw_snr_baseline_forecast' in binary_df.columns:
        finite_cgw = binary_df['cgw_snr_baseline_forecast'].replace([np.inf, -np.inf], np.nan).dropna()
        print(f"  → cgw_snr present: {len(finite_cgw)} finite values, {int((binary_df['cgw_snr_baseline_forecast'] > 0).sum())} positive")
    sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
    print(f"  → Sky-location columns: {sky_cols if sky_cols else 'none'}")

# Build tables
cgw_sim_inventory_df = build_cgw_sim_inventory_table_simple(binary_df)
cgw_sim_df = build_cgw_simulation_table_simple(binary_df)
# Attach per-simulation SGWB S/N to the CGW tables.
cgw_sim_inventory_df = _attach_sgwb(cgw_sim_inventory_df)
cgw_sim_df = _attach_sgwb(cgw_sim_df)
# Keep generic aliases for later cells that still expect the old names.
sim_df = cgw_sim_df.copy()
result_files_CGW = result_files.copy()

if not cgw_sim_df.empty:
    print(f"\n✓ CGW simulations found: {len(cgw_sim_df)}")
    print(f"  Scenarios: {sorted(cgw_sim_df['scenario'].unique().tolist())}")

    summary_cgw = (
        cgw_sim_df.groupby('scenario', as_index=False)
        .agg(
            n_sims=('sim_key', 'nunique'),
            median_cgw_snr=('loudest_cgw_snr', 'median'),
            max_cgw_snr=('loudest_cgw_snr', 'max'),
            median_nearest_D=('nearest_D', 'median'),
        )
    )
    print("\nCGW Summary by Scenario:")
    display(summary_cgw)
    print("\nFirst 10 CGW sources:")
    display(cgw_sim_df[['scenario', 'sim_index', 'loudest_cgw_snr', 'loudest_h0', 'loudest_Mc', 'loudest_f', 'loudest_D', 'loudest_z', 'loudest_ra', 'loudest_dec', 'loudest_psi', 'loudest_iota', 'loudest_phi0']].head(10))
else:
    print("✗ No CGW data found in summary.pkl.gz files (all cgw_snr ≤ 0, or the summary payload does not include cgw_snr/sky-location arrays)")

Building CGW analysis table...
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 3936 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 1988 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 3949 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 2376 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 1980 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 2378 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 4746 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: re

,scenario,n_sims,median_cgw_snr,max_cgw_snr,median_nearest_D
0,optimistic,500,4.224484,67.746338,64.533775
1,pessimistic,500,2.529375,152.119873,20.483492
2,realistic,500,3.633114,103.305496,29.565516



First 10 CGW sources:


,scenario,sim_index,loudest_cgw_snr,loudest_h0,loudest_Mc,loudest_f,loudest_D,loudest_z,loudest_ra,loudest_dec,loudest_psi,loudest_iota,loudest_phi0
0,optimistic,0,3.753683,5.135523e-15,6.960337e+09,2.099392e-08,3628.286621,1.070907,4.312500,-0.817871,1.524414,1.250977,0.921387
1,optimistic,1,3.762570,2.894038e-15,4.190078e+09,2.000784e-08,2232.062744,0.577469,5.746094,-0.051117,1.958008,0.379639,1.334961
2,optimistic,2,3.079681,2.086559e-15,4.667117e+09,1.836816e-08,5236.576172,1.886888,3.269531,-0.796875,0.484375,3.070312,2.427734
3,optimistic,3,3.813169,3.819118e-15,1.315798e+09,7.724943e-08,477.640381,0.109501,2.210938,-1.157227,2.496094,0.550293,4.437500
4,optimistic,4,2.478470,3.131827e-15,2.364231e+09,5.814313e-08,1463.907104,0.357201,6.246094,-0.351807,0.170898,1.151367,4.226562
5,optimistic,5,8.440338,7.809924e-15,1.086362e+10,1.137711e-08,3113.427246,0.871803,5.957031,-0.096741,1.298828,2.773438,0.443604
6,optimistic,6,7.252416,4.462591e-15,6.768372e+09,1.689285e-08,3304.063965,0.942855,4.507812,-0.494873,0.440918,0.030716,1.419922
7,optimistic,7,8.111524,6.691008e-15,6.175279e+09,4.166640e-08,3579.774414,1.051133,6.027344,-0.627930,1.747070,0.761230,3.822266
8,optimistic,8,2.210725,3.765720e-15,3.092268e+09,5.066422e-08,1820.775635,0.456005,4.191406,0.791504,0.388672,0.544922,1.787109
9,optimistic,9,4.280104,6.340545e-15,1.208530e+10,5.546957e-09,2683.686523,0.721826,2.552734,-1.056641,0.836426,2.865234,0.995117


In [33]:
# ============================================
# STEP 3: PLOT CGW ANALYSIS RESULTS  
# ============================================

if cgw_sim_df.empty:
    print("Skipping plots: No CGW data available")
else:
    from pathlib import Path
    Path('figures').mkdir(exist_ok=True)
    
    print("\nGenerating plots...")
    
    # Helper: extract values by scenario
    def _extract(df, col):
        return {
            s: df.loc[df['scenario'] == s, col].to_numpy()
            for s in SCENARIOS
        }
    
    # 1. Nearest distance
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for scenario in SCENARIOS:
        sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario]['nearest_D'].dropna()
        if len(sub) > 0:
            ax.hist(sub, bins=30, alpha=0.5, label=scenario, density=False)
    ax.set_xscale('log')
    ax.set_xlabel('Nearest SMBHB comoving distance [Mpc]')
    ax.set_ylabel('Count')
    ax.legend(frameon=False, loc='upper left')
    fig.tight_layout()
    fig.savefig("figures/cgw_nearest_SMBHB.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_nearest_SMBHB.pdf")
    
    # 2. CGW SNR distribution
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for scenario in SCENARIOS:
        sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario]['loudest_cgw_snr'].dropna()
        if len(sub) > 0:
            ax.hist(sub, bins=30, alpha=0.5, label=scenario, density=False)
    ax.set_xscale('log')
    ax.set_xlabel('Loudest binary CGW SNR')
    ax.set_ylabel('Count')
    ax.axvline(5.0, color='red', linestyle='--', linewidth=2, label='SNR=5 threshold')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig("figures/cgw_snr_distribution.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_snr_distribution.pdf")
    
    # 3. Loudest binary parameters
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    plot_cfg = [
        ('loudest_h0', '$h_0$', True),
        ('loudest_Mc', r'$\mathcal{M}_c$ [$M_\odot$]', True),
        ('loudest_D', r'$D_{\rm{comov}}$ [Mpc]', True),
        ('loudest_f', '$f$ [Hz]', True),
    ]
    for ax, (col, label, logx) in zip(axes.ravel(), plot_cfg):
        for scenario in SCENARIOS:
            sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario][col].dropna()
            if len(sub) > 0:
                ax.hist(sub, bins=20, alpha=0.5, label=scenario, density=False)
        ax.set_xlabel(label)
        ax.set_ylabel('Count')
        if logx:
            ax.set_xscale('log')
        ax.legend(frameon=False)
    fig.suptitle('Loudest Binary (max CGW SNR) Properties', y=1.01)
    fig.tight_layout()
    fig.savefig("figures/cgw_loudest_binary_parameters.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_loudest_binary_parameters.pdf")
    
    print("\n✓ All plots saved to figures/")
    print("  - cgw_nearest_SMBHB.pdf")
    print("  - cgw_snr_distribution.pdf")
    print("  - cgw_loudest_binary_parameters.pdf")


Generating plots...
  ✓ figures/cgw_nearest_SMBHB.pdf
  ✓ figures/cgw_snr_distribution.pdf
  ✓ figures/cgw_loudest_binary_parameters.pdf

✓ All plots saved to figures/
  - cgw_nearest_SMBHB.pdf
  - cgw_snr_distribution.pdf
  - cgw_loudest_binary_parameters.pdf


In [34]:
import sys, types
import numpy as np

# Create compatibility modules that older pickles expect.
if 'numpy._core' not in sys.modules:
    mod_core = types.ModuleType('numpy._core')
    sys.modules['numpy._core'] = mod_core

# Point numpy._core.multiarray to the real implementation
sys.modules['numpy._core.multiarray'] = np.core.multiarray
setattr(sys.modules['numpy._core'], 'multiarray', np.core.multiarray)

# Also ensure attribute aliasing some pickles use
if not hasattr(np.core.multiarray, '_reconstruct') and hasattr(np.core.multiarray, 'reconstruct'):
    setattr(np.core.multiarray, '_reconstruct', getattr(np.core.multiarray, 'reconstruct'))

In [35]:
def hist_by_scenario(data_by_scenario: dict[str, np.ndarray], bins, xlabel: str, title: str, save = False, savename = None, logx: bool = False):
    fig, ax = plt.subplots(figsize=(9, 5))
    scenario_labels = (r"$p(M) \propto M^{-1.21}$", 
         r"$p(M) \propto M^{-1.21}\exp(-\frac{M}{10^{10}\;\mathrm{M}_{\odot}})$", 
         r"$p(M) \propto M^{-1.21}\exp(-\frac{M}{10^{9}\;\mathrm{M}_{\odot}})$")
    for (scenario, label) in zip(SCENARIOS, scenario_labels):
        vals = data_by_scenario.get(scenario)
        if vals is None or len(vals) == 0:
            continue
        ax.hist(vals, bins=bins, alpha=0.45, density=False, label=label)

    if logx:
        ax.set_xscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('PDF')
    # ax.set_title(title)
    ax.legend(frameon=False)
    plt.tight_layout()
    if save:
        plt.savefig(savename)
    return fig, ax



# nearest_by_scenario = {
#     s: sim_df.loc[sim_df['scenario'] == s, 'nearest_D'].dropna().to_numpy()
#     for s in SCENARIOS
# }

# hist_by_scenario(
#     nearest_by_scenario,
#     bins=list(np.logspace(0, 4, 50)),
#     xlabel=r'$D_{\rm{comov}}$',
#     title='Nearest Binary Per Simulation by Scenario',
#     logx=True,
#     save = True,
#     savename="data/2026-04-01/nearest_SMBHB.pdf"
# )


In [36]:
def add_distribution_weights(df: pd.DataFrame, mode: str = 'all_binaries') -> pd.DataFrame:
    out = df.copy()
    if mode == 'all_binaries':
        out['weight'] = 1.0
        return out

    if mode == 'per_sim_equal':
        counts = out.groupby('sim_index')['binary_index'].transform('count').astype(float)
        out['weight'] = 1.0 / counts
        return out

    raise ValueError("mode must be 'all_binaries' or 'per_sim_equal'")


weight_mode = 'per_sim_equal'  # change to 'per_sim_equal' if desired
# dist_df = add_distribution_weights(binary_df, mode=weight_mode)

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# plot_specs = [
#     ('Mc', 'Chirp mass Mc', True),
#     ('f', 'GW frequency f [Hz]', True),
#     ('D', 'Comoving distance D', True),
# ]
# bins_mass = list(np.logspace(6, 11, 50))
# bins_f = list(np.logspace(-9, -6, 50))
# bins_dist = list(np.logspace(1, 5, 50))
# bins = [bins_mass, bins_f, bins_dist]


# for ax, (col, xlabel, logx), _bin in zip(axes, plot_specs, bins):
#     for scenario in SCENARIOS:
#         sub = dist_df[(dist_df['scenario'] == scenario) & np.isfinite(dist_df[col])]
#         if sub.empty:
#             continue
#         ax.hist(
#             sub[col].to_numpy(),
#             bins=_bin,
#             weights=sub['weight'].to_numpy(),
#             density=True,
#             alpha=0.35,
#             label=scenario,
#         )
#     if logx:
#         ax.set_xscale('log')
#     ax.set_xlabel(xlabel)
#     ax.set_ylabel('Density')
#     ax.set_xlim(min(_bin), max(_bin))

# axes[0].legend(frameon=False)
# fig.suptitle(f'Population Distributions by Scenario (weight_mode={weight_mode})')
# plt.tight_layout()
# plt.savefig("data/2026-04-01/distributions.pdf")

In [37]:
summary = (
    sim_df.groupby('scenario', as_index=False)
    .agg(
        n_sims=('sim_index', 'nunique'),
        nearest_D_median=('nearest_D', 'median'),
        nearest_D_p10=('nearest_D', lambda x: np.nanpercentile(x, 10)),
        nearest_D_p90=('nearest_D', lambda x: np.nanpercentile(x, 90)),
        loudest_h0_median=('loudest_h0', 'median'),
        loudest_h0_p10=('loudest_h0', lambda x: np.nanpercentile(x, 10)),
        loudest_h0_p90=('loudest_h0', lambda x: np.nanpercentile(x, 90)),
    )
)

summary

,scenario,n_sims,nearest_D_median,nearest_D_p10,nearest_D_p90,loudest_h0_median,loudest_h0_p10,loudest_h0_p90
0,optimistic,499,64.533775,32.929961,110.449757,4.460386e-15,1.709942e-15,1.032247e-14
1,pessimistic,500,20.483492,7.768743,35.262801,1.788779e-15,1.346063e-16,5.193622e-15
2,realistic,500,29.565516,11.936173,52.556929,3.214677e-15,1.520346e-15,7.571563e-15


# Notes on Combining Across Simulations

Default in this notebook:
- nearest/loudest metrics are per simulation (one point per sim), which is robust for scenario comparison
- distribution plots use all binaries by default (`weight_mode='all_binaries'`)

Alternative:
- set `weight_mode='per_sim_equal'` to avoid simulations with more binaries dominating parameter distributions

If you later add SNR contribution diagnostics, keep the same grouping key (`sim_id`) so all summaries stay aligned.

## Storage Recommendation

For your workflow, use one primary format by default: **compressed pickle (`.pkl.gz`)**.

Why this is the best default here:
- full-fidelity recovery of `PopulationArrays` objects
- much smaller than plain `.pkl`
- still straightforward to load in Python

When to additionally save `.npz`:
- only if disk size or data transfer is a bottleneck
- only for plotting/array workflows (not full object reconstruction)

In [38]:
# # Minimal examples
# from pathlib import Path

# from utils import load_results_pickle_gz, save_results_compact_npz

# # 1) Preferred: load full-fidelity compressed pickle
# pkl_gz_path = Path('data/2026-03-31/realistic/consistent_population_realistic_targetSNR4.0_sims10.pkl.gz')
# if pkl_gz_path.exists():
#     results_obj = load_results_pickle_gz(pkl_gz_path)
#     print('Loaded:', pkl_gz_path)
#     print('n sims:', len(results_obj.get('populations', [])))
# else:
#     print('Example file not found:', pkl_gz_path)

# # 2) Optional: create compact NPZ for plotting-only pipelines
# if 'results_obj' in locals():
#     npz_path = pkl_gz_path.with_suffix('').with_suffix('.npz')
#     save_results_compact_npz(results_obj, npz_path)
#     print('Saved compact npz:', npz_path)

## Continuous GW (CGW) SNR Diagnostics

This section reads CGW diagnostics saved in the same `compact_results` structure used in `main.py`, i.e. per simulation:
- `compact_results['populations'][i]['population']`
- `compact_results['populations'][i]['cgw_analysis']['top_sources']`

It builds per-simulation metrics across scenarios and then plots:
- nearest SMBHB distance per simulation,
- loudest-by-CGW binary (`max SNR`) per simulation: `h0`, `Mc`, `D`, and `f`,
- two sky maps for loudest-by-CGW binaries: plain dots and SNR-colored.


In [39]:
len(result_files_CGW)

1500

In [40]:
# Simplified CGW analysis: Build table directly from binary_df (works for summary.pkl.gz payloads)

def build_cgw_simulation_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Build CGW simulation table from the loaded binary dataframe."""
    if binary_df.empty or 'cgw_snr_baseline_forecast' not in binary_df.columns:
        return pd.DataFrame()

    df_cgw = binary_df[binary_df['cgw_snr_baseline_forecast'] > 0].copy()
    if df_cgw.empty:
        return pd.DataFrame()

    records = []
    for (scenario, run_id, sim_index), group in df_cgw.groupby(['scenario', 'run_id', 'sim_index'], dropna=False):
        nearest_idx = group['D'].idxmin()
        nearest = group.loc[nearest_idx]
        loudest_idx = group['cgw_snr_baseline_forecast'].idxmax()
        loudest = group.loc[loudest_idx]

        records.append({
            'scenario': scenario,
            'run_id': run_id,
            'source_file': str(loudest.get('source_file', '')),
            'sim_index': int(sim_index),
            'sim_key': f"{scenario}:{run_id}:{sim_index}",
            'nearest_D': float(nearest.get('D', np.nan)),
            'nearest_DL': float(nearest.get('D', np.nan))* (1 + float(nearest.get('z', np.nan))),
            'loudest_cgw_snr': float(loudest['cgw_snr_baseline_forecast']),
            'loudest_cgw_snr_baseline_forecast': float(loudest['cgw_snr_baseline_forecast']),
            'loudest_h0': float(loudest.get('h0', np.nan)),
            'loudest_Mc': float(loudest.get('Mc', np.nan)),
            'loudest_D': float(loudest.get('D', np.nan)),
            'loudest_z': float(loudest.get('z', np.nan)),
            'loudest_DL': float(loudest.get('D', np.nan)) * (1 + float(loudest.get('z', np.nan))),
            'loudest_f': float(loudest.get('f', np.nan)),
            'loudest_ra': float(loudest.get('ra', np.nan)) if pd.notna(loudest.get('ra')) else np.nan,
            'loudest_dec': float(loudest.get('dec', np.nan)) if pd.notna(loudest.get('dec')) else np.nan,
            'loudest_psi': float(loudest.get('psi', np.nan)) if pd.notna(loudest.get('psi')) else np.nan,
            'loudest_iota': float(loudest.get('iota', np.nan)) if pd.notna(loudest.get('iota')) else np.nan,
            'loudest_phi0': float(loudest.get('phi0', np.nan)) if pd.notna(loudest.get('phi0')) else np.nan,
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


def build_cgw_sim_inventory_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Track whether each simulation has any nonzero CGW SNR binaries."""
    if binary_df.empty or 'cgw_snr_baseline_forecast' not in binary_df.columns:
        return pd.DataFrame()

    records = []
    for (scenario, run_id, sim_index), group in binary_df.groupby(['scenario', 'run_id', 'sim_index'], dropna=False):
        records.append({
            'scenario': scenario,
            'run_id': run_id,
            'sim_index': int(sim_index),
            'sim_key': f"{scenario}:{run_id}:{sim_index}",
            'has_cgw': bool((group['cgw_snr_baseline_forecast'] > 0).any()),
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


In [41]:
# Build tables from summary.pkl.gz-backed rows
print('Reloading binary_df from summary.pkl.gz files...')
binary_df = load_population_binary_table_robust(result_files, verbose=False)
print(f'  → Reloaded {len(binary_df)} binary rows')
if not binary_df.empty:
    sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
    print(f"  → Sky-location columns loaded: {sky_cols if sky_cols else 'none'}")

Reloading binary_df from summary.pkl.gz files...
  → Reloaded 7544967 binary rows
  → Sky-location columns loaded: ['ra', 'dec', 'psi', 'iota', 'phi0']


In [ ]:
cgw_sim_inventory_df = build_cgw_sim_inventory_table_simple(binary_df)
cgw_sim_df = build_cgw_simulation_table_simple(binary_df)
# Attach per-simulation SGWB S/N to the CGW tables.
cgw_sim_inventory_df = _attach_sgwb(cgw_sim_inventory_df)
cgw_sim_df = _attach_sgwb(cgw_sim_df)
# Keep generic aliases for later cells that still expect the old names.
sim_df = cgw_sim_df.copy()
result_files_CGW = result_files.copy()

if not cgw_sim_df.empty:
    print(f"\n✓ CGW simulations found: {len(cgw_sim_df)}")
    print(f"  Scenarios: {sorted(cgw_sim_df['scenario'].unique().tolist())}")
    
    summary_cgw = (
        cgw_sim_df.groupby('scenario', as_index=False)
        .agg(
            n_sims=('sim_key', 'nunique'),
            median_cgw_snr=('loudest_cgw_snr', 'median'),
            max_cgw_snr=('loudest_cgw_snr', 'max'),
            median_nearest_D=('nearest_D', 'median'),
        )
    )
    print("\nCGW Summary by Scenario:")
    display(summary_cgw)
    print("\nFirst 10 CGW sources:")
    display(cgw_sim_df[['scenario', 'sim_index', 'loudest_cgw_snr', 'loudest_h0', 'loudest_D', 'loudest_z', 'loudest_Mc', 'loudest_f', 'loudest_ra', 'loudest_dec']].head(10))
else:
    print("✗ No CGW data found in summary.pkl.gz files (all cgw_snr ≤ 0, or the summary payload does not include cgw_snr/sky-location arrays)")


✓ CGW simulations found: 1500
  Scenarios: ['optimistic', 'pessimistic', 'realistic']

CGW Summary by Scenario:


,scenario,n_sims,median_cgw_snr,max_cgw_snr,median_nearest_D
0,optimistic,500,4.224484,67.746338,64.533775
1,pessimistic,500,2.529375,152.119873,20.483492
2,realistic,500,3.633114,103.305496,29.565516



First 10 CGW sources:


,scenario,sim_index,loudest_cgw_snr,loudest_h0,loudest_D,loudest_z,loudest_Mc,loudest_f,loudest_ra,loudest_dec
0,optimistic,0,3.753683,5.135523e-15,3628.286621,1.070907,6.960337e+09,2.099392e-08,4.312500,-0.817871
1,optimistic,1,3.762570,2.894038e-15,2232.062744,0.577469,4.190078e+09,2.000784e-08,5.746094,-0.051117
2,optimistic,2,3.079681,2.086559e-15,5236.576172,1.886888,4.667117e+09,1.836816e-08,3.269531,-0.796875
3,optimistic,3,3.813169,3.819118e-15,477.640381,0.109501,1.315798e+09,7.724943e-08,2.210938,-1.157227
4,optimistic,4,2.478470,3.131827e-15,1463.907104,0.357201,2.364231e+09,5.814313e-08,6.246094,-0.351807
5,optimistic,5,8.440338,7.809924e-15,3113.427246,0.871803,1.086362e+10,1.137711e-08,5.957031,-0.096741
6,optimistic,6,7.252416,4.462591e-15,3304.063965,0.942855,6.768372e+09,1.689285e-08,4.507812,-0.494873
7,optimistic,7,8.111524,6.691008e-15,3579.774414,1.051133,6.175279e+09,4.166640e-08,6.027344,-0.627930
8,optimistic,8,2.210725,3.765720e-15,1820.775635,0.456005,3.092268e+09,5.066422e-08,4.191406,0.791504
9,optimistic,9,4.280104,6.340545e-15,2683.686523,0.721826,1.208530e+10,5.546957e-09,2.552734,-1.056641


In [43]:
# Okabe-Ito colourblind-safe palette
scenario_labels = (r"Heavy", 
         r"Intermediate", 
         r"Light")

scenario_styles = {
    r"Heavy" :  {'color': '#0072B2', 'linestyle': '-',  'linewidth': 2.2},
    r"Intermediate":   {'color': '#E69F00', 'linestyle': '--', 'linewidth': 2.2},
    r"Light": {'color': '#D55E00', 'linestyle': ':',  'linewidth': 2.5},
}
scenario_styles = {
    'optimistic':  {'color': 'lime', 'linestyle': '-',  'linewidth': 2.2},
    'realistic':   {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2},
    'pessimistic': {'color': 'navy', 'linestyle': ':',  'linewidth': 2.5},
}

In [106]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from pathlib import Path

scenario_order = [s for s in SCENARIOS if s in set(cgw_sim_df.get('scenario', []))]

# Map original scenario names -> LaTeX labels (order must match SCENARIOS order)
scenario_label_map = {
    s: lab for s, lab in zip(scenario_order, scenario_labels)
}

scenario_styles = {
    s: style for s, style in zip(scenario_order, [
        {'color': 'lime', 'linestyle': '-',  'linewidth': 2.2},
        {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2},
        {'color': 'navy', 'linestyle': ':',  'linewidth': 2.5},
    ])
}

SCENARIO_MASS_MAP = {
    'optimistic':  r'Heavy',
    'realistic':   r'Intermediate',
    'pessimistic': r'Light',
}

def _scenario_histogram(
    ax,
    scenario_vals: dict,
    xlabel: str,
    logx: bool = False,
    show_legend: bool = True,
    show_scenario_legend: bool = True,
    vline: float = None,
    vline_label: str = None, 
    loc: str = 'best',
    bbox_anch_coords=None,
    show_ci_band: bool = False,
    ci_alpha: float = 0.08,
    num_bins: int = 28
):
    all_vals = np.concatenate([v for v in scenario_vals.values() if len(v) > 0])
    all_vals = all_vals[np.isfinite(all_vals)]
    if len(all_vals) == 0:
        return
    if logx:
        bins = np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), num_bins)
    else:
        bins = np.linspace(all_vals.min(), all_vals.max(), num_bins)
    if show_ci_band:
        print(f"\n{xlabel}:")
    for s in scenario_order:
        vals = scenario_vals.get(s, np.array([]))
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        style = scenario_styles.get(s, {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5})

        counts, edges = np.histogram(vals, bins=bins)
        ax.hist(
            vals,
            bins=bins,
            histtype='step',
            color=style['color'],
            linestyle=style['linestyle'],
            linewidth=style['linewidth'],
            label=s,
            density=False,
            zorder=2,
        )

        if show_ci_band:
            lo, hi = np.nanpercentile(vals, [2.5, 97.5])
            median = np.nanmedian(vals)
            print(f"  {s}: median = {median:.3g}, 95% CI = [{lo:.3g}, {hi:.3g}]")
            lo, hi = np.nanpercentile(vals, [0.0, 100.0])
            counts, edges = np.histogram(vals, bins=bins)
            lo_idx = np.clip(np.searchsorted(edges, lo, side='right') - 1, 0, len(counts) - 1)
            hi_idx = np.clip(np.searchsorted(edges, hi, side='right') - 1, 0, len(counts) - 1)
            for i in range(lo_idx, hi_idx + 1):
                ax.bar(
                    edges[i],
                    counts[i],
                    width=edges[i + 1] - edges[i],
                    align='edge',
                    color=style['color'],
                    alpha=0.12,
                    zorder=1,
                    linewidth=0,
                )

    ax.set_xlabel(xlabel)
    ax.set_ylabel('Number of simulations')
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    if logx:
        ax.set_xscale('log')
    if vline is not None:
        ax.axvline(vline, color='black', linestyle='--', linewidth=1.5, zorder=3)

    
    if show_legend:
        _CORNER_ANCHORS = {
            'upper left':  (0.0, 1.0),
            'upper right': (1.0, 1.0),
            'lower left':  (0.0, 0.0),
            'lower right': (1.0, 0.0),
        }

        if show_scenario_legend:
            color_handles = [
                mlines.Line2D(
                    [], [],
                    color=scenario_styles[s]['color'],
                    linestyle=scenario_styles[s]['linestyle'],
                    linewidth=scenario_styles[s]['linewidth'],
                    label=SCENARIO_MASS_MAP.get(s, scenario_label_map[s]),
                )
                for s in scenario_order if s in scenario_styles
            ]
        else:
            color_handles = []

        if vline:
            vline_handle = mlines.Line2D(
                [], [],
                color='black',
                linestyle='--',
                linewidth=1.5,
                label=vline_label,
            )
            all_handles = color_handles + [vline_handle]
        else:
            all_handles = color_handles

        if not all_handles:
            return

        legend_kwargs = dict(frameon=False, loc=loc, handlelength=1.6)
        anchor = bbox_anch_coords or _CORNER_ANCHORS.get(loc)
        if anchor is not None:
            legend_kwargs['bbox_to_anchor'] = anchor

        ax.legend(handles=all_handles, **legend_kwargs)


def _save_hist_figure(
    filename,
    scenario_vals,
    xlabel,
    *,
    logx=False,
    vline: float = None,
    vline_label: str = None, 
    figsize=(3.5, 2.8),
    legend_loc='best',
    bbox_anch_coords=None,
    show_legend=True,
    show_scenario_legend=True,
    show_ci_band=False,
):
    fig, ax = plt.subplots(figsize=figsize)
    _scenario_histogram(
        ax,
        scenario_vals=scenario_vals,
        xlabel=xlabel,
        logx=logx,
        show_legend=show_legend,
        show_scenario_legend=show_scenario_legend,
        vline=vline,
        vline_label=vline_label,
        loc=legend_loc,
        bbox_anch_coords=bbox_anch_coords,
        show_ci_band=show_ci_band,
        ci_alpha=0.08,
    )
    fig.tight_layout()
    out_path = Path('figures') / filename
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'  saved {out_path}')



_save_hist_figure(
    'nearest_SMBHB.pdf',
    _extract(cgw_sim_df, 'nearest_DL'),
    r'$D_{\mathrm{L}}$ [Mpc]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper left',
    show_legend=True,
    show_ci_band=True
)
_save_hist_figure(
    'loudest_h0_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_h0'),
    r'$h_0$',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper left',
    show_legend=True,
    show_ci_band=True,
)
_save_hist_figure(
    'loudest_Mc_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_Mc'),
    r'$\mathcal{M}$ [$M_\odot$]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper right',
    show_legend=False,
    show_ci_band=True,
)
_save_hist_figure(
    'loudest_D_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_D'),
    r'$D_{\mathrm{comov}}$ [Mpc]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper left',
    show_legend=False,
    show_ci_band=True,
)
_save_hist_figure(
    'loudest_DL_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_DL'),
    r'$D_{\mathrm{L}}$ [Mpc]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper left',
    show_legend=False,
    show_ci_band=True,
)
_save_hist_figure(
    'loudest_f_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_f'),
    r'$f$ [Hz]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper right',
    show_scenario_legend=False,
    show_ci_band=True,
    vline = 1/(86400*365.25),
    vline_label = '1/yr',
)
_save_hist_figure(
    'loudest_cgw_snr.pdf',
    _extract(cgw_sim_df, 'loudest_cgw_snr'),
    r'$(\mathrm{S/N})_{\mathrm{CW}}$',
    logx=True,
    vline=5.94,
    vline_label='Threshold',
    figsize=(3.5, 2.8),
    legend_loc='upper right',
    show_ci_band=True,
    show_legend=True,
)


$D_{\mathrm{L}}$ [Mpc]:
  optimistic: median = 65.5, 95% CI = [20.8, 139]
  realistic: median = 29.8, 95% CI = [7.39, 70.5]
  pessimistic: median = 20.6, 95% CI = [4.12, 46.5]
  saved figures/nearest_SMBHB.pdf

$h_0$:
  optimistic: median = 4.46e-15, 95% CI = [3.03e-16, 1.84e-14]
  realistic: median = 3.21e-15, 95% CI = [2.85e-16, 1.08e-14]
  pessimistic: median = 1.79e-15, 95% CI = [8.15e-17, 8.44e-15]
  saved figures/loudest_h0_CGW.pdf

$\mathcal{M}$ [$M_\odot$]:
  optimistic: median = 4.68e+09, 95% CI = [1.4e+09, 1.2e+10]
  realistic: median = 3.03e+09, 95% CI = [9.49e+08, 6.49e+09]
  pessimistic: median = 1.52e+09, 95% CI = [5.86e+08, 3.24e+09]
  saved figures/loudest_Mc_CGW.pdf

$D_{\mathrm{comov}}$ [Mpc]:
  optimistic: median = 1.4e+03, 95% CI = [119, 4.69e+03]
  realistic: median = 947, 95% CI = [80.3, 4.57e+03]
  pessimistic: median = 568, 95% CI = [36, 4.1e+03]
  saved figures/loudest_D_CGW.pdf

$D_{\mathrm{L}}$ [Mpc]:
  optimistic: median = 1.87e+03, 95% CI = [122, 1.21e+04]

In [45]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_sim_df[cgw_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_Mc'],
        sub['loudest_h0'],
        c=sub['loudest_D'],
        cmap='viridis_r',
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$D_{\mathrm{comov}}$ [Mpc]')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\mathcal{M}_c\ [M_\odot]$')
    ax.set_title(scenario_label_map.get(s, s))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$')

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_Mc_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_Mc_scatter.pdf')

# also plot h0 vs frequency to check the other axis
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_sim_df[cgw_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_f'],
        sub['loudest_Mc'],
        c=sub['loudest_D'],
        cmap='plasma',
        norm=plt.matplotlib.colors.LogNorm(),
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$\mathcal{M}_c\ [M_\odot]$')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$f$ [Hz]')
    ax.set_title(scenario_label_map.get(s, s))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$')

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_Mc_vs_f_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_f_scatter.pdf')

  saved loudest_h0_vs_Mc_scatter.pdf
  saved loudest_h0_vs_f_scatter.pdf


In [46]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_sim_df[cgw_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_Mc'],
        sub['loudest_h0'],
        c=sub['loudest_D'],
        cmap='viridis_r',
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$D_{\mathrm{comov}}$ [Mpc]')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\mathcal{M}_c\ [M_\odot]$')
    ax.set_title(scenario_label_map.get(s, s))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$')

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_Mc_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_Mc_scatter.pdf')

# also plot h0 vs frequency to check the other axis
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_sim_df[cgw_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_f'],
        sub['loudest_Mc'],
        c=sub['loudest_D'],
        cmap='plasma',
        norm=plt.matplotlib.colors.LogNorm(),
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$\mathcal{M}_c\ [M_\odot]$')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$f$ [Hz]')
    ax.set_title(scenario_label_map.get(s, s))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$')

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_Mc_vs_f_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_f_scatter.pdf')

  saved loudest_h0_vs_Mc_scatter.pdf
  saved loudest_h0_vs_f_scatter.pdf


In [127]:
# Sky maps for loudest-by-CGW binaries per simulation, with pulsar positions overlaid.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patheffects import withStroke

def _wrap_ra(ra_rad):
    ra_shifted = (ra_rad - np.pi) % (2 * np.pi)
    ra_shifted = np.where(ra_shifted > np.pi, ra_shifted - 2 * np.pi, ra_shifted)
    return -ra_shifted

def _galactic_plane():
    ra_ngp  = np.deg2rad(192.859508)
    dec_ngp = np.deg2rad(27.128336)
    l_ncp   = np.deg2rad(122.932)
    l = np.linspace(0, 2 * np.pi, 1000)
    b = np.zeros(1000)
    sin_dec = (np.sin(b) * np.sin(dec_ngp)
               + np.cos(b) * np.cos(dec_ngp) * np.sin(l_ncp - l))
    dec = np.arcsin(np.clip(sin_dec, -1, 1))
    cos_ra_minus_rangp = (np.cos(b) * np.cos(l_ncp - l)) / np.cos(dec)
    sin_ra_minus_rangp = (np.sin(b) * np.cos(dec_ngp)
                          - np.cos(b) * np.sin(dec_ngp) * np.sin(l_ncp - l)) / np.cos(dec)
    ra = (np.arctan2(sin_ra_minus_rangp, cos_ra_minus_rangp) + ra_ngp) % (2 * np.pi)
    ra  = _wrap_ra(ra)
    order = np.argsort(ra)
    ra, dec = ra[order], dec[order]
    gaps = np.where(np.abs(np.diff(ra)) > np.pi / 2)[0] + 1
    ra  = np.insert(ra.astype(float),  gaps, np.nan)
    dec = np.insert(dec.astype(float), gaps, np.nan)
    return ra, dec

if cgw_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot sky maps.')

sky_df = cgw_sim_df[
    np.isfinite(cgw_sim_df['loudest_ra']) & np.isfinite(cgw_sim_df['loudest_dec'])
].copy()
if sky_df.empty:
    raise RuntimeError('No valid RA/Dec values for loudest-by-CGW sources.')

data       = np.load('data/mpta_pulsar_sky_locations.npz')
pulsar_ra  = _wrap_ra(data['ras'])
pulsar_dec = data['decs']
gal_ra, gal_dec = _galactic_plane()

# white stroke used on all axis labels so they read over any background
_STROKE = [withStroke(linewidth=2.5, foreground='white')]
STAR_MARKER_SIZE = 30
def _aitoff_right_rim_axes_x(dec_rad, ax, fig):
    """
    For a given declination, find the axes-fraction x coordinate of the
    right rim of the Aitoff ellipse, so dec labels can hug the curved border.
    The Aitoff projection maps (ra=+pi, dec) to display coords; we convert
    those to axes fraction.
    """
    # In our _wrap_ra convention the right rim is at ra_data = -pi
    # Project that point through the Aitoff transform
    import numpy as np
    # Use a point just inside the rim to avoid projection singularities
    ra_rim = -np.pi * 0.999
    # Aitoff formula for x at the rim given dec
    z     = np.sqrt(1 + np.cos(dec_rad) * np.cos(ra_rim / 2))
    x_display = 2 * np.cos(dec_rad) * np.sin(ra_rim / 2) / z
    y_display = np.sin(dec_rad) / z
    # Convert from display (-2..2, -1..1 for Aitoff) to axes fraction
    # matplotlib's Aitoff axes maps x in [-2,2] -> axes [0,1] and y [-1,1] -> [0,1]
    x_axes = (x_display / 2 + 1) / 2   # maps [-2,2] -> [0,1]  (but flip: right rim is negative ra)
    # right rim with negative ra_rim gives negative x_display, so we want the
    # positive counterpart — the rim is symmetric, just take abs
    x_axes = (abs(x_display) / 2 + 1) / 2  # right rim is positive x side... 
    # Actually recompute with ra_rim positive for the right physical rim
    ra_rim =  np.pi * 0.999
    z      = np.sqrt(1 + np.cos(dec_rad) * np.cos(ra_rim / 2))
    x_disp = 2 * np.cos(dec_rad) * np.sin(ra_rim / 2) / z
    x_axes = (x_disp / (2 * np.sqrt(2)) + 0.5)
    return x_axes

def _style_skyax(ax, fig, ra_label_size=6, dec_label_size=6, dec_nudge=0.028):
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.5)

    # ── RA labels: through the centre along dec = 0 ──────────────────────
    # Back to every hour label (was thinned to every second one).
    ra_hours = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22]
    for h in ra_hours:
        rad = _wrap_ra(np.deg2rad(h * 15))
        ax.annotate(
            f'{h}h',
            xy=(rad, 0),
            xycoords='data',
            ha='center', va='center',
            fontsize=ra_label_size, fontweight='bold', color='dimgrey',
            path_effects=_STROKE,
        )

    # ── Dec labels ─────────────────────────────────────────────────────
    fig.canvas.draw()
    data_to_axes = ax.transData + ax.transAxes.inverted()

    for deg in range(-75, 76, 15):
        dec_rad = np.deg2rad(deg)
        ra_rim  = _wrap_ra(np.deg2rad(359.9))
        x_ax, y_ax = data_to_axes.transform((ra_rim, dec_rad))

        ax.annotate(
            f'{deg}°',
            xy=(x_ax + dec_nudge, y_ax),
            xycoords='axes fraction',
            ha='left', va='center',
            fontsize=dec_label_size, fontweight='bold', color='dimgrey',
            path_effects=_STROKE,
            annotation_clip=False,
        )

# ── ApJ two-column figure width = 3.5 in ─────────────────────────────────────
FIG_W, FIG_H = 3.5, 2.8

# ── Sky map 1: loudest binaries colored by scenario ──────────────────────────
fig = plt.figure(figsize=(FIG_W, FIG_H))
ax  = fig.add_subplot(111, projection='aitoff')
if pulsar_ra.size > 0:
    ax.scatter(
        pulsar_ra, pulsar_dec,
        s=STAR_MARKER_SIZE, marker='*',
        color='yellow',
        edgecolors='black',
        linewidths=0.4,
        alpha=0.95,
        label='Pulsars',
        zorder=4,
    )
for scenario in scenario_order:
    sub = sky_df[sky_df['scenario'] == scenario]
    if sub.empty:
        continue
    style = scenario_styles.get(scenario, {'color': '#888888'})
    ax.scatter(
        _wrap_ra(sub['loudest_ra'].to_numpy()),
        sub['loudest_dec'].to_numpy(),
        s=8, alpha=0.45,
        color=style['color'],
        edgecolors='none',
        label=scenario_label_map.get(scenario, scenario),
        zorder=3,
    )



_style_skyax(ax, fig)
ax.legend(
    loc='lower center', frameon=False,
    bbox_to_anchor=(0.5, -0.42), ncol=2,   # was -0.22
)
fig.tight_layout()
fig.savefig("figures/SNR_loudest_binary_CGW.pdf", dpi=300, bbox_inches='tight')
plt.show()

# ── Sky map 2: loudest binaries colored by CGW SNR ───────────────────────────
fig = plt.figure(figsize=(FIG_W + 0.6, FIG_H))   # slight extra width for colorbar
ax  = fig.add_subplot(111, projection='aitoff')

sc = ax.scatter(
    _wrap_ra(sky_df['loudest_ra'].to_numpy()),
    sky_df['loudest_dec'].to_numpy(),
    c=sky_df['loudest_cgw_snr'].to_numpy(),
    cmap='plasma',
    s=8, alpha=0.8,
    edgecolors='none',
    zorder=3,
)

if pulsar_ra.size > 0:
    ax.scatter(
        pulsar_ra, pulsar_dec,
        s=STAR_MARKER_SIZE, marker='*',
        color='yellow',
        edgecolors='black',
        linewidths=0.4,
        alpha=0.95,
        label='Pulsars',
        zorder=4,
    )

cbar = plt.colorbar(sc, ax=ax, pad=0.05, shrink=0.7, orientation='vertical')
cbar.set_label('CW SNR')
cbar.ax.tick_params(labelsize=7)

_style_skyax(ax, fig)
if pulsar_ra.size > 0:
    ax.legend(
        loc='lower center', frameon=False,
        bbox_to_anchor=(0.5, -0.12), ncol=2,
    )
fig.tight_layout()
fig.savefig("figures/coloured_SNR_loudest_binary_CGW.pdf", dpi=300, bbox_inches='tight')
plt.show()

## Continuous GW (CGW) SNR Diagnostics with Threshold

This section mirrors the CGW diagnostics above, but only keeps the loudest CGW source in each simulation if its CGW SNR is at or above a configurable threshold. Change `CGW_SNR_THRESHOLD` in the next cell whenever you want to tighten or relax the cut.


In [ ]:
"""
Corner plot of loudest CGW event parameters: fGW, h0, SNR, DL, Mc, SGWB SNR

One corner plot per scenario (lower-triangular 6x6 grid):
  - diagonal panels: 1D histogram of each parameter
  - off-diagonal panels: pairwise scatter (log-log)

Uses the rcParams already set by apply_apj_style() -- no manual fontsize
overrides here, so titles/labels/ticks inherit whatever apj_style.py defines.

Assumes these already exist in your notebook/session (and that
apply_apj_style() has already been called):
    cgw_threshold_sim_df  -> DataFrame with columns:
                             'scenario', 'loudest_f', 'loudest_h0',
                             'loudest_DL', 'loudest_cgw_snr', 'loudest_Mc',
                             'sgwb_snr' (attached via _attach_sgwb / merge on
                             'sim_key', per the SGWB-merge cell)
    scenario_order        -> list of scenario keys, in plotting order
    scenario_label_map    -> dict: scenario key -> display label (unused now
                              that the title is removed, kept for reference)
    scenario_styles       -> dict: scenario key -> {'color': ...}
    figure_dir / Path('figures') -> output directory
    APJ_COL_WIDTH_DOUBLE  -> from apj_style.py (7.0)

Note: 'sgwb_snr' is one value per simulation (sim_key), not per-binary, so it
is constant across all of a simulation's binaries. That's expected -- it
still shows real variation (and correlations with e.g. loudest_h0) across
the population of *simulations* within a scenario.
"""

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if cgw_threshold_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot (all simulations below threshold).')

figure_dir = Path('figures')
figure_dir.mkdir(parents=True, exist_ok=True)

# ---- parameter registry -------------------------------------------------
# key -> (column name, axis label)
PARAMS = {
    'f':   ('loudest_f',        r'$f$ [Hz]'),
    'h0':  ('loudest_h0',       r'$h_0$'),
    'snr': ('loudest_cgw_snr',  r'$(\mathrm{S/N})_{\mathrm{CW}}$'),
    'D':   ('loudest_DL',       r'$D_{\mathrm{L}}$ [Mpc]'),
    'Mc':  ('loudest_Mc',       r'$\mathcal{M}_c\ [M_\odot]$'),
    'sgwb': ('sgwb_snr',        r'$(\mathrm{S/N})_{\mathrm{SGWB}}$'),
}
PARAM_KEYS = ['f', 'h0', 'snr', 'D', 'Mc', 'sgwb']  # panel order, top-left to bottom-right
N = len(PARAM_KEYS)

if 'sgwb_snr' not in cgw_threshold_sim_df.columns:
    raise KeyError(
        "cgw_threshold_sim_df has no 'sgwb_snr' column -- run the SGWB-merge "
        "cell (get_sgwb_snr_map / _attach_sgwb) on cgw_threshold_sim_df first."
    )

# ---- diagnostic: check sgwb_snr coverage per scenario --------------------
print('sgwb_snr coverage by scenario:')
for s in scenario_order:
    sub = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == s]
    n_total = len(sub)
    n_notna = sub['sgwb_snr'].notna().sum()
    n_pos = (sub['sgwb_snr'] > 0).sum()
    print(f'  {s}: {n_total} rows, {n_notna} non-NaN, {n_pos} positive')
    if n_pos == 0:
        print(f'    -> WARNING: no usable sgwb_snr values for scenario={s!r}; '
              f'check that sim_key values match between cgw_threshold_sim_df '
              f'and the sgwb_map (scenario_label passed to get_sgwb_snr_map '
              f'may not correspond to {s!r}).')


def _cap_log_ticks(ax, axis='both', numticks=3):
    """Limit major-tick count on a log axis, and suppress minor-tick
    labels entirely (matplotlib auto-labels minor ticks like 2x10^1,
    3x10^1, ... whenever the axis spans less than one decade, regardless
    of the major-locator numticks setting -- that's what was causing the
    overlap even after lowering numticks)."""
    if axis in ('x', 'both'):
        ax.xaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=numticks))
        ax.xaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs='auto', numticks=numticks))
        ax.xaxis.set_minor_formatter(mticker.NullFormatter())
    if axis in ('y', 'both'):
        ax.yaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=numticks))
        ax.yaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs='auto', numticks=numticks))
        ax.yaxis.set_minor_formatter(mticker.NullFormatter())


def plot_corner_scenario(scenario, fname):
    sub = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == scenario]
    if sub.empty:
        print(f'  skipped {fname} (no data for scenario={scenario})')
        return

    color = scenario_styles.get(scenario, {}).get('color', '#888888')

    # Use the global apj_style rcParams as-is (font.size=12, axes.labelsize=12,
    # tick labelsize=10) -- no local shrinking. To fit that at full two-column
    # width, give the grid generous inter-panel spacing and explicit margins
    # (rather than relying on bbox_inches='tight', which crops the saved
    # figure down to its content bbox and was silently shrinking the output
    # below APJ_COL_WIDTH_DOUBLE).
    fig, axes = plt.subplots(N, N, figsize=(APJ_COL_WIDTH_DOUBLE, APJ_COL_WIDTH_DOUBLE))

    for i, ykey in enumerate(PARAM_KEYS):          # row
        for j, xkey in enumerate(PARAM_KEYS):       # col
            ax = axes[i, j]

            if j > i:
                # upper triangle: unused
                ax.axis('off')
                continue

            xcol, xlabel = PARAMS[xkey]
            xvals = sub[xcol]
            xvals = xvals[xvals > 0]

            if i == j:
                # diagonal: 1D histogram
                if len(xvals) == 0:
                    ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                            transform=ax.transAxes, fontsize=9, color='0.5')
                    ax.set_xticks([])
                    ax.set_yticks([])
                else:
                    bins = np.logspace(np.log10(xvals.min()), np.log10(xvals.max()), 25)
                    ax.hist(xvals, bins=bins, color=color, alpha=0.7, histtype='stepfilled')
                    ax.set_xscale('log')
                    ax.set_yticks([])
                    _cap_log_ticks(ax, axis='x')
            else:
                ycol, ylabel = PARAMS[ykey]
                mask = (sub[xcol] > 0) & (sub[ycol] > 0)
                if mask.sum() == 0:
                    ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                            transform=ax.transAxes, fontsize=9, color='0.5')
                    ax.set_xticks([])
                    ax.set_yticks([])
                else:
                    ax.scatter(
                        sub.loc[mask, xcol],
                        sub.loc[mask, ycol],
                        s=4,
                        alpha=0.35,
                        color=color,
                        rasterized=True,
                    )
                    ax.set_xscale('log')
                    ax.set_yscale('log')
                    _cap_log_ticks(ax, axis='both')

            # x tick labels on every column of the bottom row (standard
            # corner-plot convention -- each column is a different
            # quantity/range, so it needs its own numeric ticks).
            ax.tick_params(labelbottom=(i == N - 1))
            if i == N - 1:
                ax.set_xlabel(xlabel, labelpad=4)

            # y tick labels only on the leftmost column, and only for
            # off-diagonal (scatter) panels -- diagonal histograms have no
            # meaningful shared y-axis (counts), so their ticks stay off.
            if i != j:
                ax.tick_params(labelleft=(j == 0))
                if j == 0:
                    ax.set_ylabel(ylabel, labelpad=4)

            for spine in ax.spines.values():
                spine.set_visible(True)

    # Explicit margins + spacing (instead of tight_layout/bbox_inches='tight')
    # so the panel grid actually fills the full APJ_COL_WIDTH_DOUBLE canvas
    # at the paper's normal font sizes, with room reserved on the left/bottom
    # for the rotated y-labels and the x-labels.
    fig.subplots_adjust(left=0.09, right=0.985, bottom=0.08, top=0.985,
                         wspace=0.12, hspace=0.12)
    fig.savefig(figure_dir / fname, dpi=300)
    plt.show()
    plt.close(fig)
    print(f'  saved {fname}')


# ---- generate one corner plot per scenario ------------------------------
for s in scenario_order:
    fname = f'loudest_corner_{s}_thresh.pdf'
    plot_corner_scenario(s, fname)

CGW simulations with thresholded top-source diagnostics: 262


In [49]:
print("SCENARIOS:", SCENARIOS)
print("scenario_styles keys:", list(scenario_styles.keys()))
print("threshold_scenario_order:", threshold_scenario_order)
print("threshold_scenario_styles:", threshold_scenario_styles)
print("scenario_styles:", scenario_styles)

SCENARIOS: ('optimistic', 'realistic', 'pessimistic')
scenario_styles keys: ['optimistic', 'realistic', 'pessimistic']
threshold_scenario_order: ['optimistic', 'realistic', 'pessimistic']
threshold_scenario_styles: {'optimistic': {'color': 'lime', 'linestyle': '-', 'linewidth': 2.2}, 'realistic': {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2}, 'pessimistic': {'color': 'navy', 'linestyle': ':', 'linewidth': 2.5}}
scenario_styles: {'optimistic': {'color': 'lime', 'linestyle': '-', 'linewidth': 2.2}, 'realistic': {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2}, 'pessimistic': {'color': 'navy', 'linestyle': ':', 'linewidth': 2.5}}


In [50]:
# --- SNR threshold detection summary ---
# Count sims above threshold per scenario (already filtered in cgw_threshold_sim_df)
if not cgw_sim_df.empty:
    total_sims_df = (
        cgw_sim_df
        .groupby('scenario')['sim_key']
        .nunique()
        .rename('n_sims_total')
    )

    if not cgw_threshold_sim_df.empty and 'scenario' in cgw_threshold_sim_df.columns:
        above_threshold = (
            cgw_threshold_sim_df
            .groupby('scenario')['sim_key']
            .nunique()
            .rename('n_sims_above_threshold')
        )
    else:
        above_threshold = pd.Series(dtype=int, name='n_sims_above_threshold')

    threshold_summary = (
        total_sims_df
        .to_frame()
        .join(above_threshold, how='left')
        .fillna({'n_sims_above_threshold': 0})
        .assign(n_sims_above_threshold=lambda df: df['n_sims_above_threshold'].astype(int))
        .assign(detection_fraction=lambda df: df['n_sims_above_threshold'] / df['n_sims_total'])
        .reset_index()
    )

    # Preserve scenario ordering
    threshold_summary['scenario'] = pd.Categorical(
        threshold_summary['scenario'],
        categories=[s for s in SCENARIOS if s in threshold_summary['scenario'].values],
        ordered=True,
    )
    threshold_summary = threshold_summary.sort_values('scenario')

    print(f"\nSNR threshold: {CGW_SNR_THRESHOLD}")
    print("Simulations with at least one binary above SNR threshold, by scenario:\n")
    display(threshold_summary)
else:
    print('No `cgw_sim_df` available or it is empty.')



SNR threshold: 5.94
Simulations with at least one binary above SNR threshold, by scenario:



,scenario,n_sims_total,n_sims_above_threshold,detection_fraction
0,optimistic,500,133,0.266
2,realistic,500,78,0.156
1,pessimistic,500,51,0.102


In [105]:
if cgw_threshold_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot (all simulations below threshold).')
figure_dir = Path('figures')
figure_dir.mkdir(parents=True, exist_ok=True)

# 1) nearest SMBHB distance
fig, ax = plt.subplots(figsize=(3.5, 2.8))
_threshold_scenario_histogram(
    ax,
    scenario_vals=_threshold_extract(cgw_threshold_sim_df, 'nearest_DL'),
    xlabel=r'$D_{\rm{L}}$ [Mpc]',
    logx=True,
    show_legend=True,
    loc='upper left',
)
fig.tight_layout()
fig.savefig(figure_dir / 'nearest_SMBHB_threshold.pdf', dpi=300, bbox_inches='tight')
plt.close(fig)

# 2) loudest-by-CGW binary parameters saved individually
threshold_param_specs = [
    ('loudest_h0', r'$h_0$',                                    'loudest_h0_CGW_threshold.pdf', True),
    ('loudest_Mc', r'$\mathcal{M}_c$ [$M_\odot$]',              'loudest_Mc_CGW_threshold.pdf', True),
    ('loudest_DL',  r'$D_{\rm{L}}$ [Mpc]',                   'loudest_DL_CGW_threshold.pdf',  True),
    ('loudest_D',  r'$D_{\rm{comov}}$ [Mpc]',                   'loudest_D_CGW_threshold.pdf',  True),
    ('loudest_f',  r'$f$ [Hz]',                                  'loudest_f_CGW_threshold.pdf',  True),
    ('loudest_z',  r'$z$',                                  'loudest_z_CGW_threshold.pdf',  True),
]
for col, xlabel, filename, use_logx in threshold_param_specs:
    show_scenario_legend = (filename == 'loudest_h0_CGW_threshold.pdf')
    show_vline = (filename == 'loudest_f_CGW_threshold.pdf')
    # legend should render for both the h0 plot (scenario colors) and the f plot (vline)
    show_legend = show_scenario_legend or show_vline

    if show_vline:
        vline = 1 / (86400 * 365.25)
        vline_label = r'1/yr'
        xlim = (1.5e-9, 1.75e-7)
        legend_loc = 'upper right'
    else:
        vline = None
        vline_label = None
        xlim = None
        legend_loc = 'upper left'

    fig, ax = plt.subplots(figsize=(3.5, 2.8))

    _threshold_scenario_histogram(
        ax,
        scenario_vals=_threshold_extract(cgw_threshold_sim_df, col),
        xlabel=xlabel,
        logx=use_logx,
        show_legend=show_legend,
        show_scenario_legend=show_scenario_legend,
        loc=legend_loc,
        show_ci_band=True,
        ci_alpha=0.08,
        vline=vline,
        vline_label=vline_label,
        xlim=xlim,
    )
    fig.tight_layout()
    fig.savefig(figure_dir / filename, dpi=300, bbox_inches='tight')
    plt.close(fig)

# 3) CGW SNR distribution with threshold line
fig, ax = plt.subplots(figsize=(3.5, 2.8))
_threshold_scenario_histogram(
    ax,
    scenario_vals=_threshold_extract(cgw_threshold_sim_df, 'loudest_cgw_snr'),
    xlabel=r'$(\mathrm{S/N})_{\mathrm{CW}}$',
    logx=True,
    show_legend=True,
    vline=CGW_SNR_THRESHOLD,
    loc='upper right',
)
fig.tight_layout()
fig.savefig(figure_dir / 'loudest_cgw_snr_threshold.pdf', dpi=300, bbox_inches='tight')
plt.close(fig)


$h_0$:
  optimistic: median = 6.89e-15, 95% CI = [1.77e-16, 2.28e-14]
  realistic: median = 4.87e-15, 95% CI = [2.36e-16, 1.27e-14]
  pessimistic: median = 3.36e-16, 95% CI = [8.14e-17, 9.84e-15]

$\mathcal{M}_c$ [$M_\odot$]:
  optimistic: median = 4.77e+09, 95% CI = [1.68e+09, 1.19e+10]
  realistic: median = 2.66e+09, 95% CI = [1.26e+09, 7.23e+09]
  pessimistic: median = 1.2e+09, 95% CI = [7.34e+08, 2.71e+09]

$D_{\rm{L}}$ [Mpc]:
  optimistic: median = 1.68e+03, 95% CI = [171, 1.09e+04]
  realistic: median = 922, 95% CI = [69.3, 9.06e+03]
  pessimistic: median = 1.35e+03, 95% CI = [44.3, 1.19e+04]

$D_{\rm{comov}}$ [Mpc]:
  optimistic: median = 1.28e+03, 95% CI = [165, 4.46e+03]
  realistic: median = 780, 95% CI = [68.2, 4.03e+03]
  pessimistic: median = 1.08e+03, 95% CI = [43.9, 4.66e+03]

$f$ [Hz]:
  optimistic: median = 1.26e-08, 95% CI = [2.3e-09, 7.36e-08]
  realistic: median = 1.55e-08, 95% CI = [2.16e-09, 5.83e-08]
  pessimistic: median = 4.64e-09, 95% CI = [2.07e-09, 5.1e-08]

In [52]:
import numpy as np
from scipy.integrate import quad
import matplotlib.pyplot as plt

# --- cosmology (match whatever H0_KMS_MPC / OMEGA_MATTER / OMEGA_LAMBDA your sim used) ---
H0 = 67.66        # km/s/Mpc
Om0 = 0.3111
OL0 = 1 - Om0
c = 299792.458    # km/s

def Hz(z):
    return H0 * np.sqrt(Om0 * (1 + z)**3 + OL0)

def comoving_distance(z):
    integral, _ = quad(lambda zp: c / Hz(zp), 0, z)
    return integral  # Mpc


# --- rebuild the analytic p(z) the sampler drew from: (1+z)^-8/3 * dV_c/dz ---
z_max_used = 2.0   # your actual z_max
z_grid = np.linspace(1e-4, z_max_used, 2000)
chi_grid = np.array([comoving_distance(z) for z in z_grid])   # Mpc
Hz_grid = Hz(z_grid)

dVc_dz = chi_grid**2 / Hz_grid
weight = (1.0 + z_grid)**(-8.0/3.0) * dVc_dz
pdf_analytic = weight / np.trapz(weight, z_grid)   # normalise to a proper pdf


# --- overlay against your actual sampled/injected population ---
z_data = binary_df['z'].dropna().values

fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.hist(z_data, bins=80, density=True, histtype='stepfilled',
        alpha=0.4, color='C0', label='sampled population')
ax.plot(z_grid, pdf_analytic, color='C3', lw=1.8, label='analytic $p(z)$')
ax.axvline(z_max_used, color='k', ls='--', lw=1, label=f'$z_{{max}}$={z_max_used}')
ax.set_xlabel('z')
ax.set_ylabel('probability density')
ax.legend(fontsize=8)

print(f"z_data max: {z_data.max():.5f}  (expect ≈ {z_max_used})")
fig.savefig(figure_dir / 'z_dist.pdf', dpi=300, bbox_inches='tight')
plt.show()


z_data max: 2.00000  (expect ≈ 2.0)


In [53]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq

# --- cosmology (confirm these match what the sim used) ---
H0 = 67.66          # km/s/Mpc  (Planck18 value — change if sim used different)
Om0 = 0.3111        # matter density parameter (Planck18)
OL0 = 1 - Om0        # flat universe -> dark energy density
c = 299792.458      # km/s

def E(z):
    """Dimensionless Hubble parameter H(z)/H0 for flat LCDM."""
    return np.sqrt(Om0 * (1 + z)**3 + OL0)

def comoving_distance(z):
    """Comoving distance in Mpc for a given redshift."""
    integrand = lambda zp: 1.0 / E(zp)
    integral, _ = quad(integrand, 0, z)
    return (c / H0) * integral

def z_from_comoving_distance(dc_target, z_max=20):
    """Invert comoving_distance(z) = dc_target for z, via root-finding."""
    func = lambda z: comoving_distance(z) - dc_target
    return brentq(func, 1e-8, z_max)


# --- run the check ---
dc_col = 'loudest_D'  # <-- confirm this is the right column name

sample = cgw_threshold_sim_df[[dc_col, 'loudest_z']].dropna().sample(
    min(20, len(cgw_threshold_sim_df)), random_state=0
)

for dc, z_reported in sample.itertuples(index=False):
    z_from_dc = z_from_comoving_distance(dc)
    ratio = z_reported / z_from_dc
    print(f"D_C={dc:10.2f} Mpc   z_reported={z_reported:.5f}   "
          f"z_from_D_C={z_from_dc:.5f}   ratio={ratio:.3f}")

D_C=   1334.37 Mpc   z_reported=0.32268   z_from_D_C=0.32708   ratio=0.987
D_C=    861.46 Mpc   z_reported=0.20202   z_from_D_C=0.20447   ratio=0.988
D_C=   5287.20 Mpc   z_reported=1.91896   z_from_D_C=1.98531   ratio=0.967
D_C=   2275.79 Mpc   z_reported=0.59091   z_from_D_C=0.60103   ratio=0.983
D_C=   1037.04 Mpc   z_reported=0.24589   z_from_D_C=0.24901   ratio=0.987
D_C=    591.30 Mpc   z_reported=0.13644   z_from_D_C=0.13799   ratio=0.989
D_C=    948.54 Mpc   z_reported=0.22365   z_from_D_C=0.22642   ratio=0.988
D_C=    499.72 Mpc   z_reported=0.11471   z_from_D_C=0.11598   ratio=0.989
D_C=   1402.71 Mpc   z_reported=0.34081   z_from_D_C=0.34553   ratio=0.986
D_C=    174.84 Mpc   z_reported=0.03943   z_from_D_C=0.03983   ratio=0.990
D_C=   1177.17 Mpc   z_reported=0.28167   z_from_D_C=0.28537   ratio=0.987
D_C=   1252.46 Mpc   z_reported=0.30120   z_from_D_C=0.30522   ratio=0.987
D_C=    189.43 Mpc   z_reported=0.04275   z_from_D_C=0.04319   ratio=0.990
D_C=    409.52 Mpc   z_re

### Corner plots to compare to candidates

In [54]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from pathlib import Path

from apj_style import apply_apj_style, APJ_COL_WIDTH, APJ_COL_WIDTH_DOUBLE

apply_apj_style()

CANDIDATE_COLORS = [
    # '#E63946',  # vivid crimson
    # '#00B4D8',  # bright cyan-blue
    # '#FF9F1C',  # vivid orange
    # '#8338EC',  # bright violet
    # '#06D6A0',  # bright mint/teal green
    '#8B0000',
    '#FFD60A',  # bright yellow
    # '#EF476F',  # bright pink-red
    # '#3A86FF',  # vivid blue
]


def _prep_candidate(cand, cols, log_cols, log_input_cols=()):
    """Convert one candidate dict into (vals, errs) lists aligned with `cols`,
    in the same (possibly log10) space as the plotted data. errs[k] is
    (lo, hi) or None. This is the ONLY place that does the value/unit
    transform for candidates -- panel limits and cross-drawing both read
    from this, so they can't disagree with each other.

    Two distinct cases for a column that's plotted in log10 space:
      - c in log_input_cols: the candidate value/error you supplied is
        ALREADY log10(quantity), e.g. a paper quoting
        log10(Mc/Msun) = 9.67 (+0.11 / -0.34). Used as-is, no transform --
        applying log10() again here would be a bug (this was the actual
        source of the earlier wrong-position issue).
      - c in log_cols but NOT in log_input_cols: the candidate value/error
        you supplied is in LINEAR space (e.g. frequency in Hz) and needs
        converting to log10 space to match the plotted axis, propagating
        the (possibly asymmetric) error through the transform.
    """
    vals, errs = [], []
    for c in cols:
        v = cand.get(c); e = cand.get(f'{c}_err')
        if v is None:
            vals.append(None); errs.append(None); continue

        if c in log_input_cols:
            # already log10-valued; error is already a log-space arm length
            vals.append(v)
            errs.append(None if e is None else ((e, e) if np.isscalar(e) else tuple(e)))

        elif c in log_cols:
            # linear value -> log10, propagate asymmetric error through the transform
            if e is None:
                vals.append(np.log10(v)); errs.append(None)
            else:
                lo, hi = (e, e) if np.isscalar(e) else e
                e_lo = (np.log10(v) - np.log10(v - lo)) if lo else 0.0
                e_hi = (np.log10(v + hi) - np.log10(v)) if hi else 0.0
                vals.append(np.log10(v)); errs.append((e_lo, e_hi))

        else:
            vals.append(v)
            errs.append(None if e is None else ((e, e) if np.isscalar(e) else e))
    return vals, errs


def _var_limits(data_col, hard_bounds, cand_vals_for_col, cand_errs_for_col, pad_frac=0.10):
    """Axis limits for ONE variable (one column index), shared by every
    panel that uses this variable on either axis. Starts from the data
    range, then is explicitly widened so every candidate (value +/- its
    error) is guaranteed to fit -- this is the piece that was missing
    before, causing candidates outside the sampled range to be clipped."""
    lo, hi = data_col.min(), data_col.max()
    span = hi - lo
    pad = pad_frac * span if span > 0 else (0.1 * abs(lo) if lo != 0 else 0.5)
    lo -= pad
    hi += pad

    for v, e in zip(cand_vals_for_col, cand_errs_for_col):
        if v is None:
            continue
        e_lo, e_hi = e if e is not None else (0.0, 0.0)
        cand_lo = v - e_lo
        cand_hi = v + e_hi
        cpad = 0.15 * max(hi - lo, 1e-12)  # small breathing room around the candidate itself
        if cand_lo < lo:
            lo = cand_lo - cpad
        if cand_hi > hi:
            hi = cand_hi + cpad

    hard_lo, hard_hi = hard_bounds
    if hard_lo is not None:
        lo = max(lo, hard_lo)
    if hard_hi is not None:
        hi = min(hi, hard_hi)
    return lo, hi


def _plot_2d_panel_scatter(ax, x, y, base_color, xlim, ylim):
    """Discrete data points only -- no contour, no KDE. At N ~ 50-130 a 2D
    contour implies a resolved density shape the data can't actually
    support; the point cloud itself is the honest representation."""
    ax.scatter(x, y, s=6, color=base_color, alpha=0.55, linewidths=0,
                zorder=4, rasterized=True)  # rasterize the point cloud so
    # the saved PDF stays small/fast even at double-column size; everything
    # else (lines, text) stays vector.
    ax.set_xlim(xlim); ax.set_ylim(ylim)


def _plot_1d_panel(ax, x, base_color, xlim, bins=15):
    ax.hist(x, bins=bins, density=True, histtype='stepfilled',
             color=base_color, alpha=0.4, edgecolor=base_color, linewidth=1.0,
             range=xlim)
    # 68% CI (16/50/84th percentile): dashed-solid-dashed, standard "1-sigma equivalent"
    for q, ls in zip((0.16, 0.5, 0.84), ('--', '-', '--')):
        ax.axvline(np.quantile(x, q), color=base_color, lw=1.0, ls=ls)
    # 95% CI (2.5/97.5th percentile): dotted, thinner/lighter so it doesn't compete visually
    for q in (0.025, 0.975):
        ax.axvline(np.quantile(x, q), color=base_color, lw=0.8, ls=':', alpha=0.8)
    ax.set_xlim(xlim)


def plot_scenario_corner(
    scenario_df, scenario_label, base_color,
    cols=('loudest_Mc', 'loudest_z', 'loudest_f'),
    labels=(r'$\log_{10}(\mathcal{M}/\mathrm{M}_\odot)$', r'$\log_{10}(z)$', r'$\log_{10}(f\,/\,\mathrm{Hz})$'),
    log_cols=('loudest_Mc', 'loudest_z', 'loudest_f'),
    log_input_cols=('loudest_Mc',),  # candidate dict already gives THIS in log10 space
    bounds=None, candidates=None, filename=None, figure_dir=Path('figures'),
    bins=None,               # None -> auto-scaled to N
    column_span='single',    # 'single' -> AAS single-column width (~3.5in), 'double' -> full-page width (~7.1in)
    verbose=True,
):
    """
    Corner plot showing the discrete data points (no 2D contours -- N is too
    small, 50-130, to support a resolved 2D density estimate) plus 1D
    marginal histograms with 16/50/84% quantile lines.
 
    Candidates are drawn as a '+' cross on every 2D panel, centered exactly
    on the candidate's own parameter values (independent of the simulated
    sample), with horizontal/vertical arm lengths equal to the candidate's
    own (possibly asymmetric) uncertainty on each axis -- zero uncertainty
    on an axis collapses that arm to a point, so e.g. a candidate with a
    precisely-measured frequency and an uncertain chirp mass will show as a
    horizontal line, not a full cross. Axis limits are computed to always
    include every candidate (value +/- its error), so a candidate can never
    be clipped off just because it falls outside the simulated sample's range.
    """
    bounds = bounds or {}
    data = scenario_df[list(cols)].dropna()
    ndim = len(cols)
    n = len(data)
 
    # Figure SIZE varies by column span; font point-sizes stay fixed at the
    # apj_style.py rcParams defaults (font sizes are absolute points, not
    # scaled to figure width -- this is the standard AAS convention, same
    # type size across every figure in the paper regardless of its width).
    # Corner plots are square, so we use the column width for both dims
    # rather than apj_style's default (width, 2.8) aspect ratio, which is
    # meant for single-panel plots.
    figsize = (APJ_COL_WIDTH, APJ_COL_WIDTH) if column_span == 'single' \
        else (APJ_COL_WIDTH_DOUBLE, APJ_COL_WIDTH_DOUBLE)
 
    if bins is None:
        bins = max(18, int(np.sqrt(n)))
 
    if verbose:
        warn = ' (small N -- points shown directly, no contour fit)' if n < 150 else ''
        print(f"[{scenario_label}] N = {n}{warn}")
 
    # transform sample data into plotted (possibly log10) space
    plot_data = []
    for c in cols:
        v = data[c].values
        plot_data.append(np.log10(v) if c in log_cols else v)
    plot_data = np.array(plot_data)
 
    # candidates -> vals/errs in the SAME plotted space, computed once
    candidates = candidates or []
    cand_colors = [CANDIDATE_COLORS[k % len(CANDIDATE_COLORS)] for k in range(len(candidates))]
    cand_vals_all, cand_errs_all = [], []
    for cand in candidates:
        v, e = _prep_candidate(cand, cols, log_cols, log_input_cols=log_input_cols)
        cand_vals_all.append(v); cand_errs_all.append(e)
 
    # one shared (lo, hi) per variable/column index, guaranteed to fit the
    # data AND every candidate's value +/- its error
    var_lims = []
    for k, c in enumerate(cols):
        cand_vals_k = [cv[k] for cv in cand_vals_all]
        cand_errs_k = [ce[k] for ce in cand_errs_all]
        var_lims.append(_var_limits(plot_data[k], bounds.get(c, (None, None)),
                                     cand_vals_k, cand_errs_k))
 
    fig, axes = plt.subplots(ndim, ndim, figsize=figsize,
                              gridspec_kw={'wspace': 0.06, 'hspace': 0.06})
 
    for i in range(ndim):
        for j in range(ndim):
            ax = axes[i, j]
            if j > i:
                ax.axis('off'); continue
            if i == j:
                _plot_1d_panel(ax, plot_data[i], base_color, xlim=var_lims[i], bins=bins)
                ax.set_yticks([])
            else:
                _plot_2d_panel_scatter(ax, plot_data[j], plot_data[i], base_color,
                                        xlim=var_lims[j], ylim=var_lims[i])
 
            if i == ndim - 1: ax.set_xlabel(labels[j])
            else: ax.set_xticklabels([])
            if j == 0 and i != 0: ax.set_ylabel(labels[i])
            elif j != 0 and i != j: ax.set_yticklabels([])
 
    if candidates:
        for vals, errs, ccolor in zip(cand_vals_all, cand_errs_all, cand_colors):
            # diagonal: vertical line + shaded uncertainty span, centered on vals[i]
            for i in range(ndim):
                if vals[i] is None:
                    continue
                ax = axes[i, i]
                ax.axvline(vals[i], color=ccolor, lw=1.6, zorder=10, clip_on=False)
                if errs[i] is not None:
                    e_lo, e_hi = errs[i]
                    ax.axvspan(vals[i]-e_lo, vals[i]+e_hi, color=ccolor, alpha=0.25, zorder=5, lw=0)
 
            # off-diagonal: '+' cross centered on (vals[j], vals[i]), arm
            # lengths = that candidate's own (lo, hi) uncertainty on each axis
            for i in range(ndim):
                for j in range(i):
                    if vals[i] is None or vals[j] is None:
                        continue
                    ax = axes[i, j]
                    x0, y0 = vals[j], vals[i]
                    xlo_e, xhi_e = errs[j] if errs[j] is not None else (0.0, 0.0)
                    ylo_e, yhi_e = errs[i] if errs[i] is not None else (0.0, 0.0)
 
                    # horizontal arm (x-axis uncertainty) -- only drawn if
                    # there actually IS uncertainty on this axis
                    if xlo_e > 0 or xhi_e > 0:
                        ax.add_line(Line2D([x0-xlo_e, x0+xhi_e], [y0, y0],
                                            color=ccolor, lw=2.0, zorder=10,
                                            solid_capstyle='butt', clip_on=False))
                    # vertical arm (y-axis uncertainty) -- only drawn if
                    # there actually IS uncertainty on this axis. No arm at
                    # all here (not even a zero-length one) when e.g. f has
                    # no quoted error -- a '+' marker glyph draws its own
                    # vertical/horizontal ticks regardless of data, which
                    # would fake a vertical component that isn't real, so we
                    # use a plain dot for the center point instead (below).
                    if ylo_e > 0 or yhi_e > 0:
                        ax.add_line(Line2D([x0, x0], [y0-ylo_e, y0+yhi_e],
                                            color=ccolor, lw=2.0, zorder=10,
                                            solid_capstyle='butt', clip_on=False))
                    # ALWAYS mark the center point with a plain dot (no
                    # built-in strokes of its own), so the candidate stays
                    # visible even with zero/near-zero error on both axes.
                    ax.plot(x0, y0, marker='o', color=ccolor, ms=3.5,
                             mew=0, zorder=11, clip_on=False)
 
        handles = [Line2D([], [], color=c, lw=2.0, label=cand['label'])
                   for cand, c in zip(candidates, cand_colors)]
        fig.legend(handles=handles, loc='upper right',
                   bbox_to_anchor=(0.98, 0.98), frameon=False)
 
    # CI-style legend sits below the candidate legend (if any). Offset is
    # computed from the number of candidate entries (+1 for the legend's
    # own title padding) so the two legends never overlap regardless of
    # how many candidates were passed in.
    n_cand_rows = len(candidates) if candidates else 0
    row_height = 0.05  # approx. fraction of figure height per legend row
    ci_top = 0.98 - row_height * (n_cand_rows + 1) if candidates else 0.98
    ci_handles = [
        Line2D([], [], color='0.3', lw=1.0, ls='--', label='68% CI'),
        Line2D([], [], color='0.3', lw=0.8, ls=':', label='95% CI'),
    ]
    fig.legend(handles=ci_handles, loc='upper right',
               bbox_to_anchor=(0.98, ci_top), frameon=False)
 
    fig.align_labels()
    fig.tight_layout(pad=0.4, h_pad=0.3, w_pad=0.3)
    fname = filename or f'{scenario_label}_Mc_z_f_corner.pdf'
    figure_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_dir / fname, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return fig
 
 
# ============================================================================
# RUN CORNER PLOTS FOR EACH SCENARIO
# ============================================================================
scenario_col = 'scenario'
BOUNDS = {'loudest_z': (None, np.log10(2.0))}  # sampler upper boundary z <= 2, in log10 space
                                                 # (no lower bound: z=0 -> log10(z) = -inf)
 
# NOTE: loudest_Mc is quoted in the paper as log10(Mc/Msun) directly, with
# ASYMMETRIC errors already in log space, e.g. "9.67 +0.11/-0.34". The err
# tuple convention here is (lo, hi) -- i.e. (0.34, 0.11) for "+0.11/-0.34".
# Since 'loudest_Mc' is in log_input_cols, these are used exactly as given,
# with NO further log10() transform applied.
# loudest_f is still linear (Hz) with no quoted error, and gets log10'd as usual.
candidates = [
    {'label': 'J1536+0441',
     'loudest_Mc': 9.67, 'loudest_Mc_err': (0.34, 0.11),   # log10(Mc/Msun) = 9.67 +0.11/-0.34
     'loudest_z':  0.38894, 'loudest_z_err': (4e-5, 4e-5),
     'loudest_f':  20.84e-9, 'loudest_f_err': None},
    {'label': 'J0729+4008',
     'loudest_Mc': 9.38, 'loudest_Mc_err': (0.30, 0.11),   # log10(Mc/Msun) = 9.38 +0.11/-0.30
     'loudest_z':  0.07404, 'loudest_z_err': (2e-5, 2e-5),
     'loudest_f':  14.36e-9, 'loudest_f_err': None},
]
 
# Set this to 'single' or 'double' to match your ApJ column layout -- flip
# this one line and rerun to compare both.
COLUMN_SPAN = 'single'
 
for scenario_name, sdf in cgw_threshold_sim_df.groupby(scenario_col):
    if scenario_name not in threshold_scenario_styles:
        continue
    style = threshold_scenario_styles[scenario_name]
    plot_scenario_corner(
        sdf,
        scenario_label=scenario_name,
        base_color=style['color'],
        bounds=BOUNDS,
        candidates=candidates,
        column_span=COLUMN_SPAN,
    )

[optimistic] N = 133 (small N -- points shown directly, no contour fit)


/tmp/ipykernel_753460/1177175869.py:283: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(pad=0.4, h_pad=0.3, w_pad=0.3)


[pessimistic] N = 51 (small N -- points shown directly, no contour fit)


/tmp/ipykernel_753460/1177175869.py:283: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(pad=0.4, h_pad=0.3, w_pad=0.3)


[realistic] N = 78 (small N -- points shown directly, no contour fit)


/tmp/ipykernel_753460/1177175869.py:283: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(pad=0.4, h_pad=0.3, w_pad=0.3)


In [147]:
"""
Corner plot of loudest CGW event parameters: fGW, h0, SNR, DL, Mc

One corner plot per scenario (lower-triangular 5x5 grid):
  - diagonal panels: 1D histogram of each parameter
  - off-diagonal panels: pairwise scatter (log-log)

Uses the rcParams already set by apply_apj_style() -- no manual fontsize
overrides here, so titles/labels/ticks inherit whatever apj_style.py defines.

Assumes these already exist in your notebook/session (and that
apply_apj_style() has already been called):
    cgw_threshold_sim_df  -> DataFrame with columns:
                             'scenario', 'loudest_f', 'loudest_h0',
                             'loudest_DL', 'loudest_cgw_snr', 'loudest_Mc'
    scenario_order        -> list of scenario keys, in plotting order
    scenario_label_map    -> dict: scenario key -> display label (unused now
                              that the title is removed, kept for reference)
    scenario_styles       -> dict: scenario key -> {'color': ...}
    figure_dir / Path('figures') -> output directory
    APJ_COL_WIDTH_DOUBLE  -> from apj_style.py (7.0)
"""

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

if cgw_threshold_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot (all simulations below threshold).')

figure_dir = Path('figures')
figure_dir.mkdir(parents=True, exist_ok=True)

# ---- parameter registry -------------------------------------------------
# key -> (column name, axis label)
PARAMS = {
    'f':   ('loudest_f',        r'$f$ [Hz]'),
    'h0':  ('loudest_h0',       r'$h_0$'),
    'snr': ('loudest_cgw_snr',  r'$(\mathrm{S/N})_{\mathrm{CW}}$'),
    'D':   ('loudest_DL',       r'$D_{\mathrm{L}}$ [Mpc]'),
    'Mc':  ('loudest_Mc',       r'$\mathcal{M}\ [M_\odot]$'),
}
PARAM_KEYS = ['f', 'h0', 'snr', 'D', 'Mc']  # panel order, top-left to bottom-right
N = len(PARAM_KEYS)


def plot_corner_scenario(scenario, fname):
    sub = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == scenario]
    if sub.empty:
        print(f'  skipped {fname} (no data for scenario={scenario})')
        return

    color = scenario_styles.get(scenario, {}).get('color', '#888888')

    fig, axes = plt.subplots(N, N, figsize=(APJ_COL_WIDTH_DOUBLE, APJ_COL_WIDTH_DOUBLE))

    for i, ykey in enumerate(PARAM_KEYS):          # row
        for j, xkey in enumerate(PARAM_KEYS):       # col
            ax = axes[i, j]

            if j > i:
                # upper triangle: unused
                ax.axis('off')
                continue

            xcol, xlabel = PARAMS[xkey]
            xvals = sub[xcol]
            xvals = xvals[xvals > 0]

            if i == j:
                # diagonal: 1D histogram
                bins = np.logspace(np.log10(xvals.min()), np.log10(xvals.max()), 25)
                ax.hist(xvals, bins=bins, color=color, alpha=0.7, histtype='stepfilled')
                ax.set_xscale('log')
                ax.set_yticks([])
            else:
                ycol, ylabel = PARAMS[ykey]
                mask = (sub[xcol] > 0) & (sub[ycol] > 0)
                ax.scatter(
                    sub.loc[mask, xcol],
                    sub.loc[mask, ycol],
                    s=4,
                    alpha=0.35,
                    color=color,
                    rasterized=True,
                )
                ax.set_xscale('log')
                ax.set_yscale('log')

            # x tick labels only on the bottom row
            ax.tick_params(labelbottom=(i == N - 1))
            if i == N - 1:
                ax.set_xlabel(xlabel)

            # y tick labels only on the leftmost column, and only for
            # off-diagonal (scatter) panels -- diagonal histograms have no
            # meaningful shared y-axis (counts), so their ticks stay off.
            if i != j:
                ax.tick_params(labelleft=(j == 0))
                if j == 0:
                    ax.set_ylabel(ylabel)

            for spine in ax.spines.values():
                spine.set_visible(True)

    fig.tight_layout(pad=0.4)
    fig.savefig(figure_dir / fname, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  saved {fname}')


# ---- generate one corner plot per scenario ------------------------------
for s in scenario_order:
    fname = f'loudest_corner_{s}_thresh.pdf'
    plot_corner_scenario(s, fname)

  saved loudest_corner_optimistic_thresh.pdf
  saved loudest_corner_realistic_thresh.pdf
  saved loudest_corner_pessimistic_thresh.pdf


In [56]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_Mc'],
        sub['loudest_h0'],
        c=sub['loudest_D'],
        cmap='viridis_r',
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$D_{\mathrm{comov}}$ [Mpc]')
    cb.ax.tick_params(labelsize=7)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\mathcal{M}_c\ [M_\odot]$')
    ax.set_title(scenario_label_map.get(s, s))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$')

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_Mc_scatter_thresh.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_Mc_scatter_thresh.pdf')

# also plot h0 vs frequency to check the other axis
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_f'],
        sub['loudest_h0'],
        c=sub['loudest_Mc'],
        cmap='plasma',
        norm=plt.matplotlib.colors.LogNorm(),
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$\mathcal{M}_c\ [M_\odot]$')
    cb.ax.tick_params(labelsize=7)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$f$ [Hz]')
    ax.set_title(scenario_label_map.get(s, s))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$')

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_f_scatter_thresh.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_f_scatter_thresh.pdf')

  saved loudest_h0_vs_Mc_scatter_thresh.pdf
  saved loudest_h0_vs_f_scatter_thresh.pdf


In [109]:
"""
Pairwise scatter plots of loudest CGW event parameters:
  fGW, h0, SNR, DL (luminosity distance)

Generates all C(4,2) = 6 pairs, x scenarios = one single-panel figure per
(pair, scenario) combination -- no more 1x3 multi-scenario panels.

Assumes these already exist in your notebook/session:
    cgw_threshold_sim_df  -> DataFrame with columns:
                             'scenario', 'loudest_f', 'loudest_h0',
                             'loudest_DL', 'loudest_cgw_snr'
    scenario_order        -> list of scenario keys, in plotting order
    scenario_label_map    -> dict: scenario key -> display label
    figure_dir / Path('figures') -> output directory
"""

from pathlib import Path
from itertools import combinations
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

if cgw_threshold_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot (all simulations below threshold).')

figure_dir = Path('figures')
figure_dir.mkdir(parents=True, exist_ok=True)

# ---- parameter registry -----------------------------------------------
# key -> (column name, axis label, use log scale for this axis?)
PARAMS = {
    'f':   ('loudest_f',        r'$f$ [Hz]',                        True),
    'h0':  ('loudest_h0',       r'$h_0$',                           True),
    'snr': ('loudest_cgw_snr',  r'$(\mathrm{S/N})_{\mathrm{CW}}$',  True),
    'D':   ('loudest_DL',       r'$D_{\mathrm{L}}$ [Mpc]',          True),
}

# For each pair (x, y), choose which remaining parameter colors the points.
# (falls back to the first remaining param if not specified)
COLOR_CHOICE = {
    ('f', 'h0'):   'snr',
    ('f', 'D'):    'h0',
    ('f', 'snr'):  'h0',
    ('h0', 'D'):   'snr',
    ('h0', 'snr'): 'D',
    ('D', 'snr'):  'h0',
}

CMAP_CHOICE = {
    'f':   'plasma',
    'h0':  'viridis_r',
    'snr': 'viridis_r',
    'D':   'viridis_r',
}


def _colnorm(df, ccol):
    """Use LogNorm for strictly-positive, wide-dynamic-range columns."""
    vals = df[ccol]
    vals = vals[vals > 0]
    if len(vals) == 0:
        return None
    ratio = vals.max() / vals.min() if vals.min() > 0 else 1
    return mcolors.LogNorm() if ratio > 50 else None


def plot_pair_single_scenario(xkey, ykey, ckey, scenario, fname):
    xcol, xlabel, xlog = PARAMS[xkey]
    ycol, ylabel, ylog = PARAMS[ykey]
    ccol, clabel, _ = PARAMS[ckey]
    cmap = CMAP_CHOICE.get(ckey, 'viridis_r')

    sub = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == scenario]
    if sub.empty:
        print(f'  skipped {fname} (no data for scenario={scenario})')
        return

    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    norm = _colnorm(sub, ccol)
    sc = ax.scatter(
        sub[xcol],
        sub[ycol],
        c=sub[ccol],
        cmap=cmap,
        norm=norm,
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(clabel)
    cb.ax.tick_params(labelsize=7)
    if xlog:
        ax.set_xscale('log')
    if ylog:
        ax.set_yscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(scenario_label_map.get(scenario, scenario))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

    fig.tight_layout(pad=0.4)
    fig.savefig(figure_dir / fname, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  saved {fname}')


# ---- generate all (pair x scenario) combinations -----------------------
keys = list(PARAMS.keys())  # ['f', 'h0', 'snr', 'D']
for xkey, ykey in combinations(keys, 2):
    ckey = COLOR_CHOICE.get((xkey, ykey))
    if ckey is None:
        # fall back: first remaining param not in the pair
        remaining = [k for k in keys if k not in (xkey, ykey)]
        ckey = remaining[0]
    for s in scenario_order:
        fname = f'loudest_{ykey}_vs_{xkey}_{s}_scatter_thresh.pdf'
        plot_pair_single_scenario(xkey, ykey, ckey, s, fname)

  saved loudest_h0_vs_f_optimistic_scatter_thresh.pdf
  saved loudest_h0_vs_f_realistic_scatter_thresh.pdf
  saved loudest_h0_vs_f_pessimistic_scatter_thresh.pdf
  saved loudest_snr_vs_f_optimistic_scatter_thresh.pdf
  saved loudest_snr_vs_f_realistic_scatter_thresh.pdf
  saved loudest_snr_vs_f_pessimistic_scatter_thresh.pdf
  saved loudest_D_vs_f_optimistic_scatter_thresh.pdf
  saved loudest_D_vs_f_realistic_scatter_thresh.pdf
  saved loudest_D_vs_f_pessimistic_scatter_thresh.pdf
  saved loudest_snr_vs_h0_optimistic_scatter_thresh.pdf
  saved loudest_snr_vs_h0_realistic_scatter_thresh.pdf
  saved loudest_snr_vs_h0_pessimistic_scatter_thresh.pdf
  saved loudest_D_vs_h0_optimistic_scatter_thresh.pdf
  saved loudest_D_vs_h0_realistic_scatter_thresh.pdf
  saved loudest_D_vs_h0_pessimistic_scatter_thresh.pdf
  saved loudest_D_vs_snr_optimistic_scatter_thresh.pdf
  saved loudest_D_vs_snr_realistic_scatter_thresh.pdf
  saved loudest_D_vs_snr_pessimistic_scatter_thresh.pdf


In [58]:
# source-frame frequency vs h0, colored by redshift
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)
for ax, s in zip(axes, scenario_order):
    sub = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    f_src = sub['loudest_f'] * (1.0 + sub['loudest_z'])

    sc = ax.scatter(
        f_src,
        sub['loudest_h0'],
        c=sub['loudest_z'],
        cmap='cividis',
        norm=plt.matplotlib.colors.LogNorm(),
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$z$')
    cb.ax.tick_params(labelsize=7)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$f_{\mathrm{src}}\ [\mathrm{Hz}]$')
    ax.set_title(scenario_label_map.get(s, s))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
axes[0].set_ylabel(r'$h_0$')
fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_fsrc_scatter_thresh.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_fsrc_scatter_thresh.pdf')

  saved loudest_h0_vs_fsrc_scatter_thresh.pdf


In [59]:
# redshift vs chirp mass, colored by h0
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)
for ax, s in zip(axes, scenario_order):
    sub = cgw_sim_df[cgw_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_Mc'],
        sub['loudest_z'],
        c=sub['loudest_h0'],
        cmap='viridis_r',
        norm=plt.matplotlib.colors.LogNorm(),
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$h_0$')
    cb.ax.tick_params(labelsize=7)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\mathcal{M}_c\ [M_\odot]$')
    ax.set_title(scenario_label_map.get(s, s))
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
axes[0].set_ylabel(r'$z$')
fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_z_vs_Mc_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_z_vs_Mc_scatter.pdf')

  saved loudest_z_vs_Mc_scatter.pdf


#### Change in frequency estimate

In [60]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)
for ax, s in zip(axes, scenario_order):
    sub = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})
    delta_f = 0.05 * (sub['loudest_Mc']/10**(8.5))**(5/3) * (sub['loudest_f'] * 1e9 / 50)**(11/3) * (9/10)
    bins = np.logspace(-2, 2, 30)
    ax.hist(
        delta_f,
        bins=bins,
        color=style.get('color', '#888888'),
        alpha=0.7,
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_xlabel(r'$\Delta f$')
    ax.set_title(scenario_label_map.get(s, s))
    ax.set_xscale('log')
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel('Count')
fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_delta_f_hist_thresh.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_delta_f_hist_thresh.pdf')

  saved loudest_delta_f_hist_thresh.pdf


In [61]:
thresholds = [0.01, 0.1, 1.0, 10.0]

for s in scenario_order:
    sub = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == s]
    delta_f = 0.05 * (sub['loudest_Mc']/10**(8.5))**(5/3) * (sub['loudest_f'] * 1e9 / 50)**(11/3) * (9/10)

    print(f"\n{scenario_label_map.get(s, s)} (n={len(delta_f)}):")
    for t in thresholds:
        frac = (delta_f < t).mean()
        print(f"  delta_f < {t:>6}: {frac:6.2%}")


Heavy (n=133):
  delta_f <   0.01: 38.35%
  delta_f <    0.1: 66.92%
  delta_f <    1.0: 89.47%
  delta_f <   10.0: 97.74%

Intermediate (n=78):
  delta_f <   0.01: 37.18%
  delta_f <    0.1: 73.08%
  delta_f <    1.0: 92.31%
  delta_f <   10.0: 98.72%

Light (n=51):
  delta_f <   0.01: 82.35%
  delta_f <    0.1: 94.12%
  delta_f <    1.0: 98.04%
  delta_f <   10.0: 100.00%


In [126]:
# Sky maps for thresholded loudest-by-CGW binaries
threshold_sky_df = cgw_threshold_sim_df[
    np.isfinite(cgw_threshold_sim_df['loudest_ra']) & np.isfinite(cgw_threshold_sim_df['loudest_dec'])
].copy()

if not threshold_sky_df.empty:
    # ── Sky map 1: thresholded loudest binaries colored by scenario ─────────────────
    fig = plt.figure(figsize=(11, 5.5))
    ax  = fig.add_subplot(111, projection='aitoff')

    for scenario in threshold_scenario_order:
        sub = threshold_sky_df[threshold_sky_df['scenario'] == scenario]
        if sub.empty:
            continue
        style = threshold_scenario_styles.get(scenario, {'color': '#888888'})
        ax.scatter(
            _wrap_ra(sub['loudest_ra'].to_numpy()),
            sub['loudest_dec'].to_numpy(),
            s=18,
            alpha=0.45,
            color=style['color'],
            edgecolors='none',
            label=threshold_scenario_label_map.get(scenario, scenario),
            zorder=3,
        )

    if pulsar_ra.size > 0:
        ax.scatter(
            pulsar_ra, pulsar_dec,
            s=25, marker='*',
            color='yellow',
            edgecolors='black',
            alpha=0.9,
            label='Pulsars',
            zorder=4,
        )

    _style_skyax(ax, fig)
    ax.legend(loc='lower center', frameon=False,
              bbox_to_anchor=(0.5, -0.18), ncol=len(threshold_scenario_order) + 2)
    fig.suptitle('Sky Locations of Thresholded Loudest-by-CGW Binaries (SNR ≥ {:.1f})'.format(CGW_SNR_THRESHOLD), y=1.01)
    fig.tight_layout()
    fig.savefig("figures/SNR_loudest_binary_CGW_threshold.pdf", dpi=300, bbox_inches='tight')

    # ── Sky map 2: thresholded loudest binaries colored by CGW SNR ────────────────────
    fig = plt.figure(figsize=(11, 5.5))
    ax  = fig.add_subplot(111, projection='aitoff')

    sc = ax.scatter(
        _wrap_ra(threshold_sky_df['loudest_ra'].to_numpy()),
        threshold_sky_df['loudest_dec'].to_numpy(),
        c=threshold_sky_df['loudest_cgw_snr'].to_numpy(),
        cmap='plasma',
        s=18,
        alpha=0.8,
        edgecolors='none',
        zorder=3,
    )

    if pulsar_ra.size > 0:
        ax.scatter(
            pulsar_ra, pulsar_dec,
            s=25, marker='*',
            color='yellow',
            edgecolors='black',
            linewidths=0.5,
            alpha=0.95,
            label='Pulsars',
            zorder=4,
        )

    cbar = plt.colorbar(sc, ax=ax, pad=0.08, shrink=0.75, orientation='vertical')
    cbar.set_label('Loudest binary CGW SNR')

    _style_skyax(ax, fig)
    if pulsar_ra.size > 0:
        ax.legend(loc='lower center', frameon=False,
                  bbox_to_anchor=(0.5, -0.18), ncol=2)
    fig.suptitle('Sky Locations of Thresholded Loudest-by-CGW Binaries (SNR ≥ {:.1f}, coloured by CGW SNR)'.format(CGW_SNR_THRESHOLD), y=1.01)
    fig.tight_layout()
    fig.savefig("figures/coloured_SNR_loudest_binary_CGW_threshold.pdf", dpi=300, bbox_inches='tight')
else:
    print("No valid RA/Dec values for thresholded loudest-by-CGW sources.")

In [63]:
# ========== Configuration ==========
# Select which scenario to analyze
COMPARISON_SCENARIO = "pessimistic"  # Choose from: "optimistic", "pessimistic", "realistic"

# Toggle redshift filtering (set to None to include all redshifts, or specify max z)
MAX_REDSHIFT = None  # Set to a value like 2.0, 5.0, etc., or None for no cutoff

# Streaming parameters for memory efficiency
ALPHA_POPULATION = 0.08  # Transparency for each population line (lower = less clutter)
N_FREQ_BINS = 30         # Number of frequency bins if not supplied explicitly

# Candidate vertical lines, matching the visualisation.py interface
CANDIDATE_FREQUENCIES = None  # e.g. [1e-7, 2e-7] or None
CANDIDATE_LABELS = None       # e.g. ["Candidate 1", "Candidate 2"] or None
CANDIDATE_MASSES = None       # optional log10(Mtot/Msun) annotations for candidates

print("Configuration (Streaming plot_binaries_vs_frequency style):")
print(f"  Scenario: {COMPARISON_SCENARIO}")
print(f"  Max redshift: {MAX_REDSHIFT if MAX_REDSHIFT is not None else 'None (all)'}")
print(f"  Alpha per population: {ALPHA_POPULATION}")
print(f"  Frequency bins: {N_FREQ_BINS}")
print(f"  Candidate frequencies: {CANDIDATE_FREQUENCIES}")
print("\n✓ Will draw one step-line per population, with low alpha, on a single axes")

Configuration (Streaming plot_binaries_vs_frequency style):
  Scenario: pessimistic
  Max redshift: None (all)
  Alpha per population: 0.08
  Frequency bins: 30
  Candidate frequencies: None

✓ Will draw one step-line per population, with low alpha, on a single axes


In [64]:
def streaming_plot_binaries_vs_frequency(
    files_list,
    scenario_name,
    max_z=None,
    mass_bins=None,
    freq_bins=None,
    candidate_frequencies=None,
    candidate_labels=None,
    candidate_masses=None,
    subset_name='Subset',
    n_freq_bins=30,
    alpha_population=0.08,
):
    """Stream a `plot_binaries_vs_frequency`-style figure without loading all populations.

    Each population is drawn as its own low-alpha step line, with the same mass-bin
    colors, candidate vertical lines, log-log axes, and legend handling as the
    reference implementation in visualisation.py.
    """
    from config import Msun

    if mass_bins is None:
        mass_bins = np.arange(7.5, 10.6, 0.5)

    def _population_mass_and_frequency(population):
        arrays = _population_to_arrays(population)
        if arrays is None or 'f' not in arrays:
            return None, None

        fgw = np.asarray(arrays['f'], dtype=float)
        fgw = fgw[np.isfinite(fgw)]
        if fgw.size == 0:
            return None, None

        if 'Mtot' in arrays and arrays['Mtot'] is not None:
            total_mass = np.asarray(arrays['Mtot'], dtype=float)
        elif 'Mc' in arrays and arrays['Mc'] is not None and 'q' in arrays and arrays['q'] is not None:
            Mc = np.asarray(arrays['Mc'], dtype=float)
            q = np.asarray(arrays['q'], dtype=float)
            total_mass = Mc * (1.0 + q) ** (6.0 / 5.0) / (q ** (3.0 / 5.0))
        else:
            total_mass = None

        if total_mass is not None:
            total_mass = np.asarray(total_mass, dtype=float)
            if total_mass.size != fgw.size:
                n = min(total_mass.size, fgw.size)
                total_mass = total_mass[:n]
                fgw = fgw[:n]
            mask = np.isfinite(total_mass) & np.isfinite(fgw)
            total_mass = total_mass[mask]
            fgw = fgw[mask]
            if total_mass.size == 0:
                return None, None

        return fgw, total_mass

    # Pass 1: determine frequency range without retaining populations.
    if freq_bins is None:
        f_min = np.inf
        f_max = 0.0
        n_populations_seen = 0

        for fp in files_list:
            if infer_scenario(fp) != scenario_name:
                continue
            try:
                payload = _load_payload(fp)
            except Exception:
                continue
            if not isinstance(payload, dict):
                continue
            pops = payload.get('populations', [])
            if not isinstance(pops, list):
                continue

            for entry in pops:
                if not isinstance(entry, dict):
                    continue
                population = entry.get('population')
                if population is None:
                    continue
                if max_z is not None:
                    population = _filter_population_by_redshift(population, max_z)
                fgw, _ = _population_mass_and_frequency(population)
                if fgw is None or fgw.size == 0:
                    continue
                f_min = min(f_min, float(np.nanmin(fgw)))
                f_max = max(f_max, float(np.nanmax(fgw)))
                n_populations_seen += 1

        if n_populations_seen == 0 or not np.isfinite(f_min) or not np.isfinite(f_max):
            print(f"No usable populations found for scenario '{scenario_name}'")
            return None

        freq_bins = np.logspace(np.log10(f_min), np.log10(f_max), n_freq_bins)

    bin_centers = 0.5 * (freq_bins[:-1] + freq_bins[1:])
    n_mass = len(mass_bins) - 1

    colors = [
        "#0072B2",
        "#E69F00",
        "#009E73",
        "#D55E00",
        "#CC79A7",
        "#56B4E9",
        "#000000",
    ]
    linestyles = ['-', '--']
    cand_colors = ['r', 'm', 'c', 'y']

    fig, ax = plt.subplots(figsize=(8, 6))

    # Stream and plot each population immediately.
    n_plotted = 0
    for fp in files_list:
        if infer_scenario(fp) != scenario_name:
            continue
        try:
            payload = _load_payload(fp)
        except Exception:
            continue
        if not isinstance(payload, dict):
            continue
        pops = payload.get('populations', [])
        if not isinstance(pops, list):
            continue

        for entry in pops:
            if not isinstance(entry, dict):
                continue
            population = entry.get('population')
            if population is None:
                continue
            if max_z is not None:
                population = _filter_population_by_redshift(population, max_z)

            fgw, total_mass = _population_mass_and_frequency(population)
            if fgw is None or fgw.size == 0:
                continue

            total_counts, _ = np.histogram(fgw, bins=freq_bins)
            ax.plot(
                bin_centers,
                np.where(total_counts > 0, total_counts, 0.1),
                color='k',
                linewidth=1.0,
                alpha=alpha_population,
                linestyle='-',
                drawstyle='steps-mid',
                label='Total' if n_plotted == 0 else None,
                zorder=1,
            )

            if total_mass is not None and total_mass.size == fgw.size:
                log_mass = np.log10(total_mass)
                for i in range(n_mass):
                    mask = (log_mass >= mass_bins[i]) & (log_mass < mass_bins[i + 1])
                    if not np.any(mask):
                        continue
                    counts, _ = np.histogram(fgw[mask], bins=freq_bins)
                    ax.plot(
                        bin_centers,
                        np.where(counts > 0, counts, 0.1),
                        color=colors[i % len(colors)],
                        linewidth=1.6,
                        alpha=alpha_population,
                        linestyle=linestyles[i % len(linestyles)],
                        drawstyle='steps-mid',
                        label=(r"$%.1f < \log_{10}\!\left(M_{\rm tot}/\mathrm{M}_\odot\right) < %.1f$" % (mass_bins[i], mass_bins[i + 1])) if n_plotted == 0 else None,
                        zorder=2,
                    )
            n_plotted += 1

    if n_plotted == 0:
        print(f"No populations found for scenario '{scenario_name}'")
        return None

    # Candidate frequencies match the reference interface.
    if candidate_frequencies is not None:
        if candidate_labels is None:
            candidate_labels = [f"Candidate {j + 1}" for j in range(len(candidate_frequencies))]
        if candidate_masses is None:
            candidate_masses = [None] * len(candidate_frequencies)

        for i, (f0, label, mass_val) in enumerate(zip(candidate_frequencies, candidate_labels, candidate_masses)):
            if mass_val is not None:
                label = (
                    rf"{label} "
                    rf"$\left[\log_{{10}}\!\left(M_{{\rm tot}}/M_\odot\right)={mass_val:.2f}\right]$"
                )
            ax.axvline(
                f0,
                color=cand_colors[i % len(cand_colors)],
                linestyle='-',
                lw=3,
                alpha=1,
                label=label,
                zorder=5,
            )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Gravitational Wave Frequency [Hz]', fontweight='bold')
    ax.set_ylabel('Number of binaries', fontweight='bold')
    ax.set_title(f'Binaries by GW frequency ({subset_name})', fontweight='bold')

    from matplotlib.ticker import LogLocator
    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax.tick_params(which='both', direction='in', top=True, right=True)

    ax.set_xlim(freq_bins[0], freq_bins[-1] * 1.2)
    ax.set_ylim(0.2, ax.get_ylim()[1])

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), frameon=True, edgecolor='black')

    plt.tight_layout()
    return fig


In [65]:
# Execute the streaming comparison with the exact plot style from visualisation.py
print("\n" + "=" * 70)
print(f"Generating streaming multi-simulation frequency comparison for {COMPARISON_SCENARIO}...")
if MAX_REDSHIFT is not None:
    print(f"Redshift filter: z ≤ {MAX_REDSHIFT}")
print("=" * 70)

fig = streaming_plot_binaries_vs_frequency(
    result_files_CGW,
    scenario_name=COMPARISON_SCENARIO,
    max_z=MAX_REDSHIFT,
    mass_bins=np.arange(7.5, 10.6, 0.5),
    freq_bins=None,
    candidate_frequencies=CANDIDATE_FREQUENCIES,
    candidate_labels=CANDIDATE_LABELS,
    candidate_masses=CANDIDATE_MASSES,
    subset_name=COMPARISON_SCENARIO,
    n_freq_bins=N_FREQ_BINS,
    alpha_population=ALPHA_POPULATION,
)

if fig is not None:
    fname = f"figures/multi_sim_frequency_comparison_{COMPARISON_SCENARIO}"
    if MAX_REDSHIFT is not None:
        fname += f"_z{MAX_REDSHIFT}"
    fname += ".pdf"
    fig.savefig(fname, dpi=300, bbox_inches='tight')
    print(f"\n✓ Saved to {fname}")
    plt.close(fig)
else:
    print(f"\n✗ Could not generate plot for {COMPARISON_SCENARIO}")


Generating streaming multi-simulation frequency comparison for pessimistic...
No usable populations found for scenario 'pessimistic'

✗ Could not generate plot for pessimistic


In [66]:
# Summary printout: counts and min/max for CGW SNR histograms
import numpy as np

print('\n=== CGW SNR Summary ===')
if 'cgw_sim_df' in globals() and not cgw_sim_df.empty:
    col = 'loudest_cgw_snr_baseline_forecast'
    scenarios = sorted(cgw_sim_df['scenario'].unique())
    total_vals = []
    for s in scenarios:
        arr = cgw_sim_df.loc[cgw_sim_df['scenario'] == s, col].to_numpy(dtype=float)
        finite = arr[np.isfinite(arr)]
        total_vals.append(finite)
        if finite.size:
            print(f"Scenario '{s}': plotted {finite.size} SNR values; min={finite.min():.3g}, max={finite.max():.3g}")
        else:
            print(f"Scenario '{s}': plotted 0 SNR values")
    all_vals = np.concatenate([a for a in total_vals if a.size > 0]) if total_vals else np.array([])
    if all_vals.size:
        print(f"Global SNR: n={all_vals.size}, min={all_vals.min():.3g}, max={all_vals.max():.3g}")
    else:
        print('Global SNR: no finite values to plot')
else:
    print('No `cgw_sim_df` available or it is empty.')

print('\n=== Thresholded CGW SNR Summary ===')
if 'cgw_threshold_sim_df' in globals() and not cgw_threshold_sim_df.empty:
    col = 'loudest_cgw_snr_baseline_forecast'
    scenarios = sorted(cgw_threshold_sim_df['scenario'].unique())
    total_vals = []
    for s in scenarios:
        arr = cgw_threshold_sim_df.loc[cgw_threshold_sim_df['scenario'] == s, col].to_numpy(dtype=float)
        finite = arr[np.isfinite(arr)]
        total_vals.append(finite)
        if finite.size:
            print(f"Scenario '{s}' (thresholded): plotted {finite.size} SNR values; min={finite.min():.3g}, max={finite.max():.3g}")
        else:
            print(f"Scenario '{s}' (thresholded): plotted 0 SNR values")
    all_vals = np.concatenate([a for a in total_vals if a.size > 0]) if total_vals else np.array([])
    if all_vals.size:
        print(f"Global thresholded SNR: n={all_vals.size}, min={all_vals.min():.3g}, max={all_vals.max():.3g}")
    else:
        print('Global thresholded SNR: no finite values to plot')
else:
    print('No `cgw_threshold_sim_df` available or it is empty.')


=== CGW SNR Summary ===
Scenario 'optimistic': plotted 500 SNR values; min=1.5, max=67.7
Scenario 'pessimistic': plotted 500 SNR values; min=0.713, max=152
Scenario 'realistic': plotted 500 SNR values; min=1.43, max=103
Global SNR: n=1500, min=0.713, max=152

=== Thresholded CGW SNR Summary ===
Scenario 'optimistic' (thresholded): plotted 133 SNR values; min=5.94, max=67.7
Scenario 'pessimistic' (thresholded): plotted 51 SNR values; min=5.96, max=152
Scenario 'realistic' (thresholded): plotted 78 SNR values; min=5.97, max=103
Global thresholded SNR: n=262, min=5.94, max=152


## Cumulative SNR as a function of the number of pulsars

In [67]:
# Cumulative SNR contribution from the loudest binary in each simulation.
#
# Each curve is built from the saved `top_cgw_breakdowns` metadata in
# `summary.pkl.gz`, where `per_pulsar_rho_sq` stores the per-pulsar SNR^2
# contribution for the loudest binary in that simulation.
#
# Layout: forest-plot style, ONE single panel per scenario (no sub-columns).
# Pulsars listed top-to-bottom ranked by median per-pulsar rho^2 contribution.
# For each pulsar, a filled circle shows the median cumulative SNR fraction
# and horizontal error bars span the 95% credible interval across simulations.
# No title. One individual PDF is saved per scenario.
#
# --- Sizing: matched to ApJ's double-column page ----------------------------
# ApJ (and AASTeX in general) is a two-column journal. A wide figure like this
# is meant to span BOTH text columns via \begin{figure*}...\end{figure*},
# where \linewidth then resolves to the full text width (~7.1in) rather than
# a single column (~3.5in). PAGE_WIDTH_IN is set to that full-text-width value
# so the PDF is generated at the actual physical size it will be placed at --
# that's what avoids LaTeX rescaling it and blowing the height past the page.
#
# Height is capped at MAX_FIG_HEIGHT_IN (comfortably under a full printed
# page so the float + caption still fit). If a scenario has more pulsars than
# fit in that height at a legible row spacing, the remainder is truncated to
# a single "..." row rather than growing the figure taller or splitting into
# extra side-by-side columns.

import gzip
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D


# ---------------------------------------------------------------------------
# Sizing / font configuration
# ---------------------------------------------------------------------------
PAGE_WIDTH_IN     = 7.1   # ApJ full double-column text width -- use \begin{figure*}
                          # in LaTeX so \linewidth actually resolves to this
MAX_FIG_HEIGHT_IN = 8.4   # inches -- near-full ApJ page height budget for the float

MARGIN_HEIGHT = 0.55   # inches reserved for the x-axis label / legend (no title)
ROW_HEIGHT    = 0.15   # inches per pulsar row


MAX_ROWS = max(5, int((MAX_FIG_HEIGHT_IN - MARGIN_HEIGHT) / ROW_HEIGHT))


# ---------------------------------------------------------------------------
# Path / data helpers
# ---------------------------------------------------------------------------

def _resolve_summary_path(source_value):
    if source_value is None or pd.isna(source_value):
        return None
    source_path = Path(str(source_value))
    if source_path.is_dir():
        candidate = source_path / 'summary.pkl.gz'
        return candidate if candidate.is_file() else None
    if source_path.name == 'summary.pkl.gz' and source_path.is_file():
        return source_path
    candidate = source_path.parent / 'summary.pkl.gz'
    return candidate if candidate.is_file() else None


def _load_loudest_breakdown(summary_path, scenario_name):
    summary_path = Path(summary_path)
    if not summary_path.is_file():
        return None

    with gzip.open(summary_path, 'rb') as fh:
        payload = pickle.load(fh)

    top_breakdowns = payload.get('meta', {}).get('top_cgw_breakdowns', []) or []
    if not top_breakdowns:
        return None

    loudest = max(top_breakdowns, key=lambda item: float(item.get('cgw_snr', np.nan)))
    per_pulsar = loudest.get('per_pulsar_rho_sq', {}) or {}
    if not per_pulsar:
        return None

    ordered = sorted(per_pulsar.items(), key=lambda kv: float(kv[1]), reverse=True)
    rho_sq = np.array([max(float(v), 0.0) for _, v in ordered], dtype=float)
    total = np.sum(rho_sq)
    cumulative_snr = np.sqrt(np.cumsum(rho_sq)) / np.sqrt(total) if total > 0 else np.zeros_like(rho_sq)

    return {
        'scenario':       scenario_name,
        'summary_path':   summary_path,
        'sim_key':        summary_path.parent.name,
        'loudest_cgw_snr': float(loudest.get('cgw_snr', np.nan)),
        'pulsar_names':   [name for name, _ in ordered],
        'rho_sq':         rho_sq,
        'cumulative_snr': cumulative_snr,
    }


def _pad_curves(curves):
    if not curves:
        return np.empty((0, 0), dtype=float)
    max_len = max(len(c) for c in curves)
    padded = np.full((len(curves), max_len), np.nan, dtype=float)
    for i, c in enumerate(curves):
        if len(c) == 0:
            continue
        padded[i, :len(c)] = c
        padded[i, len(c):] = c[-1]
    return padded


# ---------------------------------------------------------------------------
# Forest-plot helpers
# ---------------------------------------------------------------------------

def _median_pulsar_order(scenario_rows, n_pulsars):
    """
    Return a list of exactly n_pulsars unique pulsar names, ordered by each
    pulsar's median rank across simulations (rank 1 = highest rho^2 contributor).
    """
    all_name_lists = [row for row in scenario_rows['pulsar_names'].tolist() if row]
    if not all_name_lists:
        return [f'Rank {i + 1}' for i in range(n_pulsars)]

    all_pulsars = set()
    for name_list in all_name_lists:
        all_pulsars.update(name_list)

    worst_rank = max(len(nl) for nl in all_name_lists)
    rank_records = {p: [] for p in all_pulsars}
    for name_list in all_name_lists:
        seen = {name: idx for idx, name in enumerate(name_list)}
        for p in all_pulsars:
            rank_records[p].append(seen.get(p, worst_rank))

    median_rank = {p: np.median(ranks) for p, ranks in rank_records.items()}

    ordered = sorted(all_pulsars, key=lambda p: median_rank[p])
    ordered = ordered[:n_pulsars]

    while len(ordered) < n_pulsars:
        ordered.append(f'Rank {len(ordered) + 1}')

    return ordered


def _build_forest_data(padded, bands):
    """
    Returns arrays of length n_pulsars:
      - median_cumul, lo95, hi95 : cumulative-SNR median / 95% CI per rank
      - marginal                 : median *marginal* SNR fraction (this
                                    pulsar's own contribution, not cumulative)
                                    -- drives the fade-out alpha
    """
    p50   = bands['p50']
    p2_5  = bands['p2_5']
    p97_5 = bands['p97_5']
    marginal = np.concatenate([[p50[0]], np.diff(p50)])
    return p50, p2_5, p97_5, marginal


def _alpha_ramp(marginal, min_alpha=0.15, threshold=0.003):
    """Full opacity while marginal >= threshold, then linearly ramps to min_alpha."""
    alphas = np.ones(len(marginal))
    significant = marginal >= threshold
    if not np.any(~significant):
        return alphas
    first_insig = np.argmax(~significant)
    n_tail = len(marginal) - first_insig
    if n_tail > 0:
        ramp = np.linspace(1.0, min_alpha, n_tail)
        alphas[first_insig:] = ramp
    return alphas


def _truncate_for_display(pulsar_names, median_cumul, lo95, hi95, alphas, max_rows):
    """
    If there are more pulsars than max_rows, keep the top (max_rows - 1)
    contributors and replace the remainder with a single "..." row (label
    only, no marker/error bar) instead of growing the figure taller.
    """
    n = len(pulsar_names)
    if n <= max_rows:
        return pulsar_names, median_cumul, lo95, hi95, alphas, False

    keep = max_rows - 1
    names = list(pulsar_names[:keep]) + ['...']

    def _pad(arr):
        return np.concatenate([np.asarray(arr)[:keep], [np.nan]])

    return names, _pad(median_cumul), _pad(lo95), _pad(hi95), _pad(alphas), True


def _draw_forest(
    ax, pulsar_names, median_cumul, lo95, hi95, alphas, color,
):
    """
    Draw the single forest-plot panel onto ax. Rows with NaN median_cumul
    (the truncation "..." row) get a label only -- no marker or error bar.
    """
    n = len(pulsar_names)
    y_pos = np.arange(n, dtype=float)

    for i in range(n):
        if np.isnan(median_cumul[i]):
            continue
        a = float(alphas[i])
        ax.plot(
            [lo95[i], hi95[i]], [y_pos[i], y_pos[i]],
            color=color, alpha=a, linewidth=1.2, solid_capstyle='butt',
        )
        cap_size = 0.3
        for x_cap in (lo95[i], hi95[i]):
            ax.plot(
                [x_cap, x_cap],
                [y_pos[i] - cap_size, y_pos[i] + cap_size],
                color=color, alpha=a, linewidth=1.2,
            )
        ax.scatter(
            median_cumul[i], y_pos[i],
            s=18, color=color, alpha=a, zorder=3, linewidths=0,
        )

    ax.axvline(1.0, color='gray', linewidth=0.6, linestyle=':', alpha=0.4)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(pulsar_names)
    ax.set_ylim(n - 0.5, -0.5)   # rank 1 at top
    ax.set_xlim(-0.02, 1.08)
    ax.set_xlabel(r'Cumulative fraction of total (S/N)$_{s}$')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.tick_params(axis='x', labelsize=tick_fontsize)
    ax.grid(True, axis='x', alpha=0.2, linewidth=0.6)
    ax.tick_params(axis='y', length=0)


# ---------------------------------------------------------------------------
# Main -- one individual PDF per scenario, single panel, no title
# ---------------------------------------------------------------------------

if 'cgw_sim_df' not in globals() or cgw_sim_df.empty:
    print('No cgw_sim_df available yet, so the cumulative SNR plots were skipped.')
else:
    breakdown_records = []
    for _, row in cgw_sim_df.iterrows():
        summary_path = _resolve_summary_path(row.get('source_file'))
        if summary_path is None:
            continue
        scenario_name = str(row.get('scenario', 'unknown'))
        record = _load_loudest_breakdown(summary_path, scenario_name)
        if record is None:
            continue
        record['sim_key']     = str(row.get('sim_key', record['sim_key']))
        record['source_file'] = str(row.get('source_file', summary_path))
        breakdown_records.append(record)

    breakdown_df = pd.DataFrame(breakdown_records)

    if breakdown_df.empty:
        print('No loudest-binary breakdowns were found in the loaded summaries.')
    else:
        scenario_values = breakdown_df['scenario'].dropna().unique().tolist()
        if 'scenario_order' in globals():
            scenarios = [s for s in scenario_order if s in scenario_values]
            scenarios.extend(s for s in scenario_values if s not in scenarios)
        else:
            scenarios = sorted(scenario_values)

        color_map = {
            s: color
            for s, color in zip(scenarios, ['lime', 'magenta', 'navy'])
        }

        print(f'Loaded loudest-binary breakdowns for {len(breakdown_df)} simulations.')
        print('Breakdowns by scenario:')
        for s in scenarios:
            print(f'  {s}: {int((breakdown_df["scenario"] == s).sum())}')
        print(f'Each figure: single panel, {PAGE_WIDTH_IN}in wide x up to '
              f'{MAX_FIG_HEIGHT_IN}in tall (up to {MAX_ROWS} pulsars before truncation).')

        figure_dir = Path('figures')
        figure_dir.mkdir(parents=True, exist_ok=True)

        

        for scenario in scenarios:
            scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario].copy()
            curves = [c for c in scenario_rows['cumulative_snr'].tolist() if len(c) > 0]
            if not curves:
                continue

            padded         = _pad_curves(curves)
            n_pulsars_full = padded.shape[1]
            bands = {
                'p2_5':  np.nanpercentile(padded,  2.5, axis=0),
                'p50':   np.nanmedian(padded, axis=0),
                'p97_5': np.nanpercentile(padded, 97.5, axis=0),
            }

            median_cumul, lo95, hi95, marginal = _build_forest_data(padded, bands)
            alphas       = _alpha_ramp(marginal)
            pulsar_names = _median_pulsar_order(scenario_rows, n_pulsars_full)
            color        = color_map[scenario]
            
            legend_elements = [
                Line2D([0], [0], marker='o', color=color, markersize=5,
                       linewidth=0, label='Median'),
                Line2D([0], [0], color=color, linewidth=1.2,
                       label='95% credible interval'),
            ]

            pulsar_names, median_cumul, lo95, hi95, alphas, truncated = _truncate_for_display(
                pulsar_names, median_cumul, lo95, hi95, alphas, MAX_ROWS,
            )
            n_pulsars = len(pulsar_names)

            fig_height = n_pulsars * ROW_HEIGHT + MARGIN_HEIGHT
            fig, ax = plt.subplots(figsize=(PAGE_WIDTH_IN, fig_height))

            _draw_forest(ax, pulsar_names, median_cumul, lo95, hi95, alphas, color)

            ax.legend(handles=legend_elements, frameon=False,
                      loc='lower left')
            fig.tight_layout()

            out_path = figure_dir / f'cgw_cumulative_snr_forest_{scenario}.pdf'
            fig.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close(fig)

            note = (f' (showing top {n_pulsars - 1} of {n_pulsars_full} pulsars, '
                     f'rest collapsed to "...")') if truncated else ''
            print(f'  saved {out_path}  ({PAGE_WIDTH_IN}in x {fig_height:.2f}in){note}')

Loaded loudest-binary breakdowns for 1500 simulations.
Breakdowns by scenario:
  optimistic: 500
  realistic: 500
  pessimistic: 500
Each figure: single panel, 7.1in wide x up to 8.4in tall (up to 52 pulsars before truncation).


NameError: name 'tick_fontsize' is not defined

In [ ]:
# Cumulative SNR contribution from the loudest binary in each simulation.
#
# Each curve is built from the saved `top_cgw_breakdowns` metadata in
# `summary.pkl.gz`, where `per_pulsar_rho_sq` stores the per-pulsar SNR^2
# contribution for the loudest binary in that simulation.
#
# Layout: forest-plot style sized for a full-page ApJ single-column figure
# (7.1 x 9.0 inches, 300 dpi). All pulsars are listed top-to-bottom ranked
# by median per-pulsar rho^2. Each row shows a median circle and a 95%
# credible-interval error bar. Font size is derived from figure height so
# rows never overlap.

import gzip
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D


# ---------------------------------------------------------------------------
# ApJ figure geometry  (single-column full-page)
# ---------------------------------------------------------------------------
APJ_WIDTH   = 7.1    # inches  (single-column text width)
APJ_HEIGHT  = 9.0    # inches  (safe full-page plot height)
APJ_DPI     = 300

# Fraction of the figure height dedicated to the plot area (axes), after
# reserving space for title (~0.25 in), xlabel (~0.30 in) and legend (~0.20 in).
_RESERVED_INCHES = 0.75   # title + xlabel + legend + breathing room


# ---------------------------------------------------------------------------
# Path / data helpers
# ---------------------------------------------------------------------------

def _resolve_summary_path(source_value):
    if source_value is None or pd.isna(source_value):
        return None
    source_path = Path(str(source_value))
    if source_path.is_dir():
        candidate = source_path / 'summary.pkl.gz'
        return candidate if candidate.is_file() else None
    if source_path.name == 'summary.pkl.gz' and source_path.is_file():
        return source_path
    candidate = source_path.parent / 'summary.pkl.gz'
    return candidate if candidate.is_file() else None


def _load_loudest_breakdown(summary_path, scenario_name):
    summary_path = Path(summary_path)
    if not summary_path.is_file():
        return None

    with gzip.open(summary_path, 'rb') as fh:
        payload = pickle.load(fh)

    top_breakdowns = payload.get('meta', {}).get('top_cgw_breakdowns', []) or []
    if not top_breakdowns:
        return None

    loudest = max(top_breakdowns, key=lambda item: float(item.get('cgw_snr', np.nan)))
    per_pulsar = loudest.get('per_pulsar_rho_sq', {}) or {}
    if not per_pulsar:
        return None

    ordered = sorted(per_pulsar.items(), key=lambda kv: float(kv[1]), reverse=True)
    rho_sq  = np.array([max(float(v), 0.0) for _, v in ordered], dtype=float)
    total   = np.sum(rho_sq)
    cumulative_snr = (
        np.sqrt(np.cumsum(rho_sq)) / np.sqrt(total) if total > 0
        else np.zeros_like(rho_sq)
    )

    return {
        'scenario':        scenario_name,
        'summary_path':    summary_path,
        'sim_key':         summary_path.parent.name,
        'loudest_cgw_snr': float(loudest.get('cgw_snr', np.nan)),
        'pulsar_names':    [name for name, _ in ordered],
        'rho_sq':          rho_sq,
        'cumulative_snr':  cumulative_snr,
    }


def _pad_curves(curves):
    if not curves:
        return np.empty((0, 0), dtype=float)
    max_len = max(len(c) for c in curves)
    padded  = np.full((len(curves), max_len), np.nan, dtype=float)
    for i, c in enumerate(curves):
        if len(c) == 0:
            continue
        padded[i, :len(c)] = c
        padded[i, len(c):] = c[-1]
    return padded


# ---------------------------------------------------------------------------
# Pulsar ordering — unique, by median rank across simulations
# ---------------------------------------------------------------------------

def _median_pulsar_order(scenario_rows, n_pulsars):
    """
    Return a list of exactly n_pulsars *unique* pulsar names ordered by each
    pulsar's median rank across simulations (rank 1 = highest rho^2).

    For every pulsar seen in any simulation we record the 0-based rank at
    which it appears (or worst_rank if absent). We then sort by median of
    those ranks, guaranteeing no duplicates and full coverage.
    """
    all_name_lists = [row for row in scenario_rows['pulsar_names'].tolist() if row]
    if not all_name_lists:
        return [f'Rank {i + 1}' for i in range(n_pulsars)]

    all_pulsars = set()
    for nl in all_name_lists:
        all_pulsars.update(nl)

    worst_rank   = max(len(nl) for nl in all_name_lists)
    rank_records = {p: [] for p in all_pulsars}
    for nl in all_name_lists:
        seen = {name: idx for idx, name in enumerate(nl)}
        for p in all_pulsars:
            rank_records[p].append(seen.get(p, worst_rank))

    median_rank = {p: np.median(ranks) for p, ranks in rank_records.items()}
    ordered     = sorted(all_pulsars, key=lambda p: median_rank[p])[:n_pulsars]

    while len(ordered) < n_pulsars:
        ordered.append(f'Rank {len(ordered) + 1}')

    return ordered


# ---------------------------------------------------------------------------
# Forest-plot helpers
# ---------------------------------------------------------------------------

def _build_forest_data(padded, bands):
    p50   = bands['p50']
    p2_5  = bands['p2_5']
    p97_5 = bands['p97_5']
    marginal = np.concatenate([[p50[0]], np.diff(p50)])
    return p50, p2_5, p97_5, marginal


def _alpha_ramp(marginal, min_alpha=0.15, threshold=0.003):
    alphas      = np.ones(len(marginal))
    significant = marginal >= threshold
    if not np.any(~significant):
        return alphas
    first_insig = int(np.argmax(~significant))
    n_tail      = len(marginal) - first_insig
    if n_tail > 0:
        alphas[first_insig:] = np.linspace(1.0, min_alpha, n_tail)
    return alphas


def _fontsize_for_height(n_rows, axes_height_inches):
    """
    Choose the largest integer font size (points) such that n_rows of text
    fit within axes_height_inches without vertical overlap.
    One point = 1/72 inch; add ~20 % leading.
    """
    max_pt = (axes_height_inches * 72.0) / (n_rows * 1.20)
    return float(np.clip(max_pt, 4.0, 8.0))


def _draw_forest(ax, pulsar_names, median_cumul, lo95, hi95, alphas, color,
                 fontsize):
    n     = len(pulsar_names)
    y_pos = np.arange(n, dtype=float)

    cap_size = 0.30

    for i in range(n):
        a = float(alphas[i])
        ax.plot(
            [lo95[i], hi95[i]], [y_pos[i], y_pos[i]],
            color=color, alpha=a, linewidth=0.9, solid_capstyle='butt',
        )
        for x_cap in (lo95[i], hi95[i]):
            ax.plot(
                [x_cap, x_cap],
                [y_pos[i] - cap_size, y_pos[i] + cap_size],
                color=color, alpha=a, linewidth=0.9,
            )
        ax.scatter(
            median_cumul[i], y_pos[i],
            s=10, color=color, alpha=a, zorder=3, linewidths=0,
        )

    ax.axvline(1.0, color='gray', linewidth=0.5, linestyle=':', alpha=0.4)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pulsar_names)
    ax.set_ylim(n - 0.5, -0.5)
    ax.set_xlim(-0.02, 1.08)
    ax.set_xlabel('Cumulative fraction of total SNR')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.tick_params(axis='x')
    ax.tick_params(axis='y', length=0)
    ax.grid(True, axis='x', alpha=0.2, linewidth=0.5)
    # Subtle alternating row shading to aid reading across the page.
    for i in range(0, n, 2):
        ax.axhspan(i - 0.5, i + 0.5, color='gray', alpha=0.04, linewidth=0)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

if 'cgw_sim_df' not in globals() or cgw_sim_df.empty:
    print('No cgw_sim_df available yet, so the cumulative SNR plots were skipped.')
else:
    breakdown_records = []
    for _, row in cgw_sim_df.iterrows():
        summary_path  = _resolve_summary_path(row.get('source_file'))
        if summary_path is None:
            continue
        scenario_name = str(row.get('scenario', 'unknown'))
        record        = _load_loudest_breakdown(summary_path, scenario_name)
        if record is None:
            continue
        record['sim_key']     = str(row.get('sim_key', record['sim_key']))
        record['source_file'] = str(row.get('source_file', summary_path))
        breakdown_records.append(record)

    breakdown_df = pd.DataFrame(breakdown_records)

    if breakdown_df.empty:
        print('No loudest-binary breakdowns were found in the loaded summaries.')
    else:
        scenario_values = breakdown_df['scenario'].dropna().unique().tolist()
        if 'scenario_order' in globals():
            scenarios = [s for s in scenario_order if s in scenario_values]
            scenarios.extend(s for s in scenario_values if s not in scenarios)
        else:
            scenarios = sorted(scenario_values)

        color_map = {
            s: color
            for s, color in zip(scenarios, ['lime', 'magenta', 'navy'])
        }
        print(f'Loaded loudest-binary breakdowns for {len(breakdown_df)} simulations.')
        print('Breakdowns by scenario:')
        for s in scenarios:
            print(f'  {s}: {int((breakdown_df["scenario"] == s).sum())}')

        figure_dir = Path('figures')
        figure_dir.mkdir(parents=True, exist_ok=True)

        legend_elements = [
            Line2D([0], [0], marker='o', color='gray', markersize=4,
                   linewidth=0, label='Median'),
            Line2D([0], [0], color='gray', linewidth=0.9,
                   label='95% credible interval'),
        ]

        # ------------------------------------------------------------------
        # One full-page ApJ figure per scenario
        # ------------------------------------------------------------------
        for scenario in scenarios:
            
            scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario].copy()
            curves = [c for c in scenario_rows['cumulative_snr'].tolist() if len(c) > 0]
            if not curves:
                continue

            padded    = _pad_curves(curves)
            n_pulsars = padded.shape[1]
            bands = {
                'p2_5':  np.nanpercentile(padded,  2.5, axis=0),
                'p50':   np.nanmedian(padded,             axis=0),
                'p97_5': np.nanpercentile(padded, 97.5, axis=0),
            }

            median_cumul, lo95, hi95, marginal = _build_forest_data(padded, bands)
            alphas       = _alpha_ramp(marginal)
            pulsar_names = _median_pulsar_order(scenario_rows, n_pulsars)
            color        = color_map[scenario]
            legend_elements_scenario = [
                Line2D([0], [0], marker='o', color=color, markersize=4,
                       linewidth=0, label='Median'),
                Line2D([0], [0], color=color, linewidth=0.9,
                       label='95% credible interval'),
            ]
            axes_height = APJ_HEIGHT - _RESERVED_INCHES
            fontsize    = _fontsize_for_height(n_pulsars, axes_height)

            fig, ax = plt.subplots(figsize=(APJ_WIDTH, APJ_HEIGHT))

            # Tight margins: left wide enough for pulsar names, right minimal.
            # We estimate name width as longest_name * fontsize * 0.60 pt/char,
            # converted to a figure fraction.
            max_name_len   = max((len(n) for n in pulsar_names), default=10)
            left_inches    = max(1.0, max_name_len * fontsize * 0.60 / 72.0)
            left_frac      = left_inches / APJ_WIDTH
            right_frac     = 0.02
            bottom_frac    = _RESERVED_INCHES * 0.40 / APJ_HEIGHT
            top_frac       = 1.0 - _RESERVED_INCHES * 0.35 / APJ_HEIGHT

            fig.subplots_adjust(
                left=left_frac, right=1.0 - right_frac,
                bottom=bottom_frac, top=top_frac,
            )

            _draw_forest(ax, pulsar_names, median_cumul, lo95, hi95,
                         alphas, color, fontsize)

            
            ax.legend(handles=legend_elements_scenario, frameon=False,
                      loc='lower left')

            out_path = figure_dir / f'cgw_cumulative_snr_forest_{scenario}.pdf'
            fig.savefig(out_path, dpi=APJ_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f'  saved {out_path}')

        # ------------------------------------------------------------------
        # Combined figure — side-by-side panels, one per scenario
        # ------------------------------------------------------------------
        scenario_data = []
        for scenario in scenarios:
            scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario].copy()
            curves = [c for c in scenario_rows['cumulative_snr'].tolist() if len(c) > 0]
            if not curves:
                continue
            padded    = _pad_curves(curves)
            n_pulsars = padded.shape[1]
            bands = {
                'p2_5':  np.nanpercentile(padded,  2.5, axis=0),
                'p50':   np.nanmedian(padded,             axis=0),
                'p97_5': np.nanpercentile(padded, 97.5, axis=0),
            }
            p50      = bands['p50']
            marginal = np.concatenate([[p50[0]], np.diff(p50)])
            scenario_data.append({
                'scenario':     scenario,
                'n_pulsars':    n_pulsars,
                'median_cumul': p50,
                'lo95':         bands['p2_5'],
                'hi95':         bands['p97_5'],
                'marginal':     marginal,
                'alphas':       _alpha_ramp(marginal),
                'pulsar_names': _median_pulsar_order(
                    breakdown_df[breakdown_df['scenario'] == scenario], n_pulsars
                ),
                'color':        color_map[scenario],
            })

        if scenario_data:
            n_panels  = len(scenario_data)
            max_n     = max(d['n_pulsars'] for d in scenario_data)

            axes_height = APJ_HEIGHT - _RESERVED_INCHES
            fontsize    = _fontsize_for_height(max_n, axes_height)

            # Width: each panel gets APJ_WIDTH inches; panels share y-labels
            # only on the leftmost panel so others can be narrower.
            panel_width = APJ_WIDTH
            fig_width   = panel_width * n_panels

            fig, axes = plt.subplots(
                1, n_panels,
                figsize=(fig_width, APJ_HEIGHT),
                sharey=False,
            )
            if n_panels == 1:
                axes = [axes]

            max_name_len = max(
                max((len(nm) for nm in d['pulsar_names']), default=10)
                for d in scenario_data
            )
            left_inches  = max(1.0, max_name_len * fontsize * 0.60 / 72.0)
            # Express margins as fractions of the *total* figure width.
            left_frac    = left_inches / fig_width
            right_frac   = 0.01
            bottom_frac  = _RESERVED_INCHES * 0.40 / APJ_HEIGHT
            top_frac     = 1.0 - _RESERVED_INCHES * 0.35 / APJ_HEIGHT
            wspace       = 0.35  # relative gap between panels

            fig.subplots_adjust(
                left=left_frac, right=1.0 - right_frac,
                bottom=bottom_frac, top=top_frac,
                wspace=wspace,
            )

            for ax, d in zip(axes, scenario_data):
                _draw_forest(
                    ax,
                    d['pulsar_names'],
                    d['median_cumul'],
                    d['lo95'],
                    d['hi95'],
                    d['alphas'],
                    d['color'],
                    fontsize,
                )
                ax.set_title(d['scenario'], pad=4)


            fig.legend(
                handles=legend_elements, frameon=False,
                loc='upper center', bbox_to_anchor=(0.5, 1.0),
                ncol=2,
            )

            combined_out = figure_dir / 'cgw_cumulative_snr_forest_combined.pdf'
            fig.savefig(combined_out, dpi=APJ_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f'  saved {combined_out}')

In [ ]:
# ---------------------------------------------------------------------------
# Top-20 pulsars per scenario (by median rank), formatted as Python tuples
# ---------------------------------------------------------------------------

import re


def _safe_var_name(scenario_name, prefix='BEST_PSRS'):
    slug = re.sub(r'[^A-Za-z0-9]+', '_', str(scenario_name)).strip('_').upper()
    return f'{prefix}_{slug}'


def _format_psr_tuple(var_name, names, per_line=3):
    lines = [f'{var_name} = (']
    for i in range(0, len(names), per_line):
        chunk = names[i:i + per_line]
        entries = ', '.join(f"'{n}'" for n in chunk)
        lines.append(f'    {entries},')
    lines.append(')')
    return '\n'.join(lines)


TOP_N = 40
per_scenario_top20 = {}   # scenario -> list of top-N pulsar names
blocks = []

for scenario in scenarios:
    scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario].copy()
    if scenario_rows.empty:
        continue

    # Use the full pulsar count available for this scenario so the median
    # ranking is computed over everything, then slice to the top N.
    n_pulsars_full = max(
        (len(nl) for nl in scenario_rows['pulsar_names'].tolist() if nl),
        default=0,
    )
    if n_pulsars_full == 0:
        continue

    ordered = _median_pulsar_order(scenario_rows, n_pulsars_full)
    top20 = ordered[:TOP_N]
    per_scenario_top20[scenario] = top20

    var_name = _safe_var_name(scenario)
    blocks.append(_format_psr_tuple(var_name, top20))

script_text = '\n\n\n'.join(blocks)

# ------------------------------------------------------------------
# Combined list across all scenarios: unique pulsars, ordered by the
# best (lowest) median rank they achieved in any scenario, restricted
# to those that appear in some scenario's top 20.
# ------------------------------------------------------------------
combined_rank = {}
for scenario, top20 in per_scenario_top20.items():
    for rank, name in enumerate(top20):
        if name not in combined_rank or rank < combined_rank[name]:
            combined_rank[name] = rank

combined_ordered = sorted(combined_rank, key=lambda n: combined_rank[n])

combined_block = _format_psr_tuple('BEST_PSRS_COMBINED', combined_ordered)

full_output = script_text + '\n\n\n' + combined_block
print(full_output)

# Optionally write it out to a .py file alongside the figures
out_path = Path('figures') / 'best_pulsars.py'
out_path.write_text(full_output + '\n')
print(f'\nsaved {out_path}')

In [ ]:
# Thresholded cumulative SNR contribution from the loudest binary in each simulation.
# This mirrors the pre-threshold figure block above, but uses cgw_threshold_sim_df.

if 'cgw_threshold_sim_df' not in globals() or cgw_threshold_sim_df.empty:
    print('No cgw_threshold_sim_df available yet, so the thresholded cumulative SNR plots were skipped.')
else:
    threshold_breakdown_records = []
    for _, row in cgw_threshold_sim_df.iterrows():
        summary_path = _resolve_summary_path(row.get('source_file'))
        if summary_path is None:
            continue

        scenario_name = str(row.get('scenario', 'unknown'))
        record = _load_loudest_breakdown(summary_path, scenario_name)
        if record is None:
            continue

        record['sim_key'] = str(row.get('sim_key', record['sim_key']))
        record['source_file'] = str(row.get('source_file', summary_path))
        threshold_breakdown_records.append(record)

    threshold_breakdown_df = pd.DataFrame(threshold_breakdown_records)

    if threshold_breakdown_df.empty:
        print('No thresholded loudest-binary breakdowns were found in the loaded summaries.')
    else:
        threshold_scenarios = sorted(threshold_breakdown_df['scenario'].dropna().unique().tolist())
        threshold_color_map = {s: color
                                    for s, color in zip(scenarios, ['lime', 'magenta', 'navy'])
                                }
        threshold_line_style_map = {
            scenario: style
            for scenario, style in zip(threshold_scenarios, ['-', '--', '-.', ':'])
        }
        while len(threshold_line_style_map) < len(threshold_scenarios):
            threshold_line_style_map[threshold_scenarios[len(threshold_line_style_map)]] = '-'

        print(f'Loaded thresholded loudest-binary breakdowns for {len(threshold_breakdown_df)} simulations.')
        print('Thresholded breakdowns by scenario:')
        for scenario in threshold_scenarios:
            scenario_count = int((threshold_breakdown_df['scenario'] == scenario).sum())
            print(f'  {scenario}: {scenario_count}')

        figure_dir = Path('figures')
        figure_dir.mkdir(parents=True, exist_ok=True)

        # One figure per configuration.
        for scenario in threshold_scenarios:
            scenario_rows = threshold_breakdown_df[threshold_breakdown_df['scenario'] == scenario].copy()
            if scenario_rows.empty:
                continue

            curves = [curve for curve in scenario_rows['cumulative_snr'].tolist() if len(curve) > 0]
            if not curves:
                continue

            padded = _pad_curves(curves)
            median_curve = np.nanmedian(padded, axis=0)
            x_vals = np.arange(1, len(median_curve) + 1)

            fig, ax = plt.subplots(figsize=(9.5, 6.0))
            color = threshold_color_map[scenario]
            line_style = threshold_line_style_map[scenario]

            for curve in curves:
                ax.plot(
                    np.arange(1, len(curve) + 1),
                    curve,
                    color=color,
                    linestyle=line_style,
                    alpha=0.12,
                    linewidth=1.0,
                )

            ax.plot(
                x_vals,
                median_curve,
                color=color,
                linestyle=line_style,
                alpha=0.95,
                linewidth=2.8,
                label=f'{scenario} median',
            )

            ax.set_title(f'Thresholded cumulative loudest-binary SNR fraction by pulsar rank: {scenario}')
            ax.set_xlabel('Pulsar rank after sorting by per-pulsar $\\rho^2$ contribution')
            ax.set_ylabel('Fraction of total SNR')
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False)
            fig.tight_layout()

            out_path = figure_dir / f'cgw_cumulative_snr_fraction_by_pulsar_rank_threshold_{scenario}.pdf'
            fig.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f'  saved {out_path}')

        # Combined figure with all configurations on the same axes.
        fig, ax = plt.subplots(figsize=(10.5, 6.5))
        legend_handles = []

        for scenario in threshold_scenarios:
            scenario_rows = threshold_breakdown_df[threshold_breakdown_df['scenario'] == scenario].copy()
            if scenario_rows.empty:
                continue

            curves = [curve for curve in scenario_rows['cumulative_snr'].tolist() if len(curve) > 0]
            if not curves:
                continue

            padded = _pad_curves(curves)
            median_curve = np.nanmedian(padded, axis=0)
            x_vals = np.arange(1, len(median_curve) + 1)
            color = threshold_color_map[scenario]
            line_style = threshold_line_style_map[scenario]

            for curve in curves:
                ax.plot(
                    np.arange(1, len(curve) + 1),
                    curve,
                    color=color,
                    linestyle=line_style,
                    alpha=0.08,
                    linewidth=0.9,
                )

            ax.plot(
                x_vals,
                median_curve,
                color=color,
                linestyle=line_style,
                alpha=0.98,
                linewidth=3.0,
            )
            legend_handles.append(
                Line2D(
                    [0], [0],
                    color=color,
                    linestyle=line_style,
                    linewidth=3.0,
                    label=f'{scenario} median',
                )
            )

        ax.set_title('Thresholded cumulative loudest-binary SNR fraction by pulsar rank across all configurations')
        ax.set_xlabel('Pulsar rank after sorting by per-pulsar $\\rho^2$ contribution')
        ax.set_ylabel('Fraction of total SNR')
        ax.grid(True, alpha=0.25)
        if legend_handles:
            ax.legend(handles=legend_handles, frameon=False, loc='best')
        fig.tight_layout()

        combined_out = figure_dir / 'cgw_cumulative_snr_fraction_by_pulsar_rank_combined_threshold.pdf'
        fig.savefig(combined_out, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f'  saved {combined_out}')

### Cumulative for all scenario plot

In [130]:
# Combined cumulative SNR plot, all scenarios overlaid on one panel.
#
# X-axis: number of pulsars included, ranked by that simulation's per-pulsar
#         rho^2 contribution to the loudest binary (rank 1 = biggest
#         contributor), i.e. "top N pulsars".
# Y-axis: cumulative fraction of total (S/N)_s for the loudest binary,
#         sqrt(cumsum(rho_sq)) / sqrt(total_rho_sq).
#
# One line (median across simulations) + shaded band (95% credible interval
# across simulations) per scenario, color-coded, single panel, no per-pulsar
# names on the x-axis (this is a pure "how many pulsars do you need" curve,
# not a forest plot).
#
# Same underlying data source as the forest-plot scripts: `top_cgw_breakdowns`
# metadata in `summary.pkl.gz`, `per_pulsar_rho_sq` for the loudest binary in
# each simulation.

import gzip
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as path_effects


# ---------------------------------------------------------------------------
# Sizing / font configuration -- single-column ApJ figure, one panel
# ---------------------------------------------------------------------------
FIG_WIDTH_IN  = 3.5    # ApJ single-column text width
FIG_HEIGHT_IN = 2.8
DPI           = 300


# Tail of the curve (once you're past the pulsars that actually matter) is
# flat and uninformative -- truncate the x-axis rather than showing 60+
# pulsars of dead space. Bump this up if your PTA needs more than this to
# saturate.
MAX_PULSARS = 82


# ---------------------------------------------------------------------------
# Path / data helpers (identical to the forest-plot scripts)
# ---------------------------------------------------------------------------

def _resolve_summary_path(source_value):
    if source_value is None or pd.isna(source_value):
        return None
    source_path = Path(str(source_value))
    if source_path.is_dir():
        candidate = source_path / 'summary.pkl.gz'
        return candidate if candidate.is_file() else None
    if source_path.name == 'summary.pkl.gz' and source_path.is_file():
        return source_path
    candidate = source_path.parent / 'summary.pkl.gz'
    return candidate if candidate.is_file() else None


def _load_loudest_breakdown(summary_path, scenario_name):
    summary_path = Path(summary_path)
    if not summary_path.is_file():
        return None

    with gzip.open(summary_path, 'rb') as fh:
        payload = pickle.load(fh)

    top_breakdowns = payload.get('meta', {}).get('top_cgw_breakdowns', []) or []
    if not top_breakdowns:
        return None

    loudest = max(top_breakdowns, key=lambda item: float(item.get('cgw_snr', np.nan)))
    per_pulsar = loudest.get('per_pulsar_rho_sq', {}) or {}
    if not per_pulsar:
        return None

    ordered = sorted(per_pulsar.items(), key=lambda kv: float(kv[1]), reverse=True)
    rho_sq = np.array([max(float(v), 0.0) for _, v in ordered], dtype=float)
    total = np.sum(rho_sq)
    cumulative_snr = np.sqrt(np.cumsum(rho_sq)) / np.sqrt(total) if total > 0 else np.zeros_like(rho_sq)

    return {
        'scenario':        scenario_name,
        'summary_path':    summary_path,
        'sim_key':         summary_path.parent.name,
        'loudest_cgw_snr': float(loudest.get('cgw_snr', np.nan)),
        'pulsar_names':    [name for name, _ in ordered],
        'rho_sq':          rho_sq,
        'cumulative_snr':  cumulative_snr,
    }


# Explicit scenario-name -> legend label mapping, keyed by the REAL
# scenario values as they appear in breakdown_df['scenario']. Edit the keys
# on the left if your actual scenario names differ from these.
SCENARIO_MCHAR_LABELS = {
    'optimistic': r'Heavy',
    'realistic':  r'Intermediate',
    'pessimistic': r'Light',
}


def _mchar_label(scenario_name):
    """
    Look up the legend label for this scenario in SCENARIO_MCHAR_LABELS.
    Falls back to the raw scenario name (with a warning) if it isn't in the
    mapping, so a typo'd key fails loudly instead of silently mislabeling.
    """
    if scenario_name in SCENARIO_MCHAR_LABELS:
        return SCENARIO_MCHAR_LABELS[scenario_name]
    print(f"  WARNING: no label mapping for scenario '{scenario_name}' -- "
          f"add it to SCENARIO_MCHAR_LABELS. Falling back to raw name.")
    return str(scenario_name)


def _pad_curves(curves):
    if not curves:
        return np.empty((0, 0), dtype=float)
    max_len = max(len(c) for c in curves)
    padded = np.full((len(curves), max_len), np.nan, dtype=float)
    for i, c in enumerate(curves):
        if len(c) == 0:
            continue
        padded[i, :len(c)] = c
        padded[i, len(c):] = c[-1]   # curve saturates once all pulsars are counted
    return padded


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

if 'cgw_sim_df' not in globals() or cgw_sim_df.empty:
    print('No cgw_sim_df available yet, so the combined cumulative SNR plot was skipped.')
else:
    breakdown_records = []
    for _, row in cgw_sim_df.iterrows():
        summary_path = _resolve_summary_path(row.get('source_file'))
        if summary_path is None:
            continue
        scenario_name = str(row.get('scenario', 'unknown'))
        record = _load_loudest_breakdown(summary_path, scenario_name)
        if record is None:
            continue
        record['sim_key']     = str(row.get('sim_key', record['sim_key']))
        record['source_file'] = str(row.get('source_file', summary_path))
        breakdown_records.append(record)

    breakdown_df = pd.DataFrame(breakdown_records)

    if breakdown_df.empty:
        print('No loudest-binary breakdowns were found in the loaded summaries.')
    else:
        scenario_values = breakdown_df['scenario'].dropna().unique().tolist()
        if 'scenario_order' in globals():
            scenarios = [s for s in scenario_order if s in scenario_values]
            scenarios.extend(s for s in scenario_values if s not in scenarios)
        else:
            scenarios = sorted(scenario_values)

        # Legend label per scenario, derived from the real scenario value --
        # NOT a hardcoded positional list, so it can't drift out of sync
        # with `scenarios` if the ordering or number of scenarios changes.
        mchar_label_map = {s: _mchar_label(s) for s in scenarios}

        # 'lime' in particular is very low-contrast as a fill against white,
        # which is most of why the bands were hard to tell apart. If the
        # dashed-edge fix below still isn't enough, swap this palette for
        # something with more separated hue/lightness, e.g.:
        #   ['tab:blue', 'tab:orange', 'tab:green']  (colorblind-friendlier)
        #   ['seagreen', 'crimson', 'navy']           (keeps navy, drops lime)
        color_map = {
            s: color
            for s, color in zip(scenarios, ['lime', 'magenta', 'navy'])
        }
        # Distinct dash pattern per scenario -- color alone wasn't enough to
        # separate the CI edges where bands overlap, so each scenario also
        # gets its own linestyle. Cycles if there are more than 3 scenarios.
        linestyle_map = {
            s: ls
            for s, ls in zip(
                scenarios,
                [
                    (0, (3, 1.5)),      # short dash
                    (0, (1, 1.3)),      # dotted
                    (0, (5, 1.5, 1, 1.5)),  # dash-dot
                ] * (len(scenarios) // 3 + 1),
            )
        }

        figure_dir = Path('figures')
        figure_dir.mkdir(parents=True, exist_ok=True)

        fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))

        max_n_plotted = 0
        for scenario in scenarios:
            scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario]
            curves = [c for c in scenario_rows['cumulative_snr'].tolist() if len(c) > 0]
            if not curves:
                continue

            padded    = _pad_curves(curves)
            n_pulsars = padded.shape[1]

            p50   = np.nanmedian(padded, axis=0)
            lo95  = np.nanpercentile(padded, 2.5, axis=0)
            hi95  = np.nanpercentile(padded, 97.5, axis=0)

            n_plot = min(n_pulsars, MAX_PULSARS)
            max_n_plotted = max(max_n_plotted, n_plot)
            x = np.arange(1, n_plot + 1)
            color = color_map[scenario]
            linestyle = linestyle_map[scenario]

            ax.fill_between(
                x, lo95[:n_plot], hi95[:n_plot],
                color=color, alpha=0.12, linewidth=0, zorder=1,
            )
            # Dashed edges at the CI boundary, with a per-scenario dash
            # pattern -- this is what actually makes overlapping bands
            # distinguishable, since flat alpha fills muddy together where
            # scenarios overlap, and same-style dashes in different colors
            # still visually merge at a glance. The fill is just a soft
            # highlight; the styled edges do the real work.
            for edge in (lo95[:n_plot], hi95[:n_plot]):
                ax.plot(
                    x, edge,
                    color=color, alpha=0.65, linewidth=0.9,
                    linestyle=linestyle, zorder=2,
                )
            ax.plot(
                x, p50[:n_plot],
                color=color, linewidth=1.8, label=mchar_label_map[scenario], zorder=3,
                path_effects=[
                    path_effects.withStroke(linewidth=2.6, foreground='white', alpha=0.8)
                ],
            )

        ax.axhline(1.0, color='gray', linewidth=0.6, linestyle=':', alpha=0.4, zorder=0)

        ax.set_xlim(1, max(max_n_plotted, 1))
        ax.set_ylim(-0.02, 1.08)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
        ax.set_xscale('log')
        ax.set_xlabel('Number of pulsars included')
        ax.set_ylabel(r'Cumulative % of (S/N)$_{s}$')
        ax.tick_params(axis='both')
        ax.grid(True, alpha=0.2, linewidth=0.5)

        ax.legend(frameon=False, loc='lower right')

        fig.tight_layout()

        out_path = figure_dir / 'cgw_cumulative_snr_combined_cdf.pdf'
        fig.savefig(out_path, dpi=DPI, bbox_inches='tight')
        plt.close(fig)
        print(f'  saved {out_path}  ({FIG_WIDTH_IN}in x {FIG_HEIGHT_IN}in)')

  saved figures/cgw_cumulative_snr_combined_cdf.pdf  (3.5in x 2.8in)


---
## Synthetic PTA Analysis

The following sections analyse the **synthetic (upgraded) PTA scenarios** produced by Stage 2.
Each summary file may contain additional SNR fields `cgw_snr_<scenario>` alongside the baseline `cgw_snr`.
We load these extra fields, pair each simulation's synthetic-PTA loudest source against its baseline loudest source,
and produce:

1. **SNR improvement histograms** — Δρ (or ρ_syn / ρ_baseline) across simulations, one histogram per synthetic scenario, coloured by population scenario.
2. **Binary parameter distributions** for the loudest source in each synthetic-PTA scenario.
3. **Detection-fraction gain** — how many additional simulations cross the SNR threshold with each upgraded PTA.
4. **Correlation / scatter plots** — baseline vs synthetic SNR to characterise which sources benefit most.
5. **Sky-map of loudest sources** per scenario.

In [69]:
# ============================================================
# SYNTHETIC PTA — STEP 1: discover extra SNR fields
# ============================================================
import gzip, pickle, re
import numpy as np
import pandas as pd
from pathlib import Path

def _discover_synthetic_snr_fields(result_files, max_probe=30):
    """Scan summary files to find all cgw_snr_<scenario> array keys."""
    found = set()
    for fp in result_files[:max_probe]:
        try:
            payload = _load_payload(fp, verbose=False)
            if not isinstance(payload, dict):
                continue
            arrays = payload.get('arrays', {})
            if not isinstance(arrays, dict):
                continue
            for k in arrays:
                if k.startswith('cgw_snr_'):
                    found.add(k)
        except Exception:
            pass
    return sorted(found)


synthetic_snr_fields = _discover_synthetic_snr_fields(result_files)
synthetic_pta_labels = [f[len('cgw_snr_'):] for f in synthetic_snr_fields]

print(f'Discovered synthetic PTA SNR fields ({len(synthetic_snr_fields)}):',
      synthetic_snr_fields if synthetic_snr_fields else '  none')
if not synthetic_snr_fields:
    print('\n  NOTE: No synthetic PTA SNR fields found in any summary file.')
    print('  Run Stage 2 with --synthetic-ptas to generate them.')
    print('  The cells below will print warnings but not crash.')

Discovered synthetic PTA SNR fields (11): ['cgw_snr_2x_precision', 'cgw_snr_2x_precision_conserved', 'cgw_snr_4x_cad_2x_prec', 'cgw_snr_4x_cad_2x_prec_conserved', 'cgw_snr_4x_cadence', 'cgw_snr_4x_cadence_conserved', 'cgw_snr_baseline_forecast', 'cgw_snr_max_cadence_top10', 'cgw_snr_max_cadence_top20', 'cgw_snr_max_cadence_top30', 'cgw_snr_max_cadence_top40']


In [ ]:
# ============================================================
# SYNTHETIC PTA — STEP 2: build per-simulation summary table
# ============================================================

def build_synthetic_pta_sim_table(result_files, synthetic_snr_fields, verbose=False):
    """
    Iterate over summary files and, for each synthetic SNR field, record
    the loudest binary's SNR and its physical parameters.
    Returns a DataFrame with one row per simulation.
    """
    if not synthetic_snr_fields:
        return pd.DataFrame()

    records = []

    for fp in result_files:
        try:
            scenario  = infer_scenario(fp)
            run_id    = infer_run_id(fp)
            run_scope = _infer_run_scope_from_path(fp)
            sim_name  = fp.parent.name
            sim_index = _infer_sim_index_from_path(fp)
            sim_key   = f'{run_scope}/{sim_name}' if run_scope else sim_name

            payload = _load_payload(fp, verbose=False)
            if not isinstance(payload, dict) or 'arrays' not in payload:
                continue
            arrays = payload['arrays']
            if not isinstance(arrays, dict) or 'f' not in arrays:
                continue

            n = len(arrays['f'])

            # Baseline loudest
            base_snr_arr     = np.asarray(arrays.get('cgw_snr', np.zeros(n)), dtype=float)
            base_loudest_idx = int(np.nanargmax(base_snr_arr))
            base_snr         = float(base_snr_arr[base_loudest_idx])

            row = {
                'scenario':   scenario,
                'run_id':     run_id,
                'run_scope':  run_scope,
                'sim_name':   sim_name,
                'sim_key':    sim_key,
                'sim_index':  int(sim_index),
                'source_file': str(fp),
                'loudest_cgw_snr': base_snr,
            }

            def _param(key, idx):
                arr = arrays.get(key)
                if arr is not None and len(arr) > idx:
                    return float(arr[idx])
                return np.nan

            for param in ('f', 'Mc', 'D_comov', 'h0', 'z', 'ra', 'dec', 'psi', 'iota', 'Mtot'):
                row[f'baseline_{param}'] = _param(param, base_loudest_idx)

            # Synthetic PTA fields
            for field in synthetic_snr_fields:
                syn_arr = np.asarray(arrays.get(field, np.zeros(n)), dtype=float)
                if np.all(syn_arr == 0):
                    row[f'loudest_{field}'] = np.nan
                    for param in ('f', 'Mc', 'D_comov', 'h0', 'z', 'ra', 'dec'):
                        row[f'{field}_{param}'] = np.nan
                else:
                    syn_idx = int(np.nanargmax(syn_arr))
                    row[f'loudest_{field}'] = float(syn_arr[syn_idx])
                    for param in ('f', 'Mc', 'D_comov', 'h0', 'z', 'ra', 'dec'):
                        row[f'{field}_{param}'] = _param(param, syn_idx)

            records.append(row)

        except Exception as e:
            if verbose:
                print(f'  Error processing {fp}: {e}')

    return pd.DataFrame(records) if records else pd.DataFrame()


print('Building synthetic PTA simulation table...')
synpta_sim_df = build_synthetic_pta_sim_table(result_files, synthetic_snr_fields, verbose=False)
synpta_sim_df = _attach_sgwb(synpta_sim_df)

if synpta_sim_df.empty:
    print('  ✗ No synthetic PTA data found — table is empty.')
else:
    print(f'  ✓ {len(synpta_sim_df)} simulations × {len(synthetic_snr_fields)} synthetic scenarios')
    print('  Synthetic scenario labels:', synthetic_pta_labels)
    print('  Population scenarios:', sorted(synpta_sim_df['scenario'].unique().tolist()))
    print(synpta_sim_df[['scenario', 'sim_key', 'loudest_cgw_snr'] +
                         [f'loudest_{f}' for f in synthetic_snr_fields]].head(5).to_string())

Building synthetic PTA simulation table...
  ✓ 1500 simulations × 11 synthetic scenarios
  Synthetic scenario labels: ['2x_precision', '2x_precision_conserved', '4x_cad_2x_prec', '4x_cad_2x_prec_conserved', '4x_cadence', '4x_cadence_conserved', 'baseline_forecast', 'max_cadence_top10', 'max_cadence_top20', 'max_cadence_top30', 'max_cadence_top40']
  Population scenarios: ['optimistic', 'pessimistic', 'realistic']
     scenario                       sim_key  loudest_cgw_snr  loudest_cgw_snr_2x_precision  loudest_cgw_snr_2x_precision_conserved  loudest_cgw_snr_4x_cad_2x_prec  loudest_cgw_snr_4x_cad_2x_prec_conserved  loudest_cgw_snr_4x_cadence  loudest_cgw_snr_4x_cadence_conserved  loudest_cgw_snr_baseline_forecast  loudest_cgw_snr_max_cadence_top10  loudest_cgw_snr_max_cadence_top20  loudest_cgw_snr_max_cadence_top30  loudest_cgw_snr_max_cadence_top40
0  optimistic  2026-07-16_optimistic/sim000         2.263206                      3.852627                                3.808210       

In [71]:
# ============================================================
# SYNTHETIC PTA — STEP 3: compute SNR ratio & delta columns
# ============================================================

if not synpta_sim_df.empty:
    for field, label in zip(synthetic_snr_fields, synthetic_pta_labels):
        syn_col  = f'loudest_{field}'
        base_col = 'loudest_cgw_snr'
        synpta_sim_df[f'snr_ratio_{label}'] = (
            synpta_sim_df[syn_col] / synpta_sim_df[base_col].replace(0, np.nan)
        )
        synpta_sim_df[f'snr_delta_{label}'] = (
            synpta_sim_df[syn_col] - synpta_sim_df[base_col]
        )

    print('Added SNR ratio and delta columns.')
    ratio_cols = [f'snr_ratio_{l}' for l in synthetic_pta_labels]
    delta_cols = [f'snr_delta_{l}' for l in synthetic_pta_labels]
    print(synpta_sim_df[['scenario', 'loudest_cgw_snr'] + ratio_cols].describe().round(3).to_string())
else:
    print('synpta_sim_df is empty — skipping ratio computation.')

Added SNR ratio and delta columns.
       loudest_cgw_snr  snr_ratio_2x_precision  snr_ratio_2x_precision_conserved  snr_ratio_4x_cad_2x_prec  snr_ratio_4x_cad_2x_prec_conserved  snr_ratio_4x_cadence  snr_ratio_4x_cadence_conserved  snr_ratio_baseline_forecast  snr_ratio_max_cadence_top10  snr_ratio_max_cadence_top20  snr_ratio_max_cadence_top30  snr_ratio_max_cadence_top40
count         1500.000                1500.000                          1500.000                  1500.000                            1500.000              1500.000                        1500.000                     1500.000                     1500.000                     1500.000                     1500.000                     1500.000
mean             2.505                   1.959                             1.942                     2.114                               1.756                 2.009                           1.978                        1.911                        1.987                        1.9

### 1 — SNR improvement: histograms across simulations

Each subplot shows the distribution of the **loudest-source SNR** for one PTA configuration
across all simulations.  Colours correspond to the three SMBHB population scenarios.
The left column shows **absolute SNR**, the right column shows **SNR ratio** (synthetic / baseline).

In [72]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

CGW_SNR_THRESHOLD = 5.94

figure_dir = Path('figures')
figure_dir.mkdir(parents=True, exist_ok=True)

POP_SCENARIO_STYLES = {
    'optimistic':  {'color': '#0072B2', 'linestyle': '-',  'linewidth': 2.0},
    'realistic':   {'color': '#E69F00', 'linestyle': '--', 'linewidth': 2.0},
    'pessimistic': {'color': '#D55E00', 'linestyle': ':',  'linewidth': 2.2},
}
SYN_COLOURS = ['#009E73', '#CC79A7', '#F0E442', '#56B4E9', '#999999']

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data available — skipping SNR improvement histograms.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]
    n_syn = len(synthetic_pta_labels)

    # ── Figure A: one row per synthetic scenario ──────────────────────────────
    fig, axes = plt.subplots(n_syn, 2, figsize=(7.0, 2.8 * n_syn), squeeze=False)

    for row_i, (label, field) in enumerate(zip(synthetic_pta_labels, synthetic_snr_fields)):
        ax_abs   = axes[row_i][0]
        ax_ratio = axes[row_i][1]

        abs_vals_by_pop   = {}
        ratio_vals_by_pop = {}
        for pop in pop_scenarios:
            sub = synpta_sim_df[synpta_sim_df['scenario'] == pop]
            a = sub[f'loudest_{field}'].to_numpy(float)
            r = sub[f'snr_ratio_{label}'].to_numpy(float)
            abs_vals_by_pop[pop]   = a[np.isfinite(a) & (a > 0)]
            ratio_vals_by_pop[pop] = r[np.isfinite(r) & (r > 0)]

        all_abs = np.concatenate(list(abs_vals_by_pop.values()))
        if all_abs.size > 0:
            bins_abs = np.logspace(np.log10(max(all_abs.min(), 1e-3)), np.log10(all_abs.max()), 30)
            for pop in pop_scenarios:
                st = POP_SCENARIO_STYLES.get(pop, {})
                if abs_vals_by_pop[pop].size:
                    ax_abs.hist(abs_vals_by_pop[pop], bins=bins_abs, histtype='step', label=pop,
                                color=st.get('color','#888'), linestyle=st.get('linestyle','-'),
                                linewidth=st.get('linewidth',1.5), density=False)
            ax_abs.axvline(CGW_SNR_THRESHOLD, color='black', linestyle='--', linewidth=1.5,
                           label=f'threshold ({CGW_SNR_THRESHOLD})')
            ax_abs.set_xscale('log')
            ax_abs.set_xlabel(r'Loudest-source $\rho$ (synthetic PTA)')
            ax_abs.set_ylabel('N simulations')
            ax_abs.set_title(label, style='italic')

        all_ratio = np.concatenate(list(ratio_vals_by_pop.values()))
        if all_ratio.size > 0:
            bins_ratio = np.logspace(np.log10(max(all_ratio.min(), 1e-2)), np.log10(all_ratio.max()), 30)
            for pop in pop_scenarios:
                st = POP_SCENARIO_STYLES.get(pop, {})
                if ratio_vals_by_pop[pop].size:
                    ax_ratio.hist(ratio_vals_by_pop[pop], bins=bins_ratio, histtype='step', label=pop,
                                  color=st.get('color','#888'), linestyle=st.get('linestyle','-'),
                                  linewidth=st.get('linewidth',1.5), density=False)
            ax_ratio.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='ratio = 1')
            ax_ratio.set_xscale('log')
            ax_ratio.set_xlabel(r'SNR ratio $\rho_{\rm syn} / \rho_{\rm baseline}$')
            ax_ratio.set_ylabel('N simulations')
            ax_ratio.set_title(f'{label} (ratio)', style='italic')

        if row_i == 0:
            handles_pop = [
                mlines.Line2D([], [], color=POP_SCENARIO_STYLES.get(p,{}).get('color','k'),
                              linestyle=POP_SCENARIO_STYLES.get(p,{}).get('linestyle','-'),
                              linewidth=2, label=p)
                for p in pop_scenarios
            ]
            ax_abs.legend(handles=handles_pop, frameon=False, loc='upper right')
            ax_ratio.legend(frameon=False, loc='upper right')

    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_snr_improvement_panels.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'  saved synpta_snr_improvement_panels.pdf')

    # ── Figure B: all synthetic scenarios overlaid, coloured by scenario label ─
    fig2, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(7.0, 3.2))

    for i, (label, field) in enumerate(zip(synthetic_pta_labels, synthetic_snr_fields)):
        colour = SYN_COLOURS[i % len(SYN_COLOURS)]
        a = synpta_sim_df[f'loudest_{field}'].to_numpy(float)
        r = synpta_sim_df[f'snr_ratio_{label}'].to_numpy(float)
        a = a[np.isfinite(a) & (a > 0)]
        r = r[np.isfinite(r) & (r > 0)]
        if a.size:
            ax_l.hist(a, bins=np.logspace(np.log10(max(a.min(),1e-3)), np.log10(a.max()), 25),
                      histtype='step', color=colour, linewidth=1.8, label=label, density=True)
        if r.size:
            ax_r.hist(r, bins=np.logspace(np.log10(max(r.min(),1e-2)), np.log10(r.max()), 25),
                      histtype='step', color=colour, linewidth=1.8, label=label, density=True)

    base_vals = synpta_sim_df['loudest_cgw_snr'].to_numpy(float)
    base_vals = base_vals[np.isfinite(base_vals) & (base_vals > 0)]
    if base_vals.size:
        ax_l.hist(base_vals, bins=np.logspace(np.log10(max(base_vals.min(),1e-3)),
                  np.log10(base_vals.max()), 25), histtype='step', color='black',
                  linewidth=2.0, linestyle='--', label='baseline', density=True)

    ax_l.axvline(CGW_SNR_THRESHOLD, color='gray', linestyle=':', linewidth=1.5)
    ax_l.set_xscale('log')
    ax_l.set_xlabel(r'Loudest-source SNR $\rho$')
    ax_l.set_ylabel('Probability density')
    ax_l.legend(frameon=False)

    ax_r.axvline(1.0, color='gray', linestyle=':', linewidth=1.5)
    ax_r.set_xscale('log')
    ax_r.set_xlabel(r'SNR ratio $\rho_{\rm syn} / \rho_{\rm baseline}$')
    ax_r.set_ylabel('Probability density')
    ax_r.legend(frameon=False)

    plt.tight_layout()
    fig2.savefig(figure_dir / 'synpta_snr_ratio_combined.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig2)
    print(f'  saved synpta_snr_ratio_combined.pdf')

  saved synpta_snr_improvement_panels.pdf
  saved synpta_snr_ratio_combined.pdf


### 2 — Binary parameters of the loudest source in each synthetic PTA

Histograms of the physical parameters ($f$, $\mathcal{M}_c$, $h_0$, $D_{\rm comov}$, inclination $\iota$)
of the **loudest** source in each simulation, separated by synthetic PTA scenario.
This mirrors the equivalent block for the baseline PTA already in the notebook.

In [73]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

figure_dir = Path('figures')

PARAM_SPECS = [
    ('f',       r'GW frequency $f$ [Hz]',                  True),
    ('Mc',      r'Chirp mass $\mathcal{M}_c$ [$M_\odot$]', True),
    ('h0',      r'Strain amplitude $h_0$',                  True),
    ('D_comov', r'Comoving distance $D$ [Mpc]',             True),
    ('iota',    r'Inclination $\iota$ [rad]',               False),
]

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping binary parameter plots.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]

    for label, field in zip(synthetic_pta_labels, synthetic_snr_fields):
        n_params = len(PARAM_SPECS)
        fig, axes = plt.subplots(1, n_params, figsize=(3.2 * n_params, 2.8), squeeze=True)

        for ax, (param_key, xlabel, use_log) in zip(axes, PARAM_SPECS):
            col      = f'{field}_{param_key}'
            base_col = f'baseline_{param_key}'

            all_vals_list = []
            for pop in pop_scenarios:
                sub = synpta_sim_df[synpta_sim_df['scenario'] == pop]
                if col in sub.columns:
                    v = sub[col].to_numpy(float)
                    all_vals_list.append(v[np.isfinite(v)])
            if base_col in synpta_sim_df.columns:
                vb = synpta_sim_df[base_col].to_numpy(float)
                all_vals_list.append(vb[np.isfinite(vb)])

            all_vals = np.concatenate(all_vals_list) if all_vals_list else np.array([])
            if all_vals.size == 0 or col not in synpta_sim_df.columns:
                ax.set_visible(False)
                continue

            bins = (np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)
                    if use_log and all_vals.min() > 0
                    else np.linspace(all_vals.min(), all_vals.max(), 25))

            if base_col in synpta_sim_df.columns:
                bv = synpta_sim_df[base_col].to_numpy(float)
                bv = bv[np.isfinite(bv)]
                if bv.size:
                    ax.hist(bv, bins=bins, histtype='step', color='black',
                            linestyle='--', linewidth=1.5, label='baseline', density=True)

            for pop in pop_scenarios:
                sub = synpta_sim_df[synpta_sim_df['scenario'] == pop]
                if col not in sub.columns:
                    continue
                v = sub[col].to_numpy(float)
                v = v[np.isfinite(v)]
                if v.size == 0:
                    continue
                st = POP_SCENARIO_STYLES.get(pop, {'color': '#888', 'linestyle': '-', 'linewidth': 1.5})
                ax.hist(v, bins=bins, histtype='step', color=st['color'],
                        linestyle=st['linestyle'], linewidth=st['linewidth'],
                        label=pop, density=True)

            if use_log and all_vals.min() > 0:
                ax.set_xscale('log')
            ax.set_xlabel(xlabel)
            ax.set_ylabel('Density')
            ax.tick_params(labelsize=7)
            ax.spines['top'].set_visible(True)
            ax.spines['right'].set_visible(True)

        axes[0].legend(frameon=False, loc='upper right')
        fig.suptitle(f'Loudest binary parameters — synthetic PTA: {label}', y=1.01)
        plt.tight_layout()
        fname = figure_dir / f'synpta_loudest_params_{label}.pdf'
        fig.savefig(fname, dpi=300, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'  saved {fname}')

  saved figures/synpta_loudest_params_2x_precision.pdf
  saved figures/synpta_loudest_params_2x_precision_conserved.pdf
  saved figures/synpta_loudest_params_4x_cad_2x_prec.pdf
  saved figures/synpta_loudest_params_4x_cad_2x_prec_conserved.pdf
  saved figures/synpta_loudest_params_4x_cadence.pdf
  saved figures/synpta_loudest_params_4x_cadence_conserved.pdf
  saved figures/synpta_loudest_params_baseline_forecast.pdf
  saved figures/synpta_loudest_params_max_cadence_top10.pdf
  saved figures/synpta_loudest_params_max_cadence_top20.pdf
  saved figures/synpta_loudest_params_max_cadence_top30.pdf
  saved figures/synpta_loudest_params_max_cadence_top40.pdf


### 3 — Detection-fraction gain

How many additional simulations cross the SNR detection threshold with each upgraded PTA?
Reported as absolute counts and fractional gain relative to the baseline detection fraction.

In [74]:
_SYNPTA_POP_STYLES = {
    'optimistic':  {'color': 'lime',    'linestyle': '-',  'linewidth': 2.2},
    'realistic':   {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2},
    'pessimistic': {'color': 'navy',    'linestyle': ':',  'linewidth': 2.5},
}

_SYNPTA_LABEL_MAP = {
    'optimistic':  r'Heavy',
    'realistic':   r'Intermediate',
    'pessimistic': r'Light',
}

_BASELINE_COL_MAP = {
    'f':       'loudest_f',
    'Mc':      'loudest_Mc',
    'h0':      'loudest_h0',
    'D_comov': 'loudest_D',
    'iota':    None,
    'snr':     'loudest_cgw_snr',   # forecast SNR used below for the SNR param
}

PARAM_SPECS = [
    ('f',       r'$f$ [Hz]',                  True),
    ('Mc',      r'$\mathcal{M}_c$ [$M_\odot$]', True),
    ('h0',      r'$h_0$',                  True),
    ('D_comov', r'$D_{\rm{comov}}$ [Mpc]',             True),
    ('iota',    r'$\iota$ [rad]',               False),
    ('snr',     r'$(\mathrm{S/N})_{\mathrm{CW}}$',                                     True),   # ← renamed key
]


In [75]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

figure_dir = Path('figures')

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping detection fraction analysis.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]

    det_records = []
    for pop in pop_scenarios:
        sub     = synpta_sim_df[synpta_sim_df['scenario'] == pop]
        n_total = len(sub)
        if n_total == 0:
            continue
        base_above    = (sub['loudest_cgw_snr'] >= CGW_SNR_THRESHOLD).sum()
        base_fraction = base_above / n_total
        det_records.append({
            'scenario': pop, 'pta': 'baseline',
            'n_total': n_total, 'n_detected': int(base_above),
            'det_fraction': base_fraction,
            'gain_abs': 0,
            'gain_frac': 0.0,
        })

        # reference for gain_frac/gain_abs is baseline_forecast (if present), not baseline
        ref_above = base_above
        if 'baseline_forecast' in synthetic_pta_labels:
            ref_field = synthetic_snr_fields[synthetic_pta_labels.index('baseline_forecast')]
            ref_col   = f'loudest_{ref_field}'
            if ref_col in sub.columns:
                ref_above = (sub[ref_col] >= CGW_SNR_THRESHOLD).sum()

        for label, field in zip(synthetic_pta_labels, synthetic_snr_fields):
            syn_col = f'loudest_{field}'
            if syn_col not in sub.columns:
                continue
            syn_above    = (sub[syn_col] >= CGW_SNR_THRESHOLD).sum()
            syn_fraction = syn_above / n_total
            det_records.append({
                'scenario': pop, 'pta': label,
                'n_total': n_total, 'n_detected': int(syn_above),
                'det_fraction': syn_fraction,
                'gain_abs':  int(syn_above - ref_above),
                'gain_frac': (syn_above - ref_above) / ref_above if ref_above > 0 else np.nan,
            })

    det_df = pd.DataFrame(det_records)

    # ── printed statistics (unchanged from previous version) ────────────────
    print(f'Detection threshold: SNR ≥ {CGW_SNR_THRESHOLD}')
    print(det_df.to_string(index=False))

    pta_labels_all = ['baseline'] + synthetic_pta_labels
    n_pta  = len(pta_labels_all)
    n_pop  = len(pop_scenarios)
    x_all  = np.arange(n_pta)
    x_syn  = np.arange(len(synthetic_pta_labels))   # right panel: synthetic only
    width  = 0.8 / n_pop

    fig, axes = plt.subplots(1, 2, figsize=(max(5.5, 2.5 * n_pta), 3.5))

    for pi, pop in enumerate(pop_scenarios):
        sub = det_df[det_df['scenario'] == pop].set_index('pta')
        st  = _SYNPTA_POP_STYLES.get(pop, {'color': '#888'})
        offset = (pi - n_pop / 2 + 0.5) * width

        fracs = [float(sub.loc[p, 'det_fraction']) if p in sub.index else 0
                 for p in pta_labels_all]
        axes[0].bar(x_all + offset, fracs, width, color=st['color'], alpha=0.8,
                    label=_SYNPTA_LABEL_MAP.get(pop, pop))

        gains = [float(sub.loc[p, 'gain_frac']) if p in sub.index else 0
                 for p in synthetic_pta_labels]
        axes[1].bar(x_syn + offset, gains, width, color=st['color'], alpha=0.8,
                    label=_SYNPTA_LABEL_MAP.get(pop, pop))

    axes[0].set_xticks(x_all)
    axes[0].set_xticklabels(pta_labels_all, rotation=20, ha='right')
    axes[0].set_ylabel('Detection fraction')
    axes[0].set_ylim(0, 1)

    axes[1].set_xticks(x_syn)
    axes[1].set_xticklabels(synthetic_pta_labels, rotation=20, ha='right')
    axes[1].set_ylabel(r'$\Delta$ detections / baseline detections')
    axes[1].axhline(0, color='black', linewidth=0.8)

    for ax in axes:
        ax.legend(frameon=False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_detection_fraction_gain.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_detection_fraction_gain.pdf')

# ── scatter: detection fraction ──────────────────────────────────────────────
# ── scatter: detection fraction ──────────────────────────────────────────────
_SCATTER_STYLES = {
    'optimistic':  {'color': 'limegreen',  'linestyle': '-',  'marker': 'o',  'linewidth': 1.6, 'ms': 5},
    'realistic':   {'color': 'magenta',    'linestyle': '--', 'marker': 's',  'linewidth': 1.6, 'ms': 5},
    'pessimistic': {'color': 'navy',       'linestyle': ':',  'marker': '^',  'linewidth': 1.8, 'ms': 5},
}

_PTA_DISPLAY_NAMES = {
    'baseline':          'Current (4.5 yr)',
    'baseline_forecast': 'Current (9.0 yr)',
    '2x_precision':      'Current (4.5 yr) +\n' + r'$2\times$ Precision (4.5 yr)',
    '4x_cadence':        'Current (4.5 yr) +\n' + r'$4\times$ Cadence (4.5 yr)',
    '4x_cad_2x_prec':    'Current (4.5 yr) + \n Combined (4.5 yr)',
    '4x_cad_4x_prec':    'Current (4.5 yr) Combined (4x/4x)',
}

# explicit desired left-to-right order: standard 4.5, standard 9, precision, cadence, both
_BASE_ORDER = ['baseline', 'baseline_forecast', '2x_precision', '4x_cadence',
               '4x_cad_2x_prec', '4x_cad_4x_prec']

_PAIR_OFFSET = 0.12  # how far unconserved/conserved shift from the tick when both exist

if not det_df.empty:
    all_pta_names = set(det_df['pta'].unique())
    pta_order = [p for p in _BASE_ORDER
                 if p in all_pta_names or f'{p}_conserved' in all_pta_names]
    x_pos = np.arange(len(pta_order))

    # per-base-name: does the unconserved and/or conserved version exist?
    has_uncons = np.array([p in all_pta_names for p in pta_order])
    has_cons   = np.array([f'{p}_conserved' in all_pta_names for p in pta_order])
    has_both   = has_uncons & has_cons

    # offsets: shift left/right only where both variants are present, else stay centered
    x_uncons = x_pos - np.where(has_both, _PAIR_OFFSET, 0.0)
    x_cons   = x_pos + np.where(has_both, _PAIR_OFFSET, 0.0)

    fig, ax = plt.subplots(figsize=(3.8, 2.8))

    pop_handles = []
    for pop in ('optimistic', 'realistic', 'pessimistic'):
        if pop not in pop_scenarios:
            continue
        sub = det_df[det_df['scenario'] == pop].set_index('pta')
        st  = _SCATTER_STYLES[pop]
        lab = _SYNPTA_LABEL_MAP.get(pop, pop)

        # standard / "unconserved" values -> filled markers, offset left when paired
        fracs = np.array([float(sub.loc[p, 'det_fraction']) if p in sub.index else np.nan
                          for p in pta_order])
        h = ax.scatter(x_uncons, fracs,
                        facecolors=st['color'], edgecolors=st['color'],
                        marker=st['marker'], s=st['ms']**2,
                        label=lab, clip_on=False, zorder=3)
        pop_handles.append(h)

        # time-conserved values -> open markers, offset right when paired
        cons_fracs = np.array([
            float(sub.loc[f'{p}_conserved', 'det_fraction'])
            if f'{p}_conserved' in sub.index else np.nan
            for p in pta_order
        ])
        ax.scatter(x_cons, cons_fracs,
                   facecolors='none', edgecolors=st['color'],
                   marker=st['marker'], s=(st['ms'] * 1.4)**2,
                   linewidths=1.2, clip_on=False, zorder=4)

    # x-axis
    tick_labels = [_PTA_DISPLAY_NAMES.get(p, p) for p in pta_order]
    ax.set_xticks(x_pos)
    ax.set_xticklabels(tick_labels, rotation=30, ha='right',
                        multialignment='right')
    ax.tick_params(axis='x', pad=2)
    ax.set_xlim(-0.5, len(pta_order) - 0.5)

    # y-axis
    ax.set_ylim(0.0, 0.85)
    ax.set_ylabel('Detection fraction')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}'))
    ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))

    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.set_xlabel('Observing Strategy')
    

    # legend 1: population scenario (color)
    pop_legend = ax.legend(handles=pop_handles, frameon=False,
                            loc='upper right', handlelength=2.2)
    ax.add_artist(pop_legend)

    # legend 2: filled vs open marker meaning
    fill_handle = plt.Line2D([], [], marker='o', linestyle='None', markersize=5,
                              markerfacecolor='black', markeredgecolor='black',
                              label='Telecsope time unconserved')
    open_handle = plt.Line2D([], [], marker='o', linestyle='None', markersize=6,
                              markerfacecolor='none', markeredgecolor='black',
                              label='Telescope time conserved')
    ax.legend(handles=[fill_handle, open_handle], frameon=False,
              loc='upper left', handlelength=2.0)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.32)
    fig.savefig(figure_dir / 'synpta_detection_fraction_scatter.png',
                dpi=800, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_detection_fraction_scatter.pdf')

Detection threshold: SNR ≥ 5.94
   scenario                      pta  n_total  n_detected  det_fraction  gain_abs  gain_frac
 optimistic                 baseline      500          30         0.060         0   0.000000
 optimistic             2x_precision      500         139         0.278         6   0.045113
 optimistic   2x_precision_conserved      500         136         0.272         3   0.022556
 optimistic           4x_cad_2x_prec      500         167         0.334        34   0.255639
 optimistic 4x_cad_2x_prec_conserved      500         117         0.234       -16  -0.120301
 optimistic               4x_cadence      500         149         0.298        16   0.120301
 optimistic     4x_cadence_conserved      500         144         0.288        11   0.082707
 optimistic        baseline_forecast      500         133         0.266         0   0.000000
 optimistic        max_cadence_top10      500         144         0.288        11   0.082707
 optimistic        max_cadence_top20  

In [76]:
# ── LaTeX detection fraction table ───────────────────────────────────────────
pta_display = {
    'baseline':       'Unchanged (4.5 yr)',
    'baseline_forecast':       'Unchanged (9.0 yr)',
    '2x_precision':   r'$2 \times$ Precision',
    '2x_precision_conserved':   r'$2 \times$ Precision (TT Cons.)',
    '4x_cadence':     r'$4 \times$ Cadence',
    '4x_cadence_conserved':     r'$4 \times$ Cadence (TT Cons.)',
    '4x_cad_2x_prec': r'Combined',
    '4x_cad_2x_prec_conserved': r'Combined (TT Cons.)',
    'max_cadence_top10': r'Max. Cadence -- Top 10',
    'max_cadence_top20': r'Max. Cadence -- Top 20',
    'max_cadence_top30': r'Max. Cadence -- Top 30',
    'max_cadence_top40': r'Max. Cadence -- Top 40',
    
}

scenario_mass_map = {
    'optimistic':  r'Heavy',
    'realistic':   r'Intermediate',
    'pessimistic': r'Light',
}

# pta_row_order = ['baseline'] + [p for p in synthetic_pta_labels if p != '4x_cad_4x_prec']
pta_row_order = [p for p in synthetic_pta_labels if p != '4x_cad_4x_prec']
if '4x_cad_4x_prec' in synthetic_pta_labels:
    pta_row_order.append('4x_cad_4x_prec')

lines = []
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\begin{tabular}{l l c c}')
lines.append(r'\toprule')
lines.append(
    r' \multicolumn{1}{c}{\shortstack{$m_{\mathrm{char}}$ \\ $[\mathrm{M}_\odot]$}}'
    r' & \multicolumn{1}{c}{\shortstack{Observing Strategy \\ (Rel. to MPTA 4.5-yr)}}'
    r' & \multicolumn{1}{c}{\shortstack{Detection \\Fraction}}'
    r' & \multicolumn{1}{c}{\shortstack{\% Rel.\\Gain}} \\'
)
lines.append(r'\midrule')

for si, pop in enumerate(('optimistic', 'realistic', 'pessimistic')):
    if pop not in pop_scenarios:
        continue
    sub = det_df[det_df['scenario'] == pop].set_index('pta')
    mass_label = scenario_mass_map[pop]
    n_rows = len(pta_row_order)

    # baseline detection fraction for this scenario
    base_frac = float(sub.loc['baseline_forecast', 'det_fraction']) if 'baseline_forecast' in sub.index else np.nan

    for ri, pta in enumerate(pta_row_order):
        if pta not in sub.index:
            continue
        frac  = float(sub.loc[pta, 'det_fraction'])
        n_det = int(sub.loc[pta, 'n_detected'])
        n_tot = int(sub.loc[pta, 'n_total'])
        frac_str = f'${frac:.2f}$'

        if pta == 'baseline_forecast':
            gain_str = r'---'
        elif not np.isnan(base_frac) and base_frac > 0:
            gain = (frac - base_frac) / base_frac * 100
            gain_str = f'${gain:+.0f}\\%$'
        else:
            gain_str = r'---'

        pta_str  = pta_display.get(pta, pta)
        mass_col = rf'\multirow{{{n_rows}}}{{*}}{{{mass_label}}}' if ri == 0 else ''

        lines.append(rf' {mass_col} & {pta_str} & {frac_str} & {gain_str} \\')

    if si < len(pop_scenarios) - 1:
        lines.append(r'\addlinespace')
        lines.append(r'\hline')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(
    r'\caption{Detection fraction (number detected / total simulations) and relative gain '
    r'compared to the MPTA 9.0-yr simulated dataset with the current observed plan for each PTA configuration and population scenario, '
    f'using an SNR threshold of {CGW_SNR_THRESHOLD}' + r'.}'
)
lines.append(r'\label{tab:detection_fraction}')
lines.append(r'\end{table}')

print('\n'.join(lines))

\begin{table}[htbp]
\centering
\begin{tabular}{l l c c}
\toprule
 \multicolumn{1}{c}{\shortstack{$m_{\mathrm{char}}$ \\ $[\mathrm{M}_\odot]$}} & \multicolumn{1}{c}{\shortstack{Observing Strategy \\ (Rel. to MPTA 4.5-yr)}} & \multicolumn{1}{c}{\shortstack{Detection \\Fraction}} & \multicolumn{1}{c}{\shortstack{\% Rel.\\Gain}} \\
\midrule
 \multirow{11}{*}{Heavy} & $2 \times$ Precision & $0.28$ & $+5\%$ \\
  & $2 \times$ Precision (TT Cons.) & $0.27$ & $+2\%$ \\
  & Combined & $0.33$ & $+26\%$ \\
  & Combined (TT Cons.) & $0.23$ & $-12\%$ \\
  & $4 \times$ Cadence & $0.30$ & $+12\%$ \\
  & $4 \times$ Cadence (TT Cons.) & $0.29$ & $+8\%$ \\
  & Unchanged (9.0 yr) & $0.27$ & --- \\
  & Max. Cadence -- Top 10 & $0.29$ & $+8\%$ \\
  & Max. Cadence -- Top 20 & $0.29$ & $+8\%$ \\
  & Max. Cadence -- Top 30 & $0.29$ & $+10\%$ \\
  & Max. Cadence -- Top 40 & $0.30$ & $+11\%$ \\
\addlinespace
\hline
 \multirow{11}{*}{Intermediate} & $2 \times$ Precision & $0.17$ & $+9\%$ \\
  & $2 \times$ Precisi

### 4 — Baseline vs synthetic SNR scatter

For each synthetic scenario a scatter plot shows the loudest-source SNR in the baseline PTA
($x$-axis) against the same quantity in the synthetic PTA ($y$-axis).
Points lying above the diagonal gain SNR with the upgraded array; those below lose it.
Colour encodes the SMBHB population scenario.

In [77]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

figure_dir = Path('figures')

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping scatter plots.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]
    n_syn = len(synthetic_pta_labels)
    fig, axes = plt.subplots(1, n_syn, figsize=(3.8 * n_syn, 3.5), squeeze=False)

    for col_i, (label, field) in enumerate(zip(synthetic_pta_labels, synthetic_snr_fields)):
        ax      = axes[0][col_i]
        syn_col = f'loudest_{field}'

        for pop in pop_scenarios:
            sub  = synpta_sim_df[synpta_sim_df['scenario'] == pop]
            x    = sub['loudest_cgw_snr'].to_numpy(float)
            y    = sub[syn_col].to_numpy(float) if syn_col in sub.columns else np.full(len(sub), np.nan)
            mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
            st   = POP_SCENARIO_STYLES.get(pop, {'color': '#888'})
            ax.scatter(x[mask], y[mask], s=6, alpha=0.4, color=st['color'], label=pop, rasterized=True)

        all_x = synpta_sim_df['loudest_cgw_snr'].to_numpy(float)
        all_y = synpta_sim_df[syn_col].to_numpy(float) if syn_col in synpta_sim_df.columns else all_x
        lim = max(np.nanmax(all_x), np.nanmax(all_y)) * 1.1 if np.any(np.isfinite(all_x)) else 10
        lo  = min(np.nanmin(all_x[all_x > 0]), np.nanmin(all_y[all_y > 0])) * 0.9 \
              if np.any(all_x > 0) else 0.1
        ax.plot([lo, lim], [lo, lim], 'k--', linewidth=1.2, alpha=0.7)
        ax.axhline(CGW_SNR_THRESHOLD, color='grey', linestyle=':', linewidth=1, alpha=0.8)
        ax.axvline(CGW_SNR_THRESHOLD, color='grey', linestyle=':', linewidth=1, alpha=0.8)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlim(lo, lim); ax.set_ylim(lo, lim)
        ax.set_xlabel(r'Baseline loudest $\rho$')
        ax.set_ylabel(fr'Synthetic ({label}) loudest $\rho$')
        ax.set_title(label, style='italic')
        if col_i == 0:
            ax.legend(frameon=False, markerscale=2)

    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_baseline_vs_synthetic_scatter.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_baseline_vs_synthetic_scatter.pdf')

  saved synpta_baseline_vs_synthetic_scatter.pdf


### 5 — SNR improvement CDF

Cumulative distribution of the **SNR ratio** $\rho_{\rm syn}/\rho_{\rm baseline}$
for each synthetic PTA and each population scenario.  The vertical dashed line marks
ratio = 1 (no improvement); the fraction of simulations to the right shows how often
the upgraded PTA finds a louder source.

In [78]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

figure_dir = Path('figures')

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping CDF plots.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]
    n_syn = len(synthetic_pta_labels)
    fig, axes = plt.subplots(1, n_syn, figsize=(3.8 * n_syn, 3.2), squeeze=False)

    for col_i, (label, field) in enumerate(zip(synthetic_pta_labels, synthetic_snr_fields)):
        ax = axes[0][col_i]
        for pop in pop_scenarios:
            sub      = synpta_sim_df[synpta_sim_df['scenario'] == pop]
            r        = sub[f'snr_ratio_{label}'].to_numpy(float)
            r        = r[np.isfinite(r) & (r > 0)]
            if r.size == 0:
                continue
            r_sorted = np.sort(r)
            cdf      = np.arange(1, len(r_sorted) + 1) / len(r_sorted)
            st       = POP_SCENARIO_STYLES.get(pop, {})
            ax.plot(r_sorted, cdf, color=st.get('color','#888'),
                    linestyle=st.get('linestyle','-'), linewidth=st.get('linewidth',1.5), label=pop)

        ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='ratio = 1')
        ax.set_xscale('log')
        ax.set_xlabel(r'SNR ratio $\rho_{\rm syn}/\rho_{\rm baseline}$')
        ax.set_ylabel('CDF')
        ax.set_title(label, style='italic')
        ax.set_ylim(0, 1)
        if col_i == 0:
            ax.legend(frameon=False, loc='upper left')

    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_snr_ratio_cdf.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_snr_ratio_cdf.pdf')

  saved synpta_snr_ratio_cdf.pdf


### 6 — Sky distribution of loudest sources in synthetic PTAs

Mollweide projection sky maps showing where the loudest binary lies in each simulation,
colour-coded by SNR.  One map per synthetic PTA scenario (plus baseline for comparison).

In [79]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

figure_dir = Path('figures')

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping sky maps.')
else:
    panels = [('baseline', 'loudest_cgw_snr', 'baseline_ra', 'baseline_dec')] + [
        (label, f'loudest_{field}', f'{field}_ra', f'{field}_dec')
        for label, field in zip(synthetic_pta_labels, synthetic_snr_fields)
    ]
    n_panels = len(panels)
    fig = plt.figure(figsize=(5.5 * n_panels, 3.5))

    for pi, (label, snr_col, ra_col, dec_col) in enumerate(panels):
        ax  = fig.add_subplot(1, n_panels, pi + 1, projection='mollweide')
        sub = synpta_sim_df.copy()

        if snr_col not in sub.columns or ra_col not in sub.columns:
            ax.set_title(f'{label}\n(no sky data)')
            continue

        ra  = sub[ra_col].to_numpy(float)
        dec = sub[dec_col].to_numpy(float)
        snr = sub[snr_col].to_numpy(float)
        mask = np.isfinite(ra) & np.isfinite(dec) & np.isfinite(snr) & (snr > 0)

        if mask.sum() == 0:
            ax.set_title(f'{label}\n(no valid positions)')
            continue

        ra_moll = ra[mask] - np.pi   # shift [0, 2π] → [-π, π] for Mollweide
        sc = ax.scatter(ra_moll, dec[mask], c=np.log10(snr[mask]),
                        cmap='viridis', s=3, alpha=0.5, rasterized=True)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_title(label, style='italic')
        ax.grid(True, alpha=0.3)
        plt.colorbar(sc, ax=ax, label=r'$\log_{10}(\rho)$', shrink=0.7, pad=0.05)

    fig.suptitle('Sky distribution of loudest sources (synthetic PTAs)', y=1.01)
    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_sky_maps.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_sky_maps.pdf')

  saved synpta_sky_maps.pdf


### 7 - Fixed up binary params plots


In [142]:
# ============================================================
# SYNTHETIC PTA — STEP 2 (revised): individual parameter plots
# per synthetic PTA scenario, mirroring _save_hist_figure style.
# ============================================================

SHOW_BASELINE_OVERLAY = True
BASELINE_ALPHA        = 0.15

# Optional per-(synpta_label, param_key) x-axis limits. Anything not listed
# here falls back to matplotlib's automatic range (the previous behavior).
# Example: XLIM_OVERRIDES = {('4x_cadence', 'f'): (1e-9, 1e-6)}
XLIM_OVERRIDES: dict = {}

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping binary parameter plots.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]

    for label, field in zip(synthetic_pta_labels, synthetic_snr_fields):
        syn_detected = synpta_sim_df[synpta_sim_df[f'loudest_{field}'] >= CGW_SNR_THRESHOLD]

        for param_key, xlabel, use_log in PARAM_SPECS:

            # ── choose the right subset ────────────────────────────────────
            if param_key == 'snr':
                subsets = [
                    (synpta_sim_df, 'all'),        # all simulations
                    (syn_detected,  'detected'),   # above threshold only
                ]
            else:
                subsets = [(syn_detected, 'detected')]
            # ──────────────────────────────────────────────────────────────

            for df_subset, subset_tag in subsets:

                if param_key == 'snr':
                    col = f'loudest_{field}'
                else:
                    col = f'{field}_{param_key}'

                base_cgw_col = _BASELINE_COL_MAP.get(param_key)

                pop_vals = {}
                for pop in pop_scenarios:
                    sub = df_subset[df_subset['scenario'] == pop]
                    if col not in sub.columns:
                        continue
                    v = sub[col].to_numpy(float)
                    v = v[np.isfinite(v)]
                    if v.size:
                        pop_vals[pop] = v

                if not pop_vals:
                    continue

                base_pop_vals = {}
                if SHOW_BASELINE_OVERLAY and base_cgw_col and not cgw_sim_df.empty:
                    for pop in pop_scenarios:
                        # for SNR 'all' panel: no threshold filter on baseline either
                        if param_key == 'snr' and subset_tag == 'all':
                            base_sub = cgw_sim_df[cgw_sim_df['scenario'] == pop]
                        else:
                            base_sub = cgw_sim_df[
                                (cgw_sim_df['scenario'] == pop) &
                                (cgw_sim_df['loudest_cgw_snr'] >= CGW_SNR_THRESHOLD)  # fixed
                            ]
                        bv = base_sub[base_cgw_col].to_numpy(float)
                        if bv.size:
                            base_pop_vals[pop] = bv

                # ── 95% credible intervals ─────────────────────────────────
                print(f'\n95% credible intervals — {label} | {param_key} | {subset_tag}')
                print(f'  {"Scenario":<15}  {"2.5%":>12}  {"Median":>12}  {"97.5%":>12}  {"N":>6}')
                print(f'  {"-"*15}  {"-"*12}  {"-"*12}  {"-"*12}  {"-"*6}')
                for pop in pop_scenarios:
                    if pop in pop_vals:
                        v = pop_vals[pop]
                        lo, med, hi = np.percentile(v, [2.5, 50, 97.5])
                        pop_label = _SYNPTA_LABEL_MAP.get(pop, pop)
                        print(f'  {pop_label:<15}  {lo:>12.4g}  {med:>12.4g}  {hi:>12.4g}  {len(v):>6}')
                if base_pop_vals:
                    print(f'  {"--- Unchanged baseline ---":<15}')
                    for pop in pop_scenarios:
                        if pop in base_pop_vals:
                            bv = base_pop_vals[pop]
                            lo, med, hi = np.percentile(bv, [2.5, 50, 97.5])
                            pop_label = _SYNPTA_LABEL_MAP.get(pop, pop)
                            print(f'  {pop_label:<15}  {lo:>12.4g}  {med:>12.4g}  {hi:>12.4g}  {len(bv):>6}')
                # ──────────────────────────────────────────────────────────

                all_vals = np.concatenate(
                    list(pop_vals.values()) + list(base_pop_vals.values())
                )
                pos_mask = (all_vals > 0) if use_log else np.ones(len(all_vals), dtype=bool)
                all_vals = all_vals[np.isfinite(all_vals) & pos_mask]
                if all_vals.size == 0:
                    continue

                bins = (np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)
                        if use_log and all_vals.min() > 0
                        else np.linspace(all_vals.min(), all_vals.max(), 25))

                fig, ax = plt.subplots(figsize=(3.5, 2.8))

                if SHOW_BASELINE_OVERLAY:
                    for pop, bv in base_pop_vals.items():
                        st = _SYNPTA_POP_STYLES.get(pop, {'color': '#888'})
                        ax.hist(bv, bins=bins, histtype='stepfilled',
                                color=st['color'], alpha=BASELINE_ALPHA,
                                density=False, zorder=1)

                for pop in pop_scenarios:
                    vals = pop_vals.get(pop)
                    if vals is None:
                        continue
                    st = _SYNPTA_POP_STYLES.get(pop, {'color': '#888', 'linestyle': '-', 'linewidth': 1.5})
                    ax.hist(vals, bins=bins, histtype='step',
                            color=st['color'], linestyle=st['linestyle'],
                            linewidth=st['linewidth'], density=False, zorder=2)

                # vertical threshold line on SNR plots
                if param_key == 'snr':
                    ax.axvline(CGW_SNR_THRESHOLD, color='black', linestyle='--',
                               linewidth=1.5, zorder=3)

                if use_log:
                    ax.set_xscale('log')

                # ── optional per-panel x-limit override ────────────────────
                xlim = XLIM_OVERRIDES.get((label, param_key))
                if xlim is not None:
                    ax.set_xlim(*xlim)
                # ────────────────────────────────────────────────────────────

                ax.set_xlabel(xlabel)
                ax.set_ylabel('Number of simulations')
                ax.spines['top'].set_visible(True)
                ax.spines['right'].set_visible(True)

                # ── legend: "Unchanged" baseline patch (if any) above a
                # bold, centered m_char heading, mass values below it ───────
                import matplotlib.patches as mpatches

                if label == '2x_precision':
                    color_handles = [
                        mlines.Line2D(
                            [], [],
                            color=_SYNPTA_POP_STYLES.get(pop, {'color': '#888', 'linestyle': '-', 'linewidth': 1.5})['color'],
                            linestyle=_SYNPTA_POP_STYLES.get(pop, {'color': '#888', 'linestyle': '-', 'linewidth': 1.5})['linestyle'],
                            linewidth=_SYNPTA_POP_STYLES.get(pop, {'color': '#888', 'linestyle': '-', 'linewidth': 1.5})['linewidth'],
                            label=SCENARIO_MASS_MAP.get(pop, _SYNPTA_LABEL_MAP.get(pop, pop)),
                        )
                        for pop in pop_scenarios if pop in pop_vals
                    ]

                    base_kwargs = dict(
                        frameon=False, loc='lower right',
                        handlelength=1.3, handletextpad=0.5,
                        labelspacing=0.3, fontsize='small',
                    )
                    anchor = (1.0, 1.0)   # 'upper right' corner in axes fraction

                    has_baseline = SHOW_BASELINE_OVERLAY and bool(base_pop_vals)
                    if has_baseline:
                        x, y = anchor
                        mass_anchor = (x, y - 0.16)   # tuck the mass legend below the baseline entry
                    else:
                        mass_anchor = anchor

                    mass_kwargs = dict(base_kwargs, bbox_to_anchor=mass_anchor)
                    mass_legend = ax.legend(
                        handles=color_handles,
                        **mass_kwargs,
                    )

                    if has_baseline:
                        ax.add_artist(mass_legend)   # keep it before the next ax.legend() call
                        baseline_handle = mpatches.Patch(
                            facecolor='grey', alpha=BASELINE_ALPHA + 0.25, label='Fiducial')
                        ax.legend(handles=[baseline_handle], **dict(base_kwargs, bbox_to_anchor=anchor))
                # ────────────────────────────────────────────────────────────


95% credible intervals — 2x_precision | f | detected
  Scenario                 2.5%        Median         97.5%       N
  ---------------  ------------  ------------  ------------  ------
  Heavy               2.305e-09     1.309e-08      7.11e-08     139
  Intermediate        2.197e-09     1.697e-08      7.68e-08      85
  Light               2.083e-09     5.232e-09     6.658e-08      55
  --- Unchanged baseline ---
  Heavy                 2.3e-09     1.262e-08     7.356e-08     133
  Intermediate        2.159e-09     1.547e-08     5.826e-08      78
  Light               2.071e-09      4.64e-09     5.104e-08      51

95% credible intervals — 2x_precision | Mc | detected
  Scenario                 2.5%        Median         97.5%       N
  ---------------  ------------  ------------  ------------  ------
  Heavy               1.689e+09     4.747e+09     1.184e+10     139
  Intermediate        1.274e+09      2.68e+09     7.147e+09      85
  Light                6.93e+08     1.231e+09 

In [81]:
from collections import defaultdict

ci_data = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

for param_key, xlabel, use_log in PARAM_SPECS:
    for label, field in zip(synthetic_pta_labels, synthetic_snr_fields):

        if param_key == 'snr':
            col = f'loudest_{field}'
        else:
            col = f'{field}_{param_key}'

        for subset_tag, df_subset in [
            ('all',      synpta_sim_df),
            ('detected', synpta_sim_df[synpta_sim_df[f'loudest_{field}'] >= CGW_SNR_THRESHOLD]),
        ]:
            if col not in df_subset.columns:
                continue
            for pop in pop_scenarios:
                vals = df_subset.loc[df_subset['scenario'] == pop, col].to_numpy(float)
                vals = vals[np.isfinite(vals)]
                if len(vals) == 0:
                    continue
                lo, med, hi = np.nanpercentile(vals, [2.5, 50, 97.5])
                ci_data[param_key][subset_tag][label][pop] = (lo, med, hi, len(vals))

    # Unchanged baseline from cgw_sim_df
    base_col = _BASELINE_COL_MAP.get(param_key)
    if base_col and not cgw_sim_df.empty:
        for subset_tag, df_subset in [
            ('all',      cgw_sim_df),
            ('detected', cgw_sim_df[cgw_sim_df['loudest_cgw_snr'] >= CGW_SNR_THRESHOLD]),
        ]:
            if base_col not in df_subset.columns:
                continue
            for pop in pop_scenarios:
                vals = df_subset.loc[df_subset['scenario'] == pop, base_col].to_numpy(float)
                vals = vals[np.isfinite(vals)]
                if len(vals) == 0:
                    continue
                lo, med, hi = np.nanpercentile(vals, [2.5, 50, 97.5])
                ci_data[param_key][subset_tag]['Unchanged'][pop] = (lo, med, hi, len(vals))

print("ci_data rebuilt. counts:")
for param_key in ci_data:
    for subset_tag in ci_data[param_key]:
        for pta_label in ci_data[param_key][subset_tag]:
            for pop, entry in ci_data[param_key][subset_tag][pta_label].items():
                print(f"  {param_key:6s} {subset_tag:8s} {pta_label:20s} {pop:12s}  n={entry[3]}")

ci_data rebuilt. counts:
  f      all      2x_precision         optimistic    n=500
  f      all      2x_precision         realistic     n=500
  f      all      2x_precision         pessimistic   n=500
  f      all      2x_precision_conserved optimistic    n=500
  f      all      2x_precision_conserved realistic     n=500
  f      all      2x_precision_conserved pessimistic   n=500
  f      all      4x_cad_2x_prec       optimistic    n=500
  f      all      4x_cad_2x_prec       realistic     n=500
  f      all      4x_cad_2x_prec       pessimistic   n=500
  f      all      4x_cad_2x_prec_conserved optimistic    n=500
  f      all      4x_cad_2x_prec_conserved realistic     n=500
  f      all      4x_cad_2x_prec_conserved pessimistic   n=500
  f      all      4x_cadence           optimistic    n=500
  f      all      4x_cadence           realistic     n=500
  f      all      4x_cadence           pessimistic   n=500
  f      all      4x_cadence_conserved optimistic    n=500
  f      all 

In [82]:
Path('tables').mkdir(exist_ok=True)
all_pta_labels = list(synthetic_pta_labels) + ['Unchanged']

import math

def _log10_mchar_str(mc_med):
    if mc_med <= 0:
        return f'${mc_med:.2e}$'
    return r'$10^{' + f'{math.log10(mc_med):.1f}' + r'}$'

def _fmt_val(med, lo, hi, decimals=2):
    fmt = f'{{:.{decimals}f}}'
    return (r'$' + fmt.format(med)
            + r'^{+' + fmt.format(hi - med) + r'}'
            + r'_{-' + fmt.format(med - lo) + r'}$')

_SCENARIO_MCHAR_LABEL = {
    'optimistic':  r'Heavy',
    'realistic':   r'Intermediate',
    'pessimistic': r'Light',
}

def _build_table(param_key, subset_tag, col_header, chirp_key='Mc',
                 conv=1.0, decimals=2, caption=None, label=None):
    param_data = ci_data.get(param_key, {}).get(subset_tag, {})

    if not param_data:
        print(f"ERROR: ci_data['{param_key}']['{subset_tag}'] is empty.")
        print("Available param keys:", list(ci_data.keys()))
        return

    present_labels = [l for l in all_pta_labels if l in param_data]
    if not present_labels:
        print("ERROR: no matching labels found in param_data.")
        print("Labels in param_data:", list(param_data.keys()))
        print("all_pta_labels:", all_pta_labels)
        return

    lines = []
    lines.append(r'\begin{table}[htbp]')
    lines.append(r'\centering')
    lines.append(r'\begin{tabular}{lll}')
    lines.append(r'\toprule')
    lines.append(r'$m_{\mathrm{char}}\ [M_\odot]$ & PTA & ' + col_header + r' \\')
    lines.append(r'\midrule')

    for pop in pop_scenarios:
        # use ground-truth label, not inferred from data
        mc_str = _SCENARIO_MCHAR_LABEL.get(pop, pop)

        n_rows = sum(
            1 for pl in present_labels
            if param_data.get(pl, {}).get(pop) is not None
        )
        if n_rows == 0:
            print(f"  WARNING: no data for scenario '{pop}', skipping.")
            continue

        first_row = True
        for pta_label in present_labels:
            entry = param_data.get(pta_label, {}).get(pop)
            if entry is None:
                continue

            lo, med, hi, n = entry
            lo, med, hi = lo * conv, med * conv, hi * conv
            val_str = _fmt_val(med, lo, hi, decimals=decimals)
            pta_str = str(pta_label).replace('_', r'\_')

            mchar_cell = (r'\multirow{' + str(n_rows) + r'}{*}{' + mc_str + r'}'
                          if first_row else '')
            first_row = False
            lines.append(f'{mchar_cell} & {pta_str} & {val_str} \\\\')

        lines.append(r'\addlinespace')

    if lines[-1] == r'\addlinespace':
        lines.pop()

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\caption{' + (caption or col_header) + r'}')
    lines.append(r'\label{' + (label or f'tab:{param_key}_{subset_tag}') + r'}')
    lines.append(r'\end{table}')

    tex = '\n'.join(lines)
    fname = Path('tables') / f'ci_{param_key}_{subset_tag}.tex'
    fname.write_text(tex)
    print(f'  saved {fname}')
    print(tex)
for param_key in ('snr', 'f'):
    for subset_tag in ('all', 'detected'):
        d = ci_data.get(param_key, {}).get(subset_tag, {})
        for pta_label, pop_dict in d.items():
            for pop, entry in pop_dict.items():
                lo, med, hi, n = entry
                print(f"  {param_key:6s} {subset_tag:8s} {pta_label:20s} {pop:12s}  "
                      f"med={med:.3g}  [{lo:.3g}, {hi:.3g}]  n={n}")    
    
# ── frequency (detected only) ─────────────────────────────────────────────
_build_table(
    param_key  = 'f',
    subset_tag = 'detected',
    col_header = 'Frequency [nHz]',
    conv       = 1e9,
    decimals   = 2,
    caption    = (r'Summary of 95\% credible intervals for the gravitational wave'
                  r' frequencies of detectable SMBHBs for each of the pulsar'
                  r' timing array observation settings used.'),
    label      = 'tab:f_pta',
)

# ── SNR detected ──────────────────────────────────────────────────────────
_build_table(
    param_key  = 'snr',
    subset_tag = 'detected',
    col_header = r'(S/N)$_{s}$',
    conv       = 1.0,
    decimals   = 2,
    caption    = (r'Summary of 95\% credible intervals for the SNR of detectable'
                  r' SMBHBs for each of the pulsar timing array observation'
                  r' settings used.'),
    label      = 'tab:snr_detected_pta',
)

# ── SNR all (includes non-detections) ────────────────────────────────────
_build_table(
    param_key  = 'snr',
    subset_tag = 'all',
    col_header = r'(S/N)$_{s}$',
    conv       = 1.0,
    decimals   = 2,
    caption    = (r'Summary of 95\% credible intervals for the SNR of all simulated'
                  r' SMBHBs (including sub-threshold) for each of the pulsar timing'
                  r' array observation settings used.'),
    label      = 'tab:snr_all_pta',
)

  snr    all      2x_precision         optimistic    med=4.31  [2.18, 17.2]  n=500
  snr    all      2x_precision         realistic     med=3.72  [1.89, 12.5]  n=500
  snr    all      2x_precision         pessimistic   med=2.59  [1.15, 20.2]  n=500
  snr    all      2x_precision_conserved optimistic    med=4.29  [2.16, 17.1]  n=500
  snr    all      2x_precision_conserved realistic     med=3.68  [1.88, 12.4]  n=500
  snr    all      2x_precision_conserved pessimistic   med=2.57  [1.14, 20.2]  n=500
  snr    all      4x_cad_2x_prec       optimistic    med=4.59  [2.21, 20.1]  n=500
  snr    all      4x_cad_2x_prec       realistic     med=3.98  [2, 12.8]  n=500
  snr    all      4x_cad_2x_prec       pessimistic   med=2.91  [1.23, 20.2]  n=500
  snr    all      4x_cad_2x_prec_conserved optimistic    med=3.71  [1.71, 14.8]  n=500
  snr    all      4x_cad_2x_prec_conserved realistic     med=3.34  [1.62, 10.1]  n=500
  snr    all      4x_cad_2x_prec_conserved pessimistic   med=2.45  [1.03, 15

### 8 — Summary statistics table

Concise numerical summary of loudest-source SNR for baseline and each synthetic scenario,
broken down by population scenario.

In [83]:
import pandas as pd
import numpy as np

if synpta_sim_df.empty:
    print('No synthetic PTA data — skipping summary table.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]

    all_cols = (['loudest_cgw_snr'] +
                [f'loudest_{f}' for f in synthetic_snr_fields] +
                [f'snr_ratio_{l}' for l in synthetic_pta_labels])

    rows = []
    for pop in pop_scenarios:
        sub = synpta_sim_df[synpta_sim_df['scenario'] == pop]
        for col in all_cols:
            if col not in sub.columns:
                continue
            v = sub[col].to_numpy(float)
            v = v[np.isfinite(v)]
            if v.size == 0:
                continue
            if col == 'loudest_cgw_snr':
                pta = 'baseline'
            elif col.startswith('loudest_cgw_snr_'):
                pta = col[len('loudest_cgw_snr_'):]
            else:
                pta = col
            rows.append({
                'pop_scenario': pop,
                'PTA': pta,
                'n_sims': v.size,
                'median': np.median(v),
                'mean': np.mean(v),
                'p16': np.percentile(v, 16),
                'p84': np.percentile(v, 84),
                'max': v.max(),
                f'frac_above_{CGW_SNR_THRESHOLD:.1f}': (v >= CGW_SNR_THRESHOLD).mean(),
            })

    summary_tbl = pd.DataFrame(rows)
    print('\nSynthetic PTA Summary Statistics')
    print('=' * 80)
    with pd.option_context('display.float_format', '{:.3g}'.format, 'display.max_columns', 70, 'display.max_rows', 70):
        display(summary_tbl)


Synthetic PTA Summary Statistics


,pop_scenario,PTA,n_sims,median,mean,p16,p84,max,frac_above_5.9
0,optimistic,baseline,500,2.42,2.96,1.53,4.05,36.2,0.06
1,optimistic,2x_precision,500,4.31,5.84,2.8,7.57,67.7,0.278
2,optimistic,2x_precision_conserved,500,4.29,5.79,2.77,7.48,65,0.272
3,optimistic,4x_cad_2x_prec,500,4.59,6.18,2.94,8.12,67.7,0.334
4,optimistic,4x_cad_2x_prec_conserved,500,3.71,4.88,2.35,6.91,42.5,0.234
5,optimistic,4x_cadence,500,4.46,5.96,2.88,7.64,67.7,0.298
6,optimistic,4x_cadence_conserved,500,4.36,5.86,2.83,7.44,64.1,0.288
7,optimistic,baseline_forecast,500,4.22,5.73,2.76,7.5,67.7,0.266
8,optimistic,max_cadence_top10,500,4.15,5.68,2.71,7.6,55.2,0.288
9,optimistic,max_cadence_top20,500,4.29,5.75,2.79,7.77,55.2,0.288


In [84]:
# ============================================================
# SYNTHETIC PTA — DIAGNOSTIC: did the loudest source change?
#
# For each simulation, compares the identity of the loudest binary
# in the synthetic PTA against the loudest binary in the baseline.
# Uses global_idx as the binary identifier. If the synthetic PTA
# promotes a previously-quiet source to the top, that simulation
# is flagged as "source changed".
#
# Requires binary_df to be loaded (has global_idx per binary).
# ============================================================

if synpta_sim_df.empty or not synthetic_snr_fields:
    print('No synthetic PTA data available.')
elif 'global_idx' not in binary_df.columns:
    print('global_idx not in binary_df — cannot match sources across PTAs.')
else:
    change_records = []

    for fp in result_files:
        try:
            scenario  = infer_scenario(fp)
            run_id    = infer_run_id(fp)
            sim_index = _infer_sim_index_from_path(fp)
            run_scope = _infer_run_scope_from_path(fp)
            sim_name  = fp.parent.name
            sim_key   = f'{run_scope}/{sim_name}' if run_scope else sim_name

            payload = _load_payload(fp, verbose=False)
            if not isinstance(payload, dict) or 'arrays' not in payload:
                continue
            arrays = payload['arrays']
            if not isinstance(arrays, dict) or 'f' not in arrays:
                continue

            n = len(arrays['f'])
            global_idx = np.asarray(arrays.get('global_idx', np.arange(n)), dtype=np.int64)

            # baseline loudest binary
            base_snr = np.asarray(arrays.get('cgw_snr_baseline_forecast', np.zeros(n)), dtype=float)
            base_loudest_idx = int(np.nanargmax(base_snr))
            base_loudest_gidx = int(global_idx[base_loudest_idx])
            base_loudest_snr  = float(base_snr[base_loudest_idx])
            base_loudest_f    = float(arrays['f'][base_loudest_idx])

            row = {
                'scenario':        scenario,
                'sim_key':         sim_key,
                'sim_index':       sim_index,
                'base_loudest_gidx': base_loudest_gidx,
                'base_loudest_snr':  base_loudest_snr,
                'base_loudest_f':    base_loudest_f,
            }

            for field in synthetic_snr_fields:
                syn_label = field[len('cgw_snr_'):]
                syn_snr   = np.asarray(arrays.get(field, np.zeros(n)), dtype=float)

                if np.all(syn_snr == 0) or np.all(~np.isfinite(syn_snr)):
                    row[f'{syn_label}_loudest_gidx']   = np.nan
                    row[f'{syn_label}_loudest_snr']    = np.nan
                    row[f'{syn_label}_loudest_f']      = np.nan
                    row[f'{syn_label}_source_changed'] = np.nan
                    row[f'{syn_label}_base_snr_rank']  = np.nan
                    row[f'{syn_label}_snr_ratio']      = np.nan
                    continue

                syn_loudest_idx  = int(np.nanargmax(syn_snr))
                syn_loudest_gidx = int(global_idx[syn_loudest_idx])
                syn_loudest_snr  = float(syn_snr[syn_loudest_idx])
                syn_loudest_f    = float(arrays['f'][syn_loudest_idx])

                source_changed = (syn_loudest_gidx != base_loudest_gidx)

                # what rank was the new loudest source in the *baseline* SNR ordering?
                # rank 1 = was already the loudest in baseline
                if source_changed:
                    base_snr_of_new_source = float(base_snr[syn_loudest_idx])
                    base_rank = int(np.sum(base_snr >= base_snr_of_new_source))
                else:
                    base_rank = 1

                row[f'{syn_label}_loudest_gidx']   = syn_loudest_gidx
                row[f'{syn_label}_loudest_snr']    = syn_loudest_snr
                row[f'{syn_label}_loudest_f']      = syn_loudest_f
                row[f'{syn_label}_source_changed'] = source_changed
                row[f'{syn_label}_base_snr_rank']  = base_rank
                row[f'{syn_label}_snr_ratio']      = (syn_loudest_snr / base_loudest_snr
                                                      if base_loudest_snr > 0 else np.nan)

            change_records.append(row)

        except Exception as e:
            continue

    change_df = pd.DataFrame(change_records)

    if change_df.empty:
        print('No records built — check that arrays contain global_idx and cgw_snr fields.')
    else:
        print(f'Analysed {len(change_df)} simulations across '
              f'{change_df["scenario"].nunique()} population scenarios.\n')

        # ── summary table ────────────────────────────────────────────────────
        summary_rows = []
        for syn_label in synthetic_pta_labels:
            changed_col = f'{syn_label}_source_changed'
            rank_col    = f'{syn_label}_base_snr_rank'
            ratio_col   = f'{syn_label}_snr_ratio'
            if changed_col not in change_df.columns:
                continue

            for pop in sorted(change_df['scenario'].unique()):
                sub = change_df[change_df['scenario'] == pop]
                valid = sub[changed_col].notna()
                n_sims      = valid.sum()
                n_changed   = sub.loc[valid, changed_col].sum()
                pct_changed = 100 * n_changed / n_sims if n_sims else np.nan

                # median baseline rank of the newly-promoted source
                # (only for sims where source changed)
                changed_mask = valid & sub[changed_col].astype(bool)
                med_rank = (sub.loc[changed_mask, rank_col].median()
                            if changed_mask.sum() else np.nan)

                # median SNR ratio (synthetic / baseline loudest)
                med_ratio = sub.loc[valid, ratio_col].median()

                summary_rows.append({
                    'synthetic PTA':      syn_label,
                    'population scenario': pop,
                    'n sims':             int(n_sims),
                    'source changed (n)': int(n_changed),
                    'source changed (%)': round(pct_changed, 1),
                    'median baseline rank\nof promoted source': (round(med_rank, 1)
                                                                  if not np.isnan(med_rank)
                                                                  else '—'),
                    'median SNR ratio\n(syn / baseline)':      round(med_ratio, 3),
                })

        summary_tbl = pd.DataFrame(summary_rows)
        print('=== Did the loudest source change between baseline and synthetic PTA? ===\n')
        display(summary_tbl)

        # ── frequency shift for changed sources ──────────────────────────────
        print('\n=== Frequency of the promoted source vs baseline loudest (changed sims only) ===\n')
        freq_rows = []
        for syn_label in synthetic_pta_labels:
            changed_col  = f'{syn_label}_source_changed'
            syn_f_col    = f'{syn_label}_loudest_f'
            if changed_col not in change_df.columns:
                continue
            for pop in sorted(change_df['scenario'].unique()):
                sub          = change_df[change_df['scenario'] == pop]
                changed_mask = sub[changed_col].astype(bool)
                if not changed_mask.any():
                    continue
                base_f = sub.loc[changed_mask, 'base_loudest_f']
                syn_f  = sub.loc[changed_mask, syn_f_col]
                freq_rows.append({
                    'synthetic PTA':       syn_label,
                    'population scenario': pop,
                    'n changed sims':      int(changed_mask.sum()),
                    'median baseline loudest f [Hz]': f'{base_f.median():.3e}',
                    'median promoted source f [Hz]':  f'{syn_f.median():.3e}',
                    'promoted source higher f (%)':   round(
                        100 * (syn_f.to_numpy() > base_f.to_numpy()).mean(), 1),
                })

        display(pd.DataFrame(freq_rows))

Analysed 1500 simulations across 3 population scenarios.

=== Did the loudest source change between baseline and synthetic PTA? ===



,synthetic PTA,population scenario,n sims,source changed (n),source changed (%),median baseline rank\nof promoted source,median SNR ratio\n(syn / baseline)
0,2x_precision,optimistic,500,14,2.8,2.0,1.005
1,2x_precision,pessimistic,500,24,4.8,2.0,1.007
2,2x_precision,realistic,500,23,4.6,2.0,1.013
3,2x_precision_conserved,optimistic,500,12,2.4,2.0,0.999
4,2x_precision_conserved,pessimistic,500,16,3.2,2.0,1.001
5,2x_precision_conserved,realistic,500,21,4.2,2.0,1.001
6,4x_cad_2x_prec,optimistic,500,51,10.2,2.0,1.017
7,4x_cad_2x_prec,pessimistic,500,83,16.6,3.0,1.029
8,4x_cad_2x_prec,realistic,500,98,19.6,2.0,1.046
9,4x_cad_2x_prec_conserved,optimistic,500,98,19.6,2.0,0.812



=== Frequency of the promoted source vs baseline loudest (changed sims only) ===



,synthetic PTA,population scenario,n changed sims,median baseline loudest f [Hz],median promoted source f [Hz],promoted source higher f (%)
0,2x_precision,optimistic,14,1.103e-08,3.892e-08,100.0
1,2x_precision,pessimistic,24,1.224e-08,5.771e-08,100.0
2,2x_precision,realistic,23,1.021e-08,4.011e-08,100.0
3,2x_precision_conserved,optimistic,12,1.334e-08,3.892e-08,91.7
4,2x_precision_conserved,pessimistic,16,1.269e-08,5.399e-08,100.0
5,2x_precision_conserved,realistic,21,1.214e-08,4.317e-08,95.2
6,4x_cad_2x_prec,optimistic,51,9.669e-09,5.634e-08,100.0
7,4x_cad_2x_prec,pessimistic,83,1.197e-08,7.091e-08,100.0
8,4x_cad_2x_prec,realistic,98,1.246e-08,5.980e-08,100.0
9,4x_cad_2x_prec_conserved,optimistic,98,1.116e-08,4.322e-08,90.8


# SGWB Summary

In [85]:
import os

RUNS_DIR = "runs"
THRESHOLD = 5.94
OUT_DIR = "./sgwb_snr_out"

DESIRED_SCENARIOS = {"optimistic", "realistic", "pessimistic"}

def discover_configs(runs_dir, desired_scenarios):
    """
    Find every run directory under `runs_dir` whose canonical scenario
    (via canonical_style_key -- i.e. the part after any leading
    YYYY-MM-DD_ date prefix) is one of `desired_scenarios`, regardless of
    which date it was run on. This is what lets '2026-07-06_optimistic'
    and '2026-07-16_optimistic' both get included instead of only
    whichever single date was hardcoded.
    """
    all_dirs = sorted(
        d for d in os.listdir(runs_dir)
        if os.path.isdir(os.path.join(runs_dir, d))
    )
    return [d for d in all_dirs if canonical_style_key(d) in desired_scenarios]

CONFIGS = discover_configs(RUNS_DIR, DESIRED_SCENARIOS)
print(f'Found {len(CONFIGS)} run dir(s) matching {sorted(DESIRED_SCENARIOS)}:')
for c in CONFIGS:
    print(f'  {c}  ->  {canonical_style_key(c)}')

configs = CONFIGS if CONFIGS else None

NameError: name 'canonical_style_key' is not defined

In [ ]:
PTA_DISPLAY = {
    'baseline':                 'Standard (4.5 yr)',
    'baseline_forecast':        'Standard (9.0 yr)',
    '2x_precision':             r'2$\times$ Precision',
    '2x_precision_conserved':   r'2$\times$ Precision (TT Cons.)',
    '4x_cadence':               r'4$\times$ Cadence',
    '4x_cadence_conserved':     r'4$\times$ Cadence (TT Cons.)',
    '4x_cad_2x_prec':           r'Combined',
    '4x_cad_2x_prec_conserved': r'Combined (TT Cons.)',
}

SCENARIO_MASS_MAP = {
    'optimistic':  r'Heavy',
    'realistic':   r'Intermediate',
    'pessimistic': r'Light',
}

SCATTER_STYLES = {
    'optimistic':  {'color': 'limegreen', 'linestyle': '-',  'marker': 'o', 'linewidth': 1.6, 'ms': 5},
    'realistic':   {'color': 'magenta',   'linestyle': '--', 'marker': 's', 'linewidth': 1.6, 'ms': 5},
    'pessimistic': {'color': 'navy',      'linestyle': ':',  'marker': '^', 'linewidth': 1.8, 'ms': 5},
}

# Scenarios plotted left -> right when present in the data (baseline itself
# is the reference point, not a forecast scenario, so it is excluded here).
DEFAULT_SCENARIO_ORDER = [
    'baseline_forecast',
    '2x_precision',
    '2x_precision_conserved',
    '4x_cadence',
    '4x_cadence_conserved',
    '4x_cad_2x_prec',
    '4x_cad_2x_prec_conserved',
]

BASELINE_KEY = 'baseline'
FORECAST_KEY = 'baseline_forecast'

SIM_DIR_RE = re.compile(r'^sim(\d+)$')

DATE_PREFIX_RE = re.compile(r'^\d{4}-\d{2}-\d{2}_(.+)$')

def canonical_style_key(cfg: str) -> str:
    """
    Normalize a config label to its canonical scenario key so that reruns
    on different dates (e.g. '2026-07-16_optimistic', '2026-07-06_optimistic')
    are treated as the same scenario everywhere — grouping, styling,
    mass lookup, and legend labels.

    Strips a leading YYYY-MM-DD_ date prefix if present, then matches the
    remainder against the known scenario keys (SCATTER_STYLES /
    SCENARIO_MASS_MAP). Falls back to the lowercased remainder if there's
    no known match.
    """
    known = set(SCENARIO_MASS_MAP) | set(SCATTER_STYLES)
    m = DATE_PREFIX_RE.match(cfg)
    remainder = (m.group(1) if m else cfg).lower()
    if remainder in known:
        return remainder
    for key in known:
        if key in remainder:
            return key
    return remainder


In [ ]:
def discover_summary_files(runs_dir: str) -> List[Path]:
    """Find every metadata/sgwb_snr_summary.json under runs_dir."""
    root = Path(runs_dir)
    if not root.is_dir():
        raise FileNotFoundError(f'runs dir not found: {runs_dir}')
    return sorted(root.glob('**/metadata/sgwb_snr_summary.json'))


def parse_sim_id(summary_path: Path) -> Optional[int]:
    """summary_path = .../sim000/metadata/sgwb_snr_summary.json"""
    sim_dir = summary_path.parent.parent.name
    m = SIM_DIR_RE.match(sim_dir)
    return int(m.group(1)) if m else None


def parse_config_label(summary_path: Path, runs_dir: Path,
                        known_configs: Optional[List[str]] = None) -> str:
    sim_dir = summary_path.parent.parent
    config_dir = sim_dir.parent

    if config_dir.resolve() == runs_dir.resolve():
        return 'all'

    return canonical_style_key(config_dir.name)


def load_records(runs_dir: str,
                  configs: Optional[List[str]] = None) -> pd.DataFrame:
    """
    Returns a tidy DataFrame with one row per (run_dir, sim_id, scenario):
        config, run_dir, sim_id, scenario, sgwb_snr, Tspan_years,
        n_active_sub_chunks, n_total_sub_chunks, source_path

    `config` is the canonical scenario key (e.g. 'optimistic'), used for
    grouping/styling/plotting. `run_dir` is the literal date_scenario
    directory name (e.g. '2026-07-16_optimistic'), which is what actually
    identifies a unique simulation batch -- sim_id alone repeats across
    reruns on different dates, so (config, sim_id) is NOT a unique key,
    but (run_dir, sim_id) is.
    """
    root = Path(runs_dir)
    files = discover_summary_files(runs_dir)
    if not files:
        raise FileNotFoundError(f'no sgwb_snr_summary.json files found under {runs_dir}')

    known_canon = {canonical_style_key(c) for c in configs} if configs else None

    rows = []
    n_bad = 0
    unknown_config_warned = set()
    for fpath in files:
        sim_id = parse_sim_id(fpath)
        if sim_id is None:
            print(f'  WARNING: could not parse sim id from {fpath}, skipping')
            n_bad += 1
            continue

        sim_dir = fpath.parent.parent
        config_dir = sim_dir.parent
        run_dir = 'all' if config_dir.resolve() == root.resolve() else config_dir.name

        cfg_label = parse_config_label(fpath, root, configs)   # canonical, unchanged
        if known_canon and cfg_label not in known_canon and cfg_label != 'all':
            if cfg_label not in unknown_config_warned:
                print(f'  WARNING: config label "{cfg_label}" not in '
                      f'CONFIGS list, keeping it anyway ({fpath})')
                unknown_config_warned.add(cfg_label)
        try:
            with open(fpath) as fh:
                summary = json.load(fh)
        except (json.JSONDecodeError, OSError) as e:
            print(f'  WARNING: failed to read {fpath}: {e}, skipping')
            n_bad += 1
            continue
        for scenario, rec in summary.items():
            if not isinstance(rec, dict) or 'sgwb_snr' not in rec:
                continue
            rows.append({
                'config':              cfg_label,   # canonical, e.g. 'optimistic'
                'run_dir':             run_dir,     # raw, e.g. '2026-07-16_optimistic'
                'sim_id':              sim_id,
                'scenario':            scenario,
                'sgwb_snr':            rec.get('sgwb_snr', np.nan),
                'Tspan_years':         rec.get('Tspan_years', np.nan),
                'n_active_sub_chunks': rec.get('n_active_sub_chunks', np.nan),
                'n_total_sub_chunks':  rec.get('n_total_sub_chunks', np.nan),
                'source_path':         str(fpath),
            })
    if n_bad:
        print(f'  ({n_bad} file(s) skipped due to parse errors)')
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} (config, sim, scenario) rows from '
          f'{len(files) - n_bad} summary files '
          f'({df["config"].nunique()} config(s), '
          f'{df["run_dir"].nunique()} run dir(s), '
          f'{df["sim_id"].nunique()} sim(s), '
          f'{df["scenario"].nunique()} scenario(s))')
    return df

In [ ]:
def add_relative_columns(df: pd.DataFrame,
                          baseline_key: str = BASELINE_KEY,
                          forecast_key: str = FORECAST_KEY) -> pd.DataFrame:
    """
    For every (run_dir, sim_id), look up that sim's baseline and
    baseline_forecast SNR, then add columns to every row of that sim:
        baseline_snr, forecast_snr,
        delta_vs_baseline, delta_vs_forecast,
        ratio_vs_baseline, ratio_vs_forecast

    Keyed on run_dir (the actual date_scenario batch) rather than the
    canonical `config`, since sim_id repeats across reruns of the same
    scenario on different dates -- (config, sim_id) is not unique, but
    (run_dir, sim_id) is, and a sim's baseline must come from its own
    run, not some other date's run of the same scenario.
    """
    key_cols = ['run_dir', 'sim_id']
    base_lookup = (df[df['scenario'] == baseline_key]
                   .set_index(key_cols)['sgwb_snr'])
    fcst_lookup = (df[df['scenario'] == forecast_key]
                   .set_index(key_cols)['sgwb_snr'])
    idx = pd.MultiIndex.from_frame(df[key_cols])
    df = df.copy()
    df['baseline_snr'] = base_lookup.reindex(idx).values
    df['forecast_snr'] = fcst_lookup.reindex(idx).values
    df['delta_vs_baseline'] = df['sgwb_snr'] - df['baseline_snr']
    df['delta_vs_forecast'] = df['sgwb_snr'] - df['forecast_snr']
    with np.errstate(divide='ignore', invalid='ignore'):
        df['ratio_vs_baseline'] = df['sgwb_snr'] / df['baseline_snr']
        df['ratio_vs_forecast'] = df['sgwb_snr'] / df['forecast_snr']
    return df

def summarise(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """
    Per (canonical_config, scenario): median, and an empirical 90% CI
    (5th/95th percentile) computed directly from the per-sim values.
    Reruns of the same population scenario on different dates (e.g.
    '2026-07-06_optimistic' and '2026-07-16_optimistic') are combined into
    a single group here via canonical_style_key, instead of being kept as
    separate rows just because their directory names differ.
    """
    records = []
    df = df.copy()
    df['canonical_config'] = df['config'].map(canonical_style_key)
    for (cfg, scen), sub in df.groupby(['canonical_config', 'scenario']):
        arr = sub[value_col].dropna().values
        if len(arr) == 0:
            med, p5, p95, n = np.nan, np.nan, np.nan, 0
        else:
            med = np.median(arr)
            p5, p95 = np.percentile(arr, [5, 95])
            n = len(arr)
        records.append({
            'config':   cfg,     # now a canonical scenario key, e.g. 'optimistic'
            'scenario': scen,
            f'{value_col}_median': med,
            f'{value_col}_p5':     p5,
            f'{value_col}_p95':    p95,
            'n_sims':   n,
        })
    return pd.DataFrame.from_records(records)

In [ ]:
_CONFIG_X_OFFSET = {
    'optimistic':  -0.12,
    'realistic':    0.0,
    'pessimistic':  0.12,
}

def plot_delta_snr(df: pd.DataFrame,
                    out_path: Optional[str] = None,
                    scenario_order: Optional[List[str]] = None,
                    relative_to: str = 'baseline',
                    configs: Optional[List[str]] = None):
    """
    relative_to: 'baseline' -> plot delta_vs_baseline (change from the
                 current 4.5yr PTA)
                 'forecast' -> plot delta_vs_forecast (change from the
                 9.0yr "do nothing extra" forecast)
    Points show the median across sims; error bars show the empirical
    5th-95th percentile range (90% CI) of the per-sim distribution.
    """
    value_col = 'delta_vs_baseline' if relative_to == 'baseline' else 'delta_vs_forecast'
    ref_label = 'baseline' if relative_to == 'baseline' else 'forecast'
    ylabel = rf'$\mathrm{{S/N}}_{{\rm scenario}} - \mathrm{{S/N}}_{{\rm {ref_label}}}$'

    present_scenarios = [s for s in df['scenario'].unique()
                         if s not in (BASELINE_KEY, FORECAST_KEY)] \
                         if relative_to == 'baseline' else \
                        [s for s in df['scenario'].unique() if s != FORECAST_KEY
                         and s != BASELINE_KEY]
    order = scenario_order or DEFAULT_SCENARIO_ORDER
    x_scenarios = [s for s in order if s in present_scenarios]
    if not x_scenarios:
        print('  WARNING: no non-baseline scenarios found — skipping plot')
        return None

    summary = summarise(df, value_col)
    cfg_list = sorted({canonical_style_key(c) for c in (configs or df['config'].unique())})
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for cfg in cfg_list:
        sub = summary[summary['config'] == cfg]
        if sub.empty:
            continue
        medians, err_lo, err_hi, xs = [], [], [], []
        for i, scen in enumerate(x_scenarios):
            row = sub[sub['scenario'] == scen]
            if row.empty:
                continue
            med = row[f'{value_col}_median'].values[0]
            p5  = row[f'{value_col}_p5'].values[0]
            p95 = row[f'{value_col}_p95'].values[0]
            if np.isnan(med):
                continue
            xs.append(i)
            medians.append(med)
            err_lo.append(med - p5)
            err_hi.append(p95 - med)
        if not xs:
            continue

        canon = canonical_style_key(cfg)
        style = SCATTER_STYLES.get(canon, {'color': None, 'marker': 'o',
                                            'linewidth': 1.5, 'ms': 5})

        mass = SCENARIO_MASS_MAP.get(canon, '')
        if mass:
            label = f'$m_{{\\mathrm{{char}}}}$' + mass + r' $M_\odot$'
        else:
            label = cfg

        offset = _CONFIG_X_OFFSET.get(canon, 0.0)
        xs_offset = [x + offset for x in xs]
        yerr = np.array([err_lo, err_hi])

        # Points only — no line joining points across scenarios.
        ax.errorbar(xs_offset, medians, yerr=yerr, label=label,
                    color=style.get('color'), linestyle='none',
                    marker=style.get('marker', 'o'), linewidth=style.get('linewidth', 1.2),
                    markersize=style.get('ms', 4), capsize=2, elinewidth=0.9,
                    markeredgewidth=0.6, markeredgecolor='black')

    ax.axhline(0.0, color='k', linewidth=0.6, alpha=0.6, zorder=0)
    ax.set_xticks(range(len(x_scenarios)))
    ax.set_xticklabels([PTA_DISPLAY.get(s, s) for s in x_scenarios],
                       rotation=30, ha='right')
    ax.set_ylabel(ylabel)
    ax.tick_params(which='both', top=True, right=True)
    ax.legend(loc='best', handletextpad=0.3, borderaxespad=0.3)
    fig.tight_layout(pad=0.4)
    if out_path:
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'  Figure saved: {out_path}')
    plt.show()
    return fig

In [ ]:
def build_detection_fraction_table(df: pd.DataFrame,
                                    threshold: float,
                                    scenario_order: Optional[List[str]] = None,
                                    configs: Optional[List[str]] = None) -> str:
    """
    For each (config, scenario), fraction of sims with sgwb_snr >= threshold.
    Returns a LaTeX tabular string.
    """
    order = scenario_order or DEFAULT_SCENARIO_ORDER
    present = [s for s in df['scenario'].unique()]
    scenarios = [s for s in order if s in present]
    scenarios += [s for s in present if s not in scenarios and s != BASELINE_KEY]
    if BASELINE_KEY in present and BASELINE_KEY not in scenarios:
        scenarios = [BASELINE_KEY] + scenarios

    cfg_list = configs or sorted(df['config'].unique())

    frac = (df.assign(detected=df['sgwb_snr'] >= threshold)
              .assign(canonical_config=lambda d: d['config'].map(canonical_style_key))
              .groupby(['canonical_config', 'scenario'])['detected']
              .mean()
              .unstack('scenario'))

    n_cols = len(scenarios)
    lines = []
    lines.append(r'\begin{table}[t]')
    lines.append(r'\centering')
    lines.append(r'\begin{tabular}{l' + 'c' * n_cols + '}')
    lines.append(r'\toprule')
    header = ['Config'] + [PTA_DISPLAY.get(s, s.replace('_', r'\_')) for s in scenarios]
    lines.append(' & '.join(header) + r' \\')
    lines.append(r'\midrule')
    for cfg in cfg_list:
        canon = canonical_style_key(cfg)
        mass = SCENARIO_MASS_MAP.get(canon, '')
        cfg_label = f'$m_{{\\mathrm{{char}}}}$' + mass + r' $M_\odot$' if mass else cfg
        cells = [cfg_label]
        for scen in scenarios:
            val = frac.loc[canon, scen] if (canon in frac.index and scen in frac.columns) else np.nan
            cells.append('--' if pd.isna(val) else f'{val:.2f}')
        lines.append(' & '.join(cells) + r' \\')
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(rf'\caption{{Fraction of realizations with SGWB OS SNR $\geq {threshold:.1f}$.}}')
    lines.append(r'\label{tab:detection_fraction}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)


In [ ]:
import os

configs = CONFIGS if CONFIGS else None
if OUT_DIR:
    os.makedirs(OUT_DIR, exist_ok=True)

print(f'Scanning {RUNS_DIR} ...')
df = load_records(RUNS_DIR, configs=configs)
df = add_relative_columns(df)

if OUT_DIR:
    csv_path = os.path.join(OUT_DIR, 'sgwb_snr_all_records.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Tidy table saved: {csv_path}')

df.head()


In [ ]:
SELECTED_SCENARIOS = [
    '2x_precision',
    '2x_precision_conserved',
    '4x_cadence',
    '4x_cadence_conserved',
    '4x_cad_2x_prec',
    '4x_cad_2x_prec_conserved',
]

fig_baseline = plot_delta_snr(
    df,
    out_path=os.path.join(OUT_DIR, 'sgwb_snr_delta_vs_baseline.pdf') if OUT_DIR else None,
    relative_to='baseline',
    configs=configs,
    scenario_order=SELECTED_SCENARIOS,
)

In [ ]:
SELECTED_SCENARIOS = [
    '2x_precision',
    '2x_precision_conserved',
    '4x_cadence',
    '4x_cadence_conserved',
    '4x_cad_2x_prec',
    '4x_cad_2x_prec_conserved',
]

fig_forecast = plot_delta_snr(
    df,
    out_path=os.path.join(OUT_DIR, 'sgwb_snr_delta_vs_forecast.pdf') if OUT_DIR else None,
    relative_to='forecast',
    configs=configs,
    scenario_order=SELECTED_SCENARIOS,
)


In [ ]:
tex = build_detection_fraction_table(df, threshold=THRESHOLD, configs=configs)
print(tex)

if OUT_DIR:
    tex_path = os.path.join(OUT_DIR, 'detection_fraction_table.tex')
    with open(tex_path, 'w') as fh:
        fh.write(tex + '\n')
    print(f'\nLaTeX table saved: {tex_path}')


In [ ]:
df.groupby(['config', 'scenario'])[['sgwb_snr', 'delta_vs_baseline', 'delta_vs_forecast']].agg(['mean', 'std'])


In [108]:
import textwrap

# Explicit line breaks, so a scenario and its "(TT Cons.)" variant wrap
# consistently (same break point) regardless of whether either one alone
# would trigger a wrap under a length threshold.
PTA_DISPLAY_WRAPPED = {
    'baseline':                 'Standard\n(4.5 yr)',
    'baseline_forecast':        'Standard\n(9.0 yr)',
    '2x_precision':             '2$\\times$ Precision\n',
    '2x_precision_conserved':   '2$\\times$ Precision\n(TT Cons.)',
    '4x_cadence':               '4$\\times$ Cadence\n',
    '4x_cadence_conserved':     '4$\\times$ Cadence\n(TT Cons.)',
    '4x_cad_2x_prec':           'Combined\n',
    '4x_cad_2x_prec_conserved': 'Combined\n(TT Cons.)',
}

def _wrap_label(label: str, width: int = 14) -> str:
    """
    Break a scenario display label onto two lines if it's long, preferring
    to break right before a trailing parenthetical like '(TT Cons.)' rather
    than mid-word. Falls back to textwrap's normal word-wrap if there's no
    parenthetical to hinge on.
    """
    if len(label) <= width:
        return label
    if '(' in label:
        head, _, tail = label.partition('(')
        return f'{head.rstrip()}\n({tail}'
    wrapped = textwrap.wrap(label, width=width)
    return '\n'.join(wrapped) if wrapped else label


def plot_delta_snr_horizontal(df: pd.DataFrame,
                               out_path: Optional[str] = None,
                               scenario_order: Optional[List[str]] = None,
                               relative_to: str = 'baseline',
                               configs: Optional[List[str]] = None,
                               row_height: float = 0.34,
                               wrap_width: int = 14):
    value_col = 'delta_vs_baseline' if relative_to == 'baseline' else 'delta_vs_forecast'
    ref_label = 'baseline' if relative_to == 'baseline' else 'forecast'
    xlabel = rf'$(\mathrm{{S/N}})_{{\rho,\rm scenario}} - (\mathrm{{S/N}})_{{\rho,\rm {ref_label}}}$'

    present_scenarios = [s for s in df['scenario'].unique()
                         if s not in (BASELINE_KEY, FORECAST_KEY)] \
                         if relative_to == 'baseline' else \
                        [s for s in df['scenario'].unique() if s != FORECAST_KEY
                         and s != BASELINE_KEY]
    order = scenario_order or DEFAULT_SCENARIO_ORDER
    y_scenarios = [s for s in order if s in present_scenarios]
    if not y_scenarios:
        print('  WARNING: no non-baseline scenarios found — skipping plot')
        return None

    summary = summarise(df, value_col)
    cfg_list = (
        sorted({canonical_style_key(c) for c in configs})
        if configs else
        sorted(df['config'].unique())
    )

    y_scenarios_plot = list(reversed(y_scenarios))
    y_pos = np.arange(len(y_scenarios_plot))

    n_cfg = len(cfg_list)
    spread = 0.28
    cfg_y_offset = {
        cfg: (i - (n_cfg - 1) / 2) * (spread / max(n_cfg - 1, 1))
        for i, cfg in enumerate(cfg_list)
    }

    # Tighter height: less per-row space, smaller fixed margin.
    fig_height = row_height * len(y_scenarios_plot) + 0.5
    fig, ax = plt.subplots(figsize=(3.5, fig_height))

    for cfg in cfg_list:
        sub = summary[summary['config'] == cfg]
        if sub.empty:
            continue
        medians, err_lo, err_hi, ys = [], [], [], []
        for i, scen in enumerate(y_scenarios_plot):
            row = sub[sub['scenario'] == scen]
            if row.empty:
                continue
            med = row[f'{value_col}_median'].values[0]
            p5  = row[f'{value_col}_p5'].values[0]
            p95 = row[f'{value_col}_p95'].values[0]
            if np.isnan(med):
                continue
            ys.append(i)
            medians.append(med)
            err_lo.append(med - p5)
            err_hi.append(p95 - med)
        if not ys:
            continue

        canon = canonical_style_key(cfg)
        style = SCATTER_STYLES.get(canon, {'color': None, 'marker': 'o',
                                            'linewidth': 1.5, 'ms': 5})
        mass = SCENARIO_MASS_MAP.get(canon, '')
        label = (f'$m_{{\\mathrm{{char}}}}$' + mass + r' $M_\odot$') if mass else cfg

        ys_offset = [y + cfg_y_offset[cfg] for y in ys]
        xerr = np.array([err_lo, err_hi])

        ax.errorbar(medians, ys_offset, xerr=xerr, label=label,
                    color=style.get('color'), linestyle='none',
                    marker=style.get('marker', 'o'), linewidth=style.get('linewidth', 1.2),
                    markersize=style.get('ms', 4), capsize=2, elinewidth=0.9,
                    markeredgewidth=0.6, markeredgecolor='black')

    ax.axvline(0.0, color='k', linewidth=0.6, alpha=0.6, zorder=0)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([
        PTA_DISPLAY_WRAPPED.get(s) or _wrap_label(PTA_DISPLAY.get(s, s), width=wrap_width)
        for s in y_scenarios_plot
    ])
    ax.set_xlabel(xlabel)
    ax.tick_params(which='both', top=True, right=True)
    ax.legend(loc='best', handletextpad=0.3, borderaxespad=0.3)

    # Tight y-margins so rows sit close to the axes edges too, not just
    # close to each other.
    ax.set_ylim(-0.5, len(y_scenarios_plot) - 0.5)
    fig.tight_layout(pad=0.3)
    if out_path:
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'  Figure saved: {out_path}')
    plt.show()
    return fig

In [109]:
SELECTED_SCENARIOS = [
    '2x_precision',
    '2x_precision_conserved',
    '4x_cadence',
    '4x_cadence_conserved',
    '4x_cad_2x_prec',
    '4x_cad_2x_prec_conserved',
]

fig_baseline = plot_delta_snr_horizontal(
    df,
    out_path=os.path.join(OUT_DIR, 'sgwb_snr_delta_vs_forecast_horizontal.pdf') if OUT_DIR else None,
    relative_to='forecast',
    configs=configs,
    scenario_order=SELECTED_SCENARIOS,
)

  Figure saved: ./sgwb_snr_out/sgwb_snr_delta_vs_forecast_horizontal.pdf


# NANOGrav Comparison Plots

In [ ]:
from apj_style import APJ_COL_WIDTH, APJ_COL_WIDTH_DOUBLE, APJ_FIGSIZE

FIGSIZE_SINGLE_COL = APJ_FIGSIZE                       # (3.5, 2.8), was (3.5, 2.5)
FIGSIZE_DOUBLE_COL = (APJ_COL_WIDTH_DOUBLE, 2.8)       # was (7.0, 4.2)
FIGSIZE = FIGSIZE_SINGLE_COL

# =============================================================================
# Parameters — edit these
# =============================================================================

RUN_DIR = "nanograv_comp_runs/2026-07-27_realistic"

SIM_IDS = None

ACTIVE_SHARDS_ONLY = True

MASS_BINS = np.arange(7.5, 10.6, 0.5)
N_FREQ_BINS = 30

CANDIDATE_FREQUENCIES = [14e-9, 21e-9]
CANDIDATE_LABELS      = ['14 nHz', '21 nHz']
CANDIDATE_MASSES      = [9.75, 10.05]
# NEW: redshift associated with each candidate. Order must match
# CANDIDATE_FREQUENCIES / CANDIDATE_LABELS / CANDIDATE_MASSES above.
CANDIDATE_REDSHIFTS   = [0.074, 0.379]

CI_LOW_PCT, CI_HIGH_PCT = 2.5, 97.5

# Base path — per-figure suffixes are appended automatically (see bottom).
OUT_PNG_BASE = "figures/freq_mass_z_hist_population"

# =============================================================================
# Fixed bin edges — must match stage1_setup.py exactly
# =============================================================================

_HIST_F_BINS    = np.logspace(-9, -6, 61)
_HIST_MTOT_BINS = np.arange(7.5, 10.61, 0.1)
_HIST_Z_BINS    = np.linspace(0.0, 3.0, 61)
_HIST_Z_CENTERS = 0.5 * (_HIST_Z_BINS[:-1] + _HIST_Z_BINS[1:])

# =============================================================================
# Discover sims + shard histogram files
# =============================================================================

def _sim_dirs(run_dir, sim_ids=None):
    run_dir = Path(run_dir)
    all_sims = sorted(run_dir.glob('sim[0-9][0-9][0-9]'))
    if sim_ids is None:
        return all_sims
    wanted = {f'sim{s:03d}' for s in sim_ids}
    return [d for d in all_sims if d.name in wanted]


def _active_shard_set(sim_dir):
    handoff_path = sim_dir / 'metadata' / 'phase_handoff.json'
    if not handoff_path.is_file():
        return None
    with open(handoff_path) as fh:
        handoff = json.load(fh)
    return {tuple(x) for x in handoff.get('active_shard_ids', [])}


_HIST_FNAME_RE = re.compile(r'^(\d{3})_(\d{3})\.npz$')

def _hist_files_for_sim(sim_dir, active_only):
    hist_dir = sim_dir / 'metadata' / 'freq_mass_z_hist'
    if not hist_dir.is_dir():
        return []

    active_set = _active_shard_set(sim_dir) if active_only else None
    if active_only and active_set is None:
        print(f'  ⚠ {sim_dir.name}: no phase_handoff.json yet — skipping '
              f'(set ACTIVE_SHARDS_ONLY=False to include unfiltered anyway)')
        return []

    files = []
    for p in sorted(hist_dir.glob('*.npz')):
        m = _HIST_FNAME_RE.match(p.name)
        if not m:
            print(f'  ⚠ Skipping unrecognized file: {p.name}')
            continue
        shard = (int(m.group(1)), int(m.group(2)))
        if active_only and shard not in active_set:
            continue
        files.append(p)
    return files


sim_dirs = _sim_dirs(RUN_DIR, SIM_IDS)
if not sim_dirs:
    raise FileNotFoundError(f'No sim### folders found under {RUN_DIR}')

hist_files_by_sim = {d.name: _hist_files_for_sim(d, ACTIVE_SHARDS_ONLY) for d in sim_dirs}
n_total_files = sum(len(v) for v in hist_files_by_sim.values())
print(f'Found {len(sim_dirs)} sim folder(s) under {RUN_DIR}')
for name, files in hist_files_by_sim.items():
    print(f'  {name}: {len(files)} histogram shard file(s)')
print(f'Total histogram shard files to sum: {n_total_files}')
if n_total_files == 0:
    raise RuntimeError(
        'No histogram files found. Did stage 1 run with --freq-mass-hist? '
        'Or, if ACTIVE_SHARDS_ONLY=True, has stage 2 finished for these sims?'
    )

# =============================================================================
# Load per-sim histograms — sims kept separate (no cross-sim summing here).
# =============================================================================

hist_shape = (len(_HIST_F_BINS) - 1, len(_HIST_MTOT_BINS) - 1, len(_HIST_Z_BINS) - 1)

per_sim_counts = {}
bad_files = []

for sim_name, files in hist_files_by_sim.items():
    sim_counts = np.zeros(hist_shape, dtype=np.float64)
    n_loaded_this_sim = 0

    for fpath in files:
        data = None
        try:
            data = np.load(fpath)
        except ValueError:
            try:
                data = np.load(fpath, allow_pickle=True)
            except Exception as e:
                print(f'  ⚠ Skipping unreadable/corrupt file {fpath}: {e}')
                bad_files.append(fpath)
                continue
        except Exception as e:
            print(f'  ⚠ Skipping unreadable/corrupt file {fpath}: {e}')
            bad_files.append(fpath)
            continue

        try:
            if not (np.allclose(data['f_bins'], _HIST_F_BINS)
                    and np.allclose(data['mtot_log_bins'], _HIST_MTOT_BINS)
                    and np.allclose(data['z_bins'], _HIST_Z_BINS)):
                print(f'  ⚠ Skipping {fpath}: bin edges do not match expected fixed edges')
                bad_files.append(fpath)
                continue
            sim_counts += data['counts']
        finally:
            data.close()

        n_loaded_this_sim += 1

    if n_loaded_this_sim == 0:
        print(f'  ⚠ {sim_name}: no valid shard files loaded — excluding this sim entirely')
        continue

    per_sim_counts[sim_name] = sim_counts

n_sims_used = len(per_sim_counts)
print(f'\nUsable sims: {n_sims_used} / {len(sim_dirs)}')
if bad_files:
    print(f'⚠ {len(bad_files)} file(s) could not be read/used and were skipped:')
    for f in bad_files:
        print(f'   {f}')
if n_sims_used < 2:
    raise RuntimeError(
        f'Need at least 2 usable sims to compute a median + credible interval '
        f'across sims — only found {n_sims_used}.'
    )

sim_names_used = sorted(per_sim_counts.keys())

# =============================================================================
# Shared f-rebinning setup (independent of z_max)
# =============================================================================

f_fine_centers    = np.sqrt(_HIST_F_BINS[:-1] * _HIST_F_BINS[1:])
mtot_fine_centers = 0.5 * (_HIST_MTOT_BINS[:-1] + _HIST_MTOT_BINS[1:])

freq_bins_display = np.logspace(
    np.log10(_HIST_F_BINS[0]), np.log10(_HIST_F_BINS[-1]), N_FREQ_BINS
)
freq_bin_centers_display = 0.5 * (freq_bins_display[:-1] + freq_bins_display[1:])

_fine_idx = np.clip(
    np.digitize(f_fine_centers, freq_bins_display) - 1, 0, N_FREQ_BINS - 2
)

def _rebin_counts_by_f(counts_1d_by_f_fine):
    out = np.zeros(len(freq_bin_centers_display))
    for i, w in zip(_fine_idx, counts_1d_by_f_fine):
        out[i] += w
    return out

def _median_ci(stack_2d):
    median = np.median(stack_2d, axis=0)
    lo = np.percentile(stack_2d, CI_LOW_PCT, axis=0)
    hi = np.percentile(stack_2d, CI_HIGH_PCT, axis=0)
    return median, lo, hi

# =============================================================================
# NEW: compute median + CI restricted to a maximum redshift (z_max=None means
# use the full stored z range, 0-3, i.e. no cut at all).
# =============================================================================

def _compute_median_ci(z_max=None):
    """
    Returns (total_median, total_lo, total_hi, mass_bin_series) where
    mass_bin_series is a list of (lo_m, hi_m, median, lo, hi), restricted to
    binaries with z <= z_max (or the full 0-3 range if z_max is None).
    """
    if z_max is None:
        z_mask = np.ones(len(_HIST_Z_CENTERS), dtype=bool)
    else:
        z_mask = _HIST_Z_CENTERS <= z_max
        if not np.any(z_mask):
            raise ValueError(
                f'z_max={z_max} is below the lowest histogram z bin center '
                f'({_HIST_Z_CENTERS[0]:.3f}) — no binaries would be included.'
            )

    # Sum each sim's 3D counts over the selected z slice -> (f, mtot) per sim
    per_sim_f_mtot = {
        s: per_sim_counts[s][:, :, z_mask].sum(axis=2) for s in sim_names_used
    }

    total_display_per_sim = np.stack([
        _rebin_counts_by_f(per_sim_f_mtot[s].sum(axis=1))
        for s in sim_names_used
    ])
    total_median, total_lo, total_hi = _median_ci(total_display_per_sim)

    mass_bin_series = []
    for i in range(len(MASS_BINS) - 1):
        lo_m, hi_m = MASS_BINS[i], MASS_BINS[i + 1]
        mtot_mask = (mtot_fine_centers >= lo_m) & (mtot_fine_centers < hi_m)
        if not np.any(mtot_mask):
            zeros = np.zeros(len(freq_bin_centers_display))
            mass_bin_series.append((lo_m, hi_m, zeros, zeros.copy(), zeros.copy()))
            continue
        stacked = np.stack([
            _rebin_counts_by_f(per_sim_f_mtot[s][:, mtot_mask].sum(axis=1))
            for s in sim_names_used
        ])
        median, lo, hi = _median_ci(stacked)
        mass_bin_series.append((lo_m, hi_m, median, lo, hi))

    return total_median, total_lo, total_hi, mass_bin_series

print(f'\nWill compute median + [{CI_LOW_PCT}, {CI_HIGH_PCT}] percentile CI '
      f'across {n_sims_used} independent sims, for each requested z cut.')

# =============================================================================
# Plotting — factored into a function so it can be reused for the overall
# figure and each redshift-cumulative candidate figure.
# =============================================================================

_MASS_COLORS = [
    "#A6FF00",   # lime       (matches STRATEGY_COLORS["unchanged"])
    "#FF2DAA",   # magenta    (matches STRATEGY_COLORS["cadence"])
    # "navy",      #            (matches STRATEGY_COLORS["precision"])
    "#00E5FF",   # cyan       (matches STRATEGY_COLORS["combined"])
    "#FF8C00",   # vivid orange
    "#9B30FF",   # violet
    "#00FFA3",   # spring green
]
_CANDIDATE_COLORS = ["#FFD700", "#FF3B3B", "#FF69B4", "#00FFA3"]

def _make_plot(total_median, total_lo, total_hi, mass_bin_series,
                candidate_indices, out_path, z_max=None, y_axis_label = r'Number of Binaries'):
    """
    candidate_indices: indices into CANDIDATE_FREQUENCIES/LABELS/MASSES to
    draw as vertical lines on this particular figure (lets each redshift-cut
    figure show only the candidates that are "in range" at that depth).
    """
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    fig, ax = plt.subplots(figsize=FIGSIZE)

    def _plot_series(x, median, lo, hi, color, label, linestyle='-'):
        ax.plot(x, median, color=color, linewidth=1.2, linestyle=linestyle,
                 drawstyle='steps-mid', label=label)
        yerr_lo = np.clip(median - lo, 0, None)
        yerr_hi = np.clip(hi - median, 0, None)
        ax.errorbar(x, median, yerr=[yerr_lo, yerr_hi], fmt='none',
                    ecolor=color, elinewidth=0.6, capsize=1.5, alpha=0.6)

    _plot_series(freq_bin_centers_display, total_median, total_lo, total_hi,
                 color='k', label='Total', linestyle='-')

    for i, (lo_m, hi_m, median, lo, hi) in enumerate(mass_bin_series):
        if median.sum() == 0:
            continue
        color = _MASS_COLORS[i % len(_MASS_COLORS)]
        label = r"$%.1f < \log_{10}\!\left(M_{\rm tot}/\mathrm{M}_\odot\right) < %.1f$" % (lo_m, hi_m)
        _plot_series(freq_bin_centers_display, median, lo, hi, color=color,
                     label=label, linestyle='--')

    # =========================================================================
    # Candidate annotations: vertical line (no legend entry) + a label
    # stacked in a fixed left-hand column, connected to its line by an arrow.
    # All labels share the same x anchor (axes fraction, left side) and are
    # stacked at evenly spaced y positions -- so only the arrow length differs
    # (a function of how far right that candidate's frequency sits), not the
    # label's horizontal position or the arrow's vertical placement logic.
    # =========================================================================
    from matplotlib.transforms import blended_transform_factory

    TEXT_X_FRAC = 0.22
    TEXT_Y_TOP  = 0.95
    TEXT_Y_STEP = 0.09

    blended = blended_transform_factory(ax.transData, ax.transAxes)

    for stack_pos, i in enumerate(candidate_indices):
        f     = CANDIDATE_FREQUENCIES[i]
        label = CANDIDATE_LABELS[i]
        color = _CANDIDATE_COLORS[i % len(_CANDIDATE_COLORS)]

        # Vertical line only -- no label here, so it doesn't appear in the
        # main legend (labels are drawn as annotations instead).
        ax.axvline(f, color=color, linestyle='-', lw=1.5, alpha=1)

        text_y = TEXT_Y_TOP - stack_pos * TEXT_Y_STEP

        ax.annotate(
            label,
            xy=(f, text_y), xycoords=blended,
            xytext=(TEXT_X_FRAC, text_y), textcoords='axes fraction',
            ha='left', va='center',
            color=color,
            arrowprops=dict(
                arrowstyle='-|>', color=color,
                lw=1.0, shrinkA=2, shrinkB=2,
                connectionstyle='arc3,rad=0.0',
            ),
        )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$f$ [Hz]')
    ax.set_ylabel(y_axis_label)

    ax.xaxis.set_major_locator(LogLocator(base=10.0, numticks=15))
    ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100))
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=15))
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100))
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True,
                   bottom=True, left=True, length=4, width=0.8)
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True,
                   bottom=True, left=True, length=2, width=0.6)

    ax.set_xlim(2.08e-9, 300e-9)
    nonzero = total_median[total_median > 0]
    ax.set_ylim(0.2, (nonzero.max() if len(nonzero) else 1) * 15.5)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.8)

    # Legend now only ever contains "Total" + mass-bin entries, since
    # candidate axvlines no longer pass label=... (their labels are drawn as
    # annotations above instead).
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), frameon=False,
              framealpha=0.9, edgecolor='black', handlelength=2.2,
              borderpad=0.5, labelspacing=0.4)

    plt.tight_layout(pad=0.5)
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0.02)
    print(f'Saved: {out_path}')

    z_note = 'all z' if z_max is None else f'z <= {z_max:.3f}'
    print(f'  ({z_note}; median across {n_sims_used} sims, '
          f'{CI_LOW_PCT:.1f}\u2013{CI_HIGH_PCT:.1f} percentile CI; '
          f'candidates shown: {[CANDIDATE_LABELS[i] for i in candidate_indices] or "none"})')

    plt.show()
    plt.close(fig)

# =============================================================================
# Build the figure sequence:
#   1. Overall: full z range (0-3), all candidates shown.
#   2. One figure per candidate, sorted by ascending redshift, each cut to
#      that candidate's z and cumulatively including all candidates with
#      redshift <= that cutoff (so figure k shows candidates 1..k).
# =============================================================================

if CANDIDATE_FREQUENCIES:
    n_candidates = len(CANDIDATE_FREQUENCIES)
    for name, lst in [('CANDIDATE_LABELS', CANDIDATE_LABELS),
                       ('CANDIDATE_MASSES', CANDIDATE_MASSES),
                       ('CANDIDATE_REDSHIFTS', CANDIDATE_REDSHIFTS)]:
        if lst is None or len(lst) != n_candidates:
            raise ValueError(
                f'{name} must be the same length as CANDIDATE_FREQUENCIES '
                f'({n_candidates}); got {None if lst is None else len(lst)}.'
            )

    # Sort candidate indices by ascending redshift so "first candidate
    # redshift" / "second candidate redshift" is well-defined regardless of
    # the order the lists were written in.
    order = sorted(range(n_candidates), key=lambda i: CANDIDATE_REDSHIFTS[i])
else:
    order = []

# --- Figure 1: overall, full z range, all candidates ---
total_median, total_lo, total_hi, mass_bin_series = _compute_median_ci(z_max=None)
_make_plot(total_median, total_lo, total_hi, mass_bin_series,
           candidate_indices=list(order),
           out_path=f'{OUT_PNG_BASE}_all.pdf',
           z_max=None)

# --- Figures 2..N+1: cumulative per-candidate redshift cuts ---
for k in range(len(order)):
    included = order[:k + 1]          # this candidate + all lower-redshift ones
    z_cut_idx = order[k]
    z_max = CANDIDATE_REDSHIFTS[z_cut_idx]
    y_label = rf'Number of Binaries ($z<{z_max}$)' 
    total_median, total_lo, total_hi, mass_bin_series = _compute_median_ci(z_max=z_max)

    safe_label = re.sub(r'[^a-zA-Z0-9]+', '_', CANDIDATE_LABELS[z_cut_idx]).strip('_').lower()
    out_path = f'{OUT_PNG_BASE}_zmax_{safe_label}.pdf'

    _make_plot(total_median, total_lo, total_hi, mass_bin_series,
               candidate_indices=included,
               out_path=out_path,
               z_max=z_max,
                y_axis_label = y_label)

  ⚠ sim700: no phase_handoff.json yet — skipping (set ACTIVE_SHARDS_ONLY=False to include unfiltered anyway)
  ⚠ sim706: no phase_handoff.json yet — skipping (set ACTIVE_SHARDS_ONLY=False to include unfiltered anyway)
  ⚠ sim718: no phase_handoff.json yet — skipping (set ACTIVE_SHARDS_ONLY=False to include unfiltered anyway)
  ⚠ sim719: no phase_handoff.json yet — skipping (set ACTIVE_SHARDS_ONLY=False to include unfiltered anyway)
  ⚠ sim720: no phase_handoff.json yet — skipping (set ACTIVE_SHARDS_ONLY=False to include unfiltered anyway)
  ⚠ sim721: no phase_handoff.json yet — skipping (set ACTIVE_SHARDS_ONLY=False to include unfiltered anyway)
  ⚠ sim722: no phase_handoff.json yet — skipping (set ACTIVE_SHARDS_ONLY=False to include unfiltered anyway)
Found 52 sim folder(s) under nanograv_comp_runs/2026-07-27_realistic
  sim700: 0 histogram shard file(s)
  sim701: 2 histogram shard file(s)
  sim702: 1 histogram shard file(s)
  sim703: 1 histogram shard file(s)
  sim704: 1 histogram sh

# Poster plots

In [52]:
# ── Poster version: Detection fraction (color = strategy, not population) ────
_STRATEGY_STYLES = {
    'baseline':        {'color': '#A6FF00', 'marker': 'o'},  # lime - Unchanged
    '4x_cadence':      {'color': '#FF2DAA', 'marker': 'o'},  # magenta - Higher Cadence
    'hi_prec':         {'color': '#FF9800', 'marker': 'o'},  # orange - Higher Precision
    '4x_cad_2x_prec':  {'color': '#00E5FF', 'marker': 'o'},  # cyan - Combined
}
# Nice display names in the desired order
_PTA_DISPLAY_NAMES = {
    'baseline': 'Unchanged',
    'hi_prec': 'Higher\nPrecision',
    '4x_cadence': 'Higher\nCadence',
    '4x_cad_2x_prec': 'Higher \nCadence & \nPrecision',
}
_LEGEND_LABELS = {
    'baseline': 'Unchanged',
    'hi_prec': 'Higher Precision',
    '4x_cadence': 'Higher Cadence',
    '4x_cad_2x_prec': 'Combined',
}
# Population now distinguished by marker shape instead of color
_POP_MARKERS = {
    "optimistic":  "o",   # circle
    "realistic":   "s",   # square
    "pessimistic": "^",   # triangle
}
_POP_DISPLAY_NAMES = {
    "optimistic": "Heavy",
    "realistic": "Intermediate",
    "pessimistic": "Light",
}

if not det_df.empty:
    pta_order = [
        'baseline',
        'hi_prec',
        '4x_cadence',
        '4x_cad_2x_prec',
    ]
    pta_order = [p for p in pta_order if p in det_df['pta'].unique()]
    x_pos = np.arange(len(pta_order))

    fig, ax = plt.subplots(figsize=(9, 5.2))
    fig.patch.set_facecolor("black")
    ax.set_facecolor("black")

    # ----------------------------------------------------------------
    # Plot: one point per (population, strategy) combo.
    # Color = strategy (consistent across all populations).
    # Marker shape = population.
    # X position = strategy (jittered slightly per population so points
    # at the same x don't fully overlap).
    # ----------------------------------------------------------------
    jitter = {"optimistic": -0.12, "realistic": 0.0, "pessimistic": 0.12}

    pop_handles = []
    for pop in ("optimistic", "realistic", "pessimistic"):
        if pop not in pop_scenarios:
            continue
        sub = det_df[det_df["scenario"] == pop].set_index("pta")

        xs, ys, colors = [], [], []
        for i, p in enumerate(pta_order):
            if p not in sub.index:
                continue
            xs.append(x_pos[i] + jitter[pop])
            ys.append(float(sub.loc[p, "det_fraction"]))
            colors.append(_STRATEGY_STYLES[p]["color"])

        h = ax.scatter(
            xs,
            ys,
            s=220,
            c=colors,
            marker=_POP_MARKERS[pop],
            edgecolor='0.85',
            linewidth=1.8,
            zorder=3,
            clip_on=False,
        )
        # Proxy handle for population legend (marker shape, neutral color)
        pop_handles.append(
            plt.Line2D(
                [], [], marker=_POP_MARKERS[pop], color='white',
                linestyle='None', markersize=14,
                markerfacecolor='none', markeredgecolor='white', markeredgewidth=1.8,
                label=_POP_DISPLAY_NAMES[pop],
            )
        )

    # ----------------------------------------------------------------
    # Axes
    # ----------------------------------------------------------------
    ax.set_xticks(x_pos)
    ax.set_xticklabels(
        [_PTA_DISPLAY_NAMES[p] for p in pta_order],
        fontsize=18,
        color="white"
    )
    ax.set_ylabel(
        "Detection Fraction",
        fontsize=22,
        color="white",
        labelpad=18,
    )
    ax.set_xlabel(
        "Observing Strategy (NANOGrav 15-year like PTA)",
        fontsize=22,
        color="white",
        labelpad=18,
    )
    ax.tick_params(axis='x', pad=12)
    ax.tick_params(axis='y', pad=8)

    ax.set_ylim(0, 0.85)
    ax.tick_params(axis='y', colors='white', labelsize=18)
    ax.tick_params(axis='x', colors='white', length=0)
    for spine in ax.spines.values():
        spine.set_color("white")
        spine.set_linewidth(2)
    ax.grid(axis='y', color='white', alpha=0.15, linewidth=1)

    # ----------------------------------------------------------------
    # Two legends: strategy (color swatches) + population (marker shapes)
    # ----------------------------------------------------------------
    strategy_handles = [
        plt.Line2D(
            [], [], marker='o', color='none', linestyle='None', markersize=14,
            markerfacecolor=_STRATEGY_STYLES[p]["color"], markeredgecolor='0.85',
            markeredgewidth=1.5, label=_LEGEND_LABELS[p],
        )
        for p in pta_order
    ]


    leg2 = ax.legend(
        handles=pop_handles,
        title="Population Model",
        fontsize=14,
        title_fontsize=16,
        frameon=False,
        loc='lower right',
        # bbox_to_anchor=(0.0, 1.0),
    )
    leg2.get_title().set_color("white")
    leg2.get_title().set_fontweight("bold")
    for text in leg2.get_texts():
        text.set_color("white")

    plt.subplots_adjust(left=0.14, bottom=0.18)
    fig.tight_layout()
    fig.savefig(
        figure_dir / "synpta_detection_fraction_scatter_poster.png",
        dpi=1500,
        facecolor="black",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

In [ ]:
# ------------------------------------------------------------------
# Poster figure: 3 panels, each with 4 histograms.
# Colors now map to OBSERVING STRATEGY (same across all panels),
# population is distinguished only by the white panel title.
# ------------------------------------------------------------------
POP_TITLES = {
    "optimistic": "Heavy",
    "realistic":  "Intermediate",
    "pessimistic": "Light",
}

SYNPTA_FIELD_MAP = {
    "cadence":   "cgw_snr_4x_cadence",
    "precision": "cgw_snr_hi_prec",
    "combined":  "cgw_snr_4x_cad_2x_prec",
}

# Color now keyed by strategy, not population
STRATEGY_COLORS = {
    "unchanged": "#A6FF00",   # lime
    "cadence":   "#FF2DAA",   # magenta
    "precision": "#FF9800",   # orange
    "combined":  "#00E5FF",   # cyan
}
LINESTYLES = {
    "unchanged": "--",
    "cadence":   ":",
    "precision": "-.",
    "combined":  "-",
}
LINEWIDTHS = {
    "unchanged": 2,
    "cadence":   2.5,
    "precision": 2.5,
    "combined":  3.5,
}
LABELS = {
    "unchanged": "Unchanged",
    "cadence": "Higher Cadence",
    "precision": "Higher Precision",
    "combined": "Higher Cadence \n& Precision",
}
DRAW_ORDER = ["unchanged", "cadence", "precision", "combined"]

param_col = "f"

fig, axes = plt.subplots(
    3, 1,
    figsize=(7, 15),
    sharex=True,
    sharey=True,
)
fig.patch.set_facecolor("black")
plt.subplots_adjust(
    hspace=0.20,
    left=0.14,
    right=0.97,
    bottom=0.06,
    top=0.85,
)
fig.suptitle(
    "Detectable Binary \nGravitational Wave \nFrequency",
    fontsize=34,          # increase to 32–36 if it's a large poster
    fontweight="heavy",   # or "bold"
    color="white",
    y=0.99,              # move title slightly higher
)

for ax, pop in zip(axes, pop_scenarios):
    ax.set_facecolor("black")

    sub_base = synpta_sim_df[synpta_sim_df["scenario"] == pop]

    vals = {}
    unchanged_vals = sub_base[f"baseline_{param_col}"].to_numpy(float)
    vals["unchanged"] = unchanged_vals[np.isfinite(unchanged_vals)]

    for key, field in SYNPTA_FIELD_MAP.items():
        col = f"{field}_{param_col}"
        if col in sub_base.columns:
            v = sub_base[col].to_numpy(float)
            v = v[np.isfinite(v)]
            if v.size:
                vals[key] = v
        else:
            print(f"  [warning] column not found: {col}")

    all_vals = np.concatenate(list(vals.values()))
    all_vals = all_vals[all_vals > 0]
    bins = np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)

    # "Unchanged" gets a light shaded fill behind it (background reference)
    ax.hist(
        vals["unchanged"],
        bins=bins,
        histtype="stepfilled",
        color=STRATEGY_COLORS["unchanged"],
        alpha=0.0,
        zorder=1,
    )

    for key in DRAW_ORDER:
        if key not in vals:
            continue
        ax.hist(
            vals[key],
            bins=bins,
            histtype="step",
            color=STRATEGY_COLORS[key],
            linestyle=LINESTYLES[key],
            linewidth=LINEWIDTHS[key],
            label=LABELS[key],
            zorder=DRAW_ORDER.index(key) + 2,
        )

    ax.set_xscale("log")
    ax.set_title(
        POP_TITLES[pop],
        color="white",
        fontweight="bold",
        pad=12,
    )
    ax.tick_params(colors="white", labelsize=16, width=1.5)
    for spine in ax.spines.values():
        spine.set_color("white")
        spine.set_linewidth(2)
    ax.grid(axis="y", color="white", alpha=0.12)
    ax.set_ylabel("Number of Simulations", color="white", labelpad=15)

    leg = ax.legend(
        title="Observing Strategy",
        labelcolor="white",
        frameon=False,
        loc="upper right",
        bbox_to_anchor=(1.02, 1.0),
        handletextpad=0.3,
    )
    leg.get_title().set_color("white")
    leg.get_title().set_fontweight("bold")
    for text in leg.get_texts():
        text.set_color("white")

axes[-1].set_xlabel("Gravitational Wave Frequency [Hz]", color="white", labelpad=15)

fig.savefig(
    Path("figures") / "poster_frequency_panels.png",
    dpi=1500,
    facecolor="black",
    bbox_inches="tight",
)
plt.show()